# 07 · Extracción canónica de proyectos de generación del BOE

Este notebook extrae exclusivamente la información necesaria para el TFM:

1. **plantas de generación eléctrica** como entidades raíz, agrupables y buscables;
2. **componentes asociados** —almacenamiento y sistema de evacuación/conexión— como entidades secundarias no agrupables;
3. **actuaciones administrativas actuales** con una semántica única de destinatarios;
4. nombres, magnitudes principales, participantes, localizaciones y relaciones materiales necesarias para reconstruir la cronología.

Principio de diseño:

```text
generation_assets = raíces persistentes del proyecto
associated_components = elementos vinculados, nunca raíces de agrupación
targets=["event"] = actuación sobre todo el proyecto del evento
targets concretos = actuación sobre un subconjunto inequívoco
```

La evidencia de una entidad demuestra su existencia. Los enlaces planta–componente y los targets se resuelven con el contexto completo del acto, sin exigir que una única cita contenga simultáneamente toda la estructura.


### Cambio visible de la versión 23

- La clasificación operativa pasa a ser binaria: proyecto de generación específico / no relevante.
- El piloto distingue validación estructural de exactitud de alcance mediante etiquetas externas de evaluación.
- Los anuncios de contratación y los proyectos no energéticos con fotovoltaica auxiliar no se convierten en proyectos de generación.
- Estas reglas no utilizan identificadores BOE y no alteran silenciosamente las extracciones: cada ajuste determinista queda registrado.


**Versión de contrato 25 · revisión operativa 25.1.** La semántica de extracción no cambia respecto de la versión 25. Esta revisión:

- elimina el `FutureWarning` producido al acumular intentos con columnas completamente nulas;
- hace explícita la composición del piloto: **43 documentos específicos** y **57 no relevantes**;
- añade una auditoría tabular de los 100 resultados con nombres de plantas, tecnologías, componentes y actuaciones;
- refuerza la invariancia `sin eventos ⇔ no relevante` sin modificar ninguna extracción ya validada.


In [1]:
from __future__ import annotations

import asyncio
import json
import re
import unicodedata
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timezone
from enum import Enum
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from typing import Annotated, Any, Literal, TypeAlias, TypedDict
from uuid import uuid4

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    TypeAdapter,
    field_validator,
    model_validator,
)
from typing_extensions import Self

# Pydantic AI se importa de forma opcional para que el contrato y las pruebas
# deterministas puedan ejecutarse también en entornos sin acceso al modelo.
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.ollama import OllamaModel
    from pydantic_ai.output import NativeOutput
    from pydantic_ai.providers.ollama import OllamaProvider
    from pydantic_ai.usage import RunUsage, UsageLimits
    PYDANTIC_AI_AVAILABLE = True
except ModuleNotFoundError:
    Agent = Any
    OllamaModel = Any
    NativeOutput = None
    OllamaProvider = Any
    PYDANTIC_AI_AVAILABLE = False

    @dataclass
    class RunUsage:  # fallback exclusivo para pruebas deterministas
        requests: int = 0
        input_tokens: int = 0
        output_tokens: int = 0
        total_tokens: int = 0

    class UsageLimits:
        def __init__(self, request_limit: int) -> None:
            self.request_limit = request_limit


def validate_required_columns(
    dataframe: pd.DataFrame,
    required_columns: set[str],
) -> None:
    missing = required_columns - set(dataframe.columns)
    if missing:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(missing)}")


def find_project_root(start: Path | None = None) -> Path:
    """Localiza la raíz del proyecto buscando pyproject.toml."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        f"No se encontró pyproject.toml desde {start}. "
        "Ejecuta el notebook dentro del repositorio del TFM."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_CANDIDATES_DOCS_TEXT_PATH = (
    SILVER_DIR
    / "boe_candidates_docs_text"
    / "boe_candidates_docs_text.parquet"
)

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
BOE_AI_EXTRACTION_ATTEMPTS_PATH = (
    SILVER_BOE_AI_DIR / "boe_ai_extractions_attempts.parquet"
)

BOE_AI_REVIEW_QUEUE_PATH = SILVER_BOE_AI_DIR / "boe_ai_review_queue.parquet"
BOE_AI_MANUAL_REVIEWS_PATH = SILVER_BOE_AI_DIR / "boe_ai_manual_reviews.parquet"
BOE_AI_QUALITY_METRICS_PATH = SILVER_BOE_AI_DIR / "boe_ai_quality_metrics.parquet"
BOE_AI_MANUAL_REVIEW_DIR = DATA_DIR / "manual" / "boe_ai_reviews"

# Tablas derivadas. Solo generation_asset_mentions alimentará la agrupación.
PUBLICATION_EVENTS_PATH = SILVER_BOE_AI_DIR / "publication_events.parquet"
GENERATION_ASSET_MENTIONS_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_mentions.parquet"
)
GENERATION_ASSET_NAMES_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_names.parquet"
)
ASSOCIATED_COMPONENTS_PATH = (
    SILVER_BOE_AI_DIR / "associated_components.parquet"
)
ASSOCIATED_COMPONENT_NAMES_PATH = (
    SILVER_BOE_AI_DIR / "associated_component_names.parquet"
)
ASSOCIATED_COMPONENT_GENERATION_LINKS_PATH = (
    SILVER_BOE_AI_DIR / "associated_component_generation_links.parquet"
)
ADMINISTRATIVE_ACTIONS_PATH = (
    SILVER_BOE_AI_DIR / "administrative_actions.parquet"
)
ADMINISTRATIVE_ACTION_TARGETS_PATH = (
    SILVER_BOE_AI_DIR / "administrative_action_targets.parquet"
)
PARTICIPANT_MENTIONS_PATH = (
    SILVER_BOE_AI_DIR / "participant_mentions.parquet"
)
LOCATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "location_mentions.parquet"
GENERATION_RELATIONS_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_relations.parquet"
)
TECHNICAL_MENTIONS_PATH = SILVER_BOE_AI_DIR / "technical_mentions.parquet"
CASE_FILE_REFERENCES_PATH = (
    SILVER_BOE_AI_DIR / "case_file_references.parquet"
)

## 1. Contrato de extracción

In [2]:
NonEmptyText: TypeAlias = Annotated[
    str,
    StringConstraints(strip_whitespace=True, min_length=1),
]

GenerationAssetRef: TypeAlias = Annotated[
    str,
    StringConstraints(pattern=r"^generation_asset_[1-9][0-9]*$"),
]

ComponentRef: TypeAlias = Annotated[
    str,
    StringConstraints(pattern=r"^component_[1-9][0-9]*$"),
]

EntityRef: TypeAlias = Annotated[
    str,
    StringConstraints(
        pattern=(
            r"^(?:generation_asset_[1-9][0-9]*|component_[1-9][0-9]*)$"
        )
    ),
]

ActionTargetRef: TypeAlias = Annotated[
    str,
    StringConstraints(
        pattern=(
            r"^(?:event|generation_asset_[1-9][0-9]*|component_[1-9][0-9]*)$"
        )
    ),
]

BOEId: TypeAlias = Annotated[
    str,
    StringConstraints(pattern=r"^BOE-[AB]-[0-9]{4}-[0-9]+$"),
]


class ContractModel(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
    )


def _text_key(value: str) -> str:
    return " ".join(value.split()).casefold()


def _deduplicate_strings(values: list[str]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for value in values:
        key = _text_key(value)
        if key not in seen:
            seen.add(key)
            result.append(value)
    return result


def _reference_sort_key(value: str) -> tuple[int, int]:
    if value == "event":
        return (0, 0)
    prefix, number = value.rsplit("_", 1)
    return (1 if prefix == "generation_asset" else 2, int(number))


def _canonicalize_refs(values: list[str]) -> list[str]:
    return sorted(set(values), key=_reference_sort_key)


class ClassificationStatus(str, Enum):
    CLASSIFIED = "classified"
    UNCERTAIN = "uncertain"


class DocumentScope(str, Enum):
    NOT_RELEVANT_FOR_GENERATION_PROJECTS = (
        "not_relevant_for_generation_projects"
    )
    GENERATION_PROJECT_SPECIFIC = "generation_project_specific"


class GenerationType(str, Enum):
    PHOTOVOLTAIC = "fotovoltaica"
    WIND = "eolica"
    CONCENTRATED_SOLAR_POWER = "termosolar"
    HYDROPOWER = "hidroelectrica"
    GEOTHERMAL = "geotermica"
    BIOMASS = "biomasa"
    BIOGAS = "biogas"
    OTHER_GENERATION = "otra_generacion"


class AssociatedComponentType(str, Enum):
    ENERGY_STORAGE = "almacenamiento"
    EVACUATION_SYSTEM = "sistema_evacuacion"
    ELECTRICAL_SUBSTATION = "subestacion_electrica"
    POWER_LINE = "linea_electrica"
    GRID_CONNECTION = "conexion_red"
    OTHER_ASSOCIATED_COMPONENT = "otro_componente_asociado"


class TechnicalAttributeType(str, Enum):
    INSTALLED_POWER = "potencia_instalada"
    PEAK_POWER = "potencia_pico"
    STORAGE_POWER = "potencia_almacenamiento"
    STORAGE_CAPACITY = "capacidad_almacenamiento"
    VOLTAGE = "tension"
    UNIT_COUNT = "numero_unidades"
    UNIT_POWER = "potencia_unitaria"
    OTHER = "otra"


class ParticipantRole(str, Enum):
    PROMOTER = "promotor"
    CO_PROMOTER = "copromotor"
    HOLDER = "titular"
    OPERATOR = "operador"
    APPLICANT = "solicitante"
    TRANSFEROR = "cedente"
    TRANSFEREE = "cesionario"
    OTHER = "otro"
    UNKNOWN = "desconocido"


class AdministrativeLocationLevel(str, Enum):
    MUNICIPALITY = "municipio"
    PROVINCE = "provincia"
    AUTONOMOUS_COMMUNITY = "comunidad_autonoma"


class GenerationRelationType(str, Enum):
    HYBRIDIZED_WITH = "hibrida_con"
    REPLACES = "sustituye_a"


class AdministrativeActionType(str, Enum):
    APPLICATION_SUBMISSION = "solicitud_tramitacion"
    ENVIRONMENTAL_APPLICATION_SUBMISSION = "solicitud_tramitacion_ambiental"
    DOCUMENTATION_CORRECTION = "subsanacion_documentacion"
    REQUIREMENTS_VERIFICATION = "verificacion_requisitos_tramitacion"
    ERROR_CORRECTION = "correccion_errores"
    PUBLIC_INFORMATION = "informacion_publica"
    ENVIRONMENTAL_IMPACT_ASSESSMENT = "evaluacion_impacto_ambiental"
    ENVIRONMENTAL_IMPACT_STATEMENT = "declaracion_impacto_ambiental"
    ENVIRONMENTAL_IMPACT_REPORT = "informe_impacto_ambiental"
    ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT = (
        "informe_determinacion_afeccion_ambiental"
    )
    PRIOR_ADMINISTRATIVE_AUTHORIZATION = "autorizacion_administrativa_previa"
    CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION = (
        "autorizacion_administrativa_construccion"
    )
    OPERATING_AUTHORIZATION = "autorizacion_explotacion"
    WATER_CONCESSION = "concesion_aguas"
    PUBLIC_UTILITY_DECLARATION = "declaracion_utilidad_publica"
    FORCED_EXPROPRIATION = "expropiacion_forzosa"
    AFFECTED_ASSETS_AND_RIGHTS_LIST = "relacion_bienes_derechos_afectados"
    PRIOR_OCCUPATION_RECORDS = "levantamiento_actas_previas_ocupacion"
    OCCUPATION_RECORDS = "actas_ocupacion"
    AUTHORIZATION_MODIFICATION = "modificacion_autorizacion"
    DEADLINE_EXTENSION = "prorroga"
    OWNERSHIP_CHANGE = "cambio_titularidad"
    PROCEDURE_TERMINATION = "terminacion_procedimiento"
    OTHER = "otro"
    UNKNOWN = "desconocido"


class AdministrativeDecision(str, Enum):
    REQUESTED = "solicitado"
    CORRECTED = "subsanado"
    REQUIREMENTS_VERIFIED = "requisitos_verificados"
    RECTIFIED = "rectificado"
    SUBMITTED_TO_PUBLIC_INFORMATION = "sometido_informacion_publica"
    ANNOUNCED = "convocado"
    FORMULATED = "formulado"
    FAVORABLE = "favorable"
    UNFAVORABLE = "desfavorable"
    NO_SIGNIFICANT_ADVERSE_ENVIRONMENTAL_EFFECTS = (
        "sin_efectos_adversos_significativos"
    )
    ORDINARY_ENVIRONMENTAL_ASSESSMENT_REQUIRED = (
        "requiere_evaluacion_ambiental_ordinaria"
    )
    FURTHER_ENVIRONMENTAL_ASSESSMENT_REQUIRED = (
        "requiere_evaluacion_ambiental_adicional"
    )
    FURTHER_ENVIRONMENTAL_ASSESSMENT_NOT_REQUIRED = (
        "no_requiere_evaluacion_ambiental_adicional"
    )
    AUTHORIZED = "autorizado"
    DECLARED = "declarado"
    MODIFIED = "modificado"
    EXTENDED = "prorrogado"
    DENIED = "denegado"
    CLOSED = "archivado"
    WITHDRAWN = "desistido"
    INADMISSIBLE = "inadmitido"
    OTHER = "otro"
    UNKNOWN = "desconocido"


_ALLOWED_DECISIONS_BY_ACTION_TYPE: dict[
    AdministrativeActionType,
    set[AdministrativeDecision],
] = {
    AdministrativeActionType.APPLICATION_SUBMISSION: {
        AdministrativeDecision.REQUESTED,
    },
    AdministrativeActionType.ENVIRONMENTAL_APPLICATION_SUBMISSION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
    },
    AdministrativeActionType.DOCUMENTATION_CORRECTION: {
        AdministrativeDecision.CORRECTED,
    },
    AdministrativeActionType.REQUIREMENTS_VERIFICATION: {
        AdministrativeDecision.REQUIREMENTS_VERIFIED,
    },
    AdministrativeActionType.ERROR_CORRECTION: {
        AdministrativeDecision.RECTIFIED,
    },
    AdministrativeActionType.PUBLIC_INFORMATION: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.FAVORABLE,
        AdministrativeDecision.UNFAVORABLE,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.NO_SIGNIFICANT_ADVERSE_ENVIRONMENTAL_EFFECTS,
        AdministrativeDecision.ORDINARY_ENVIRONMENTAL_ASSESSMENT_REQUIRED,
    },
    AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.FAVORABLE,
        AdministrativeDecision.UNFAVORABLE,
        AdministrativeDecision.FURTHER_ENVIRONMENTAL_ASSESSMENT_REQUIRED,
        AdministrativeDecision.FURTHER_ENVIRONMENTAL_ASSESSMENT_NOT_REQUIRED,
    },
    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.OPERATING_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.WATER_CONCESSION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.PUBLIC_UTILITY_DECLARATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.DECLARED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.FORCED_EXPROPRIATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DECLARED,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.AFFECTED_ASSETS_AND_RIGHTS_LIST: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.PRIOR_OCCUPATION_RECORDS: {
        AdministrativeDecision.ANNOUNCED,
        AdministrativeDecision.FORMULATED,
    },
    AdministrativeActionType.OCCUPATION_RECORDS: {
        AdministrativeDecision.ANNOUNCED,
        AdministrativeDecision.FORMULATED,
    },
    AdministrativeActionType.AUTHORIZATION_MODIFICATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.MODIFIED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.DEADLINE_EXTENSION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.EXTENDED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.OWNERSHIP_CHANGE: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.PROCEDURE_TERMINATION: {
        AdministrativeDecision.CLOSED,
        AdministrativeDecision.WITHDRAWN,
        AdministrativeDecision.INADMISSIBLE,
    },
}


class TechnicalMention(ContractModel):
    attribute_type: TechnicalAttributeType
    value_raw: NonEmptyText
    evidence: NonEmptyText


class GenerationAssetMention(ContractModel):
    """Planta de generación que será raíz de agrupación y búsqueda."""

    local_generation_asset_ref: GenerationAssetRef
    names_raw: list[NonEmptyText] = Field(min_length=1)
    generation_type: GenerationType
    technical_mentions: list[TechnicalMention] = Field(default_factory=list)
    evidence: NonEmptyText

    @field_validator("names_raw")
    @classmethod
    def canonicalize_names(cls, values: list[str]) -> list[str]:
        return _deduplicate_strings(values)


class AssociatedComponent(ContractModel):
    """Componente vinculado a plantas, pero nunca raíz de agrupación."""

    local_component_ref: ComponentRef
    component_type: AssociatedComponentType
    names_raw: list[NonEmptyText] = Field(default_factory=list)
    description_raw: NonEmptyText | None = None
    related_generation_asset_refs: list[GenerationAssetRef] = Field(
        default_factory=list
    )
    technical_mentions: list[TechnicalMention] = Field(default_factory=list)
    evidence: NonEmptyText

    @field_validator("names_raw")
    @classmethod
    def canonicalize_names(cls, values: list[str]) -> list[str]:
        return _deduplicate_strings(values)

    @field_validator("related_generation_asset_refs")
    @classmethod
    def canonicalize_related_refs(cls, values: list[str]) -> list[str]:
        return _canonicalize_refs(values)


class ParticipantMention(ContractModel):
    """Participante del proyecto representado por el PublicationEvent."""

    participant_name_raw: NonEmptyText
    participant_role: ParticipantRole
    evidence: NonEmptyText


class AdministrativeLocationMention(ContractModel):
    """Localización administrativa del proyecto representado por el evento."""

    location_name_raw: NonEmptyText
    location_level: AdministrativeLocationLevel
    province_hint_raw: NonEmptyText | None = None
    autonomous_community_hint_raw: NonEmptyText | None = None
    evidence: NonEmptyText

    @model_validator(mode="after")
    def validate_hints(self) -> Self:
        if (
            self.location_level == AdministrativeLocationLevel.PROVINCE
            and self.province_hint_raw is not None
        ):
            raise ValueError(
                "Una provincia no debe repetirse en province_hint_raw."
            )
        if self.location_level == AdministrativeLocationLevel.AUTONOMOUS_COMMUNITY:
            if self.province_hint_raw is not None or self.autonomous_community_hint_raw is not None:
                raise ValueError(
                    "Una comunidad autónoma no debe contener hints."
                )
        return self


class GenerationAssetRelation(ContractModel):
    source_generation_asset_ref: GenerationAssetRef
    target_generation_asset_ref: GenerationAssetRef
    relation_type: GenerationRelationType
    evidence: NonEmptyText

    @model_validator(mode="after")
    def validate_relation(self) -> Self:
        if self.source_generation_asset_ref == self.target_generation_asset_ref:
            raise ValueError("Una relación no puede ser autorreferencial.")
        return self


class AdministrativeAction(ContractModel):
    action_type: AdministrativeActionType
    decision: AdministrativeDecision
    is_modification: bool = False
    targets: list[ActionTargetRef] = Field(default_factory=list)
    evidence: NonEmptyText

    @field_validator("targets")
    @classmethod
    def canonicalize_targets(cls, values: list[str]) -> list[str]:
        return _canonicalize_refs(values)

    @model_validator(mode="after")
    def validate_action(self) -> Self:
        if "event" in self.targets and len(self.targets) != 1:
            raise ValueError(
                "El target 'event' no puede combinarse con referencias concretas."
            )
        if self.action_type not in {
            AdministrativeActionType.OTHER,
            AdministrativeActionType.UNKNOWN,
        } and self.decision not in {
            AdministrativeDecision.OTHER,
            AdministrativeDecision.UNKNOWN,
        }:
            allowed = _ALLOWED_DECISIONS_BY_ACTION_TYPE.get(self.action_type)
            if allowed is not None and self.decision not in allowed:
                raise ValueError(
                    "Combinación administrativa incoherente: "
                    f"{self.action_type.value} + {self.decision.value}."
                )
        return self


class PublicationEvent(ContractModel):
    generation_assets: list[GenerationAssetMention] = Field(min_length=1)
    associated_components: list[AssociatedComponent] = Field(default_factory=list)
    administrative_actions: list[AdministrativeAction] = Field(min_length=1)
    participants: list[ParticipantMention] = Field(default_factory=list)
    administrative_locations: list[AdministrativeLocationMention] = Field(
        default_factory=list
    )
    generation_relations: list[GenerationAssetRelation] = Field(
        default_factory=list
    )
    case_file_references: list[NonEmptyText] = Field(default_factory=list)
    event_summary: NonEmptyText

    @model_validator(mode="after")
    def validate_event_graph(self) -> Self:
        generation_refs = [
            asset.local_generation_asset_ref for asset in self.generation_assets
        ]
        expected_generation_refs = [
            f"generation_asset_{index}"
            for index in range(1, len(generation_refs) + 1)
        ]
        if generation_refs != expected_generation_refs:
            raise ValueError(
                "Las referencias de plantas deben ser consecutivas y estar ordenadas: "
                f"{expected_generation_refs}."
            )

        component_refs = [
            component.local_component_ref
            for component in self.associated_components
        ]
        expected_component_refs = [
            f"component_{index}"
            for index in range(1, len(component_refs) + 1)
        ]
        if component_refs != expected_component_refs:
            raise ValueError(
                "Las referencias de componentes deben ser consecutivas y estar ordenadas: "
                f"{expected_component_refs}."
            )

        valid_generation_refs = set(generation_refs)
        valid_entity_refs = valid_generation_refs | set(component_refs)

        generation_identity_keys: set[tuple[str, tuple[str, ...]]] = set()
        for asset in self.generation_assets:
            identity_key = (
                asset.generation_type.value,
                tuple(sorted(_text_key(name) for name in asset.names_raw)),
            )
            if identity_key in generation_identity_keys:
                raise ValueError(
                    "Existen plantas de generación duplicadas con el mismo "
                    "tipo y los mismos nombres."
                )
            generation_identity_keys.add(identity_key)

        for component in self.associated_components:
            missing = (
                set(component.related_generation_asset_refs)
                - valid_generation_refs
            )
            if missing:
                raise ValueError(
                    "AssociatedComponent referencia plantas inexistentes: "
                    f"{sorted(missing)}."
                )

        for action in self.administrative_actions:
            concrete_targets = set(action.targets) - {"event"}
            missing = concrete_targets - valid_entity_refs
            if missing:
                raise ValueError(
                    "AdministrativeAction contiene targets inexistentes: "
                    f"{sorted(missing)}."
                )
        for relation in self.generation_relations:
            missing = {
                relation.source_generation_asset_ref,
                relation.target_generation_asset_ref,
            } - valid_generation_refs
            if missing:
                raise ValueError(
                    "GenerationAssetRelation contiene referencias inexistentes: "
                    f"{sorted(missing)}."
                )



        return self


class BOEAIExtraction(ContractModel):
    classification_status: ClassificationStatus
    document_scope: DocumentScope | None = None
    classification_reason: NonEmptyText
    publication_events: list[PublicationEvent] = Field(default_factory=list)
    extraction_notes: NonEmptyText | None = None

    @field_validator("document_scope", mode="before")
    @classmethod
    def normalize_legacy_document_scope(cls, value: Any) -> Any:
        # Compatibilidad de lectura con extracciones anteriores. La versión 23
        # expone solo dos clases al modelo y a las tablas vigentes.
        if value == "energy_general":
            return DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        return value

    @model_validator(mode="after")
    def validate_classification(self) -> Self:
        if self.classification_status == ClassificationStatus.UNCERTAIN:
            if self.document_scope is not None or self.publication_events:
                raise ValueError(
                    "Una clasificación incierta no debe contener scope ni eventos."
                )
            return self

        if self.document_scope is None:
            raise ValueError("Una publicación clasificada debe tener document_scope.")

        project_specific = (
            self.document_scope == DocumentScope.GENERATION_PROJECT_SPECIFIC
        )
        if project_specific and not self.publication_events:
            raise ValueError(
                "Una publicación específica de proyecto debe contener eventos."
            )
        if not project_specific and self.publication_events:
            raise ValueError(
                "Solo una publicación específica de proyecto puede contener eventos."
            )
        return self


class BOEProjectExtraction(BOEAIExtraction):
    boe_id: BOEId
    publication_date: date


def build_boe_project_extraction(
    ai_extraction: BOEAIExtraction,
    *,
    boe_id: str,
    publication_date: date,
) -> BOEProjectExtraction:
    return BOEProjectExtraction(
        **ai_extraction.model_dump(),
        boe_id=boe_id,
        publication_date=publication_date,
    )

## 2. Instrucciones del agente

In [3]:
CORE_INSTRUCTIONS = r"""
Eres un extractor canónico de publicaciones del Boletín Oficial del Estado
relativas a proyectos de generación eléctrica.

Devuelve exclusivamente BOEAIExtraction. No añadas markdown, comentarios,
campos no definidos, conocimiento externo ni inferencias no respaldadas.

OBJETIVO DEL TFM

La salida se utilizará para agrupar publicaciones por planta de generación,
ordenar sus actuaciones y reconstruir su cronología administrativa. La entidad
central y buscable es siempre la planta de generación, no su infraestructura.

MODELO MÍNIMO

1. generation_assets
   - Incluye exclusivamente plantas de generación eléctrica con uno o varios
     nombres literales en names_raw.
   - Una frase descriptiva solo puede ser nombre de planta cuando el proyecto
     principal publicado sea realmente una instalación de generación eléctrica.
     No conviertas en planta el nombre de una obra hidráulica, regadío, edificio,
     carretera, depuradora u otro proyecto sectorial porque incluya paneles solares.
   - Conserva íntegramente la denominación oficial de una planta cuando sea literal.
     No inventes un nombre corto.
   - Cada planta independiente es una raíz posterior de agrupación y búsqueda.
   - No incluyas almacenamiento, líneas, subestaciones, evacuación ni conexión.
   - Si una instalación híbrida contiene tecnologías de generación distintas,
     crea una planta por tecnología. Pueden compartir exactamente los mismos
     nombres literales; no inventes sufijos como «eólica» o «fotovoltaica».
   - Usa otra_generacion solo si la fuente identifica una tecnología eléctrica
     que no encaja en la taxonomía.

2. associated_components
   - Incluye solo componentes necesarios para comprender la actuación actual:
     almacenamiento y sistema de evacuación/conexión asociado.
   - Nunca son raíces de agrupación.
   - Representa líneas, subestaciones, posiciones y conexión de un mismo
     proyecto como un único sistema_evacuacion, salvo que el acto actual afecte
     inequívocamente a componentes distintos y separarlos sea imprescindible.
   - names_raw puede estar vacío. description_raw puede ser null.
   - related_generation_asset_refs puede dejarse vacío si el vínculo no aparece
     en la misma cita: se resolverá determinísticamente con el contexto completo.

3. administrative_actions.targets
   - target='event' significa que la actuación afecta al proyecto completo del
     PublicationEvent: todas sus plantas y componentes asociados.
   - Usa referencias concretas únicamente cuando la actuación afecte a un
     subconjunto inequívoco, por ejemplo solo a un almacenamiento o solo al
     sistema de evacuación.
   - Si el texto dice «la planta X y su infraestructura», usa ['event'] cuando
     ambas constituyen todo el evento.
   - Si dice «infraestructura de evacuación de la planta X», no deduzcas que la
     actuación afecta también a la planta; el target puede ser el componente.
   - targets puede dejarse vacío si la cita no permite resolverlo sin el título;
     la canonicalización lo completará.

4. participants y administrative_locations
   - Son atributos del proyecto representado por el evento.
   - Extrae solo participantes y localizaciones explícitos y útiles para
     identificar o contextualizar el proyecto.

5. generation_relations
   - Conserva únicamente hibridación o sustitución material entre plantas.
   - Compartir evacuación no crea una relación entre plantas.

CLASIFICACIÓN BINARIA

- generation_project_specific: el objeto administrativo actual permite identificar
  al menos una planta de generación eléctrica independiente y una actuación de su
  ciclo administrativo. La planta será una entidad buscable y agrupable.
- not_relevant_for_generation_projects: cualquier otro documento, incluidos los
  energéticos generales. No crees publication_events.
- Son no relevantes los anuncios de contratación, licitación, adjudicación o
  suministro de placas para autoconsumo en edificios. Una contratación no es una
  autorización administrativa de construcción.
- Son no relevantes los proyectos de abastecimiento, depuración, regadío, bombeo,
  carreteras, ferrocarriles u otras obras cuyo objeto principal no sea la planta de
  generación, aunque incorporen fotovoltaica o renovables como alimentación auxiliar.
- Son no relevantes las instalaciones autónomas de almacenamiento, evacuación,
  transporte, distribución o gas cuando el objeto administrativo actual no identifique
  una planta de generación asociada. No conviertas una planta de almacenamiento en
  generation_asset: el almacenamiento solo puede ser associated_component de una
  planta de generación identificable.
- La mera presencia de vocabulario energético o de paneles fotovoltaicos no basta.
- Una concesión de aguas destinada a una central o aprovechamiento hidroeléctrico
  identificable sí forma parte de la cronología de generación.

GRANULARIDAD

- Un PublicationEvent representa un proyecto de generación independiente.
- Separa plantas independientes en eventos distintos aunque compartan una
  resolución o una infraestructura de evacuación.
- Mantén varias plantas en el mismo evento solo ante hibridación, sustitución o
  almacenamiento integrado documentado.
- Una publicación puede contener varias actuaciones actuales del mismo proyecto;
  mantenlas en un único evento.

BARRERA TEMPORAL

Extrae únicamente el objeto administrativo actual de la publicación. No conviertas
antecedentes, autorizaciones históricas o renuncias previas en actuaciones con la
fecha de publicación actual.

EVIDENCE

- Debe ser literal y verificable en título o cuerpo.
- Puede usar '[...]' para enlazar fragmentos literales separados.
- La evidencia de una entidad prueba su existencia; no tiene que contener además
  todos sus vínculos.
- La evidencia de una actuación prueba el tipo y la decisión. Los targets se
  resolverán con el título y el contexto completo cuando sea necesario.
- No reconstruyas frases ni corrijas la redacción del BOE.

ALCANCE

Extrae solo nombres, tecnología, potencia/capacidad principal, participantes,
localizaciones, componentes necesarios, expedientes y actuaciones actuales.
Omite detalle técnico que no contribuya a identidad, agrupación o cronología.
"""

TAXONOMY_GUIDANCE = r"""
DECISIONES ADMINISTRATIVAS

- «se somete a información pública la solicitud de X»:
  action_type=X y decision=sometido_informacion_publica. No añadas además una
  actuación genérica informacion_publica.
- Una solicitud no es una autorización. Una formalización de contrato tampoco es
  una autorización administrativa energética.
- Una concesión de aguas para producción hidroeléctrica usa concesion_aguas; no la
  conviertas en autorizacion_administrativa_previa.
- Una DIA o informe ambiental solo es producto final cuando se formula o
  resuelve. Durante información pública usa evaluacion_impacto_ambiental.
- is_modification=True solo cuando el acto actual modifica una autorización o
  declaración previa del mismo tipo.

COMPONENTES

- almacenamiento, baterías o BESS -> almacenamiento;
- evacuación, líneas, subestaciones, posiciones y conexión asociadas a una planta
  -> sistema_evacuacion agregado.

MAGNITUDES

- potencia de planta -> potencia_instalada o potencia_pico;
- potencia de almacenamiento -> potencia_almacenamiento;
- energía en MWh -> capacidad_almacenamiento;
- tensión en kV -> tension.
Conserva value_raw literalmente. No realices cálculos ni conversiones.
"""

DECISION_EXAMPLES = r"""
EJEMPLO 1 — CARBO

Texto: «se somete a Información Pública la solicitud de Declaración, en concreto,
de Utilidad Pública de la planta solar fotovoltaica Carbo [...] e infraestructura
de evacuación a 30 kV».

- generation_asset_1: Carbo, fotovoltaica;
- component_1: sistema_evacuacion vinculado a generation_asset_1;
- acción declaracion_utilidad_publica + sometido_informacion_publica;
- targets=['event'], porque el acto alcanza a todo el evento: planta e
  infraestructura.

EJEMPLO 2 — ARMUS

Una instalación híbrida «Armus Solar» integra 35 MW eólicos y 49,88 MW
fotovoltaicos. Se formula un informe ambiental para el módulo de almacenamiento
«Armus», de 20 MW y 80 MWh, y su infraestructura de evacuación.

- dos generation_assets con el mismo nombre literal y tecnologías distintas;
- component_1 almacenamiento vinculado a ambas plantas;
- component_2 sistema_evacuacion vinculado a ambas plantas;
- relación hibrida_con entre las plantas;
- la actuación ambiental tiene targets=['component_1', 'component_2'], porque
  el objeto actual comprende el almacenamiento y su evacuación, no las plantas
  existentes con las que se hibrida.

EJEMPLO 3 — ACTO SOBRE EVACUACIÓN

Si se convocan actas previas para «la infraestructura de evacuación de la planta
X», conserva X como generation_asset, crea el sistema de evacuación como
associated_component y dirige la actuación solo al component. La preposición
«de» identifica la infraestructura; no convierte automáticamente la planta en
destinataria jurídica.

EJEMPLO 4 — INFRAESTRUCTURA SIN PLANTA

Si el documento solo trata una línea o subestación y no permite identificar una
planta de generación con nombre propio, no crees evento: usa
not_relevant_for_generation_projects.

EJEMPLO 5 — FOTOVOLTAICA AUXILIAR

Un contrato para instalar placas de autoconsumo en un edificio, o una obra de
abastecimiento/regadío que incorpora paneles para alimentar bombas, no constituye
un proyecto de generación buscable: usa not_relevant_for_generation_projects.
"""

AGENT_INSTRUCTIONS = "\n\n".join([
    CORE_INSTRUCTIONS.strip(),
    TAXONOMY_GUIDANCE.strip(),
    DECISION_EXAMPLES.strip(),
])


## 3. Modelo y configuración reproducible

In [4]:
ModelProvider = Literal["gemini", "ollama"]

MODEL_PROVIDER: ModelProvider = "gemini"
# MODEL_PROVIDER: ModelProvider = "ollama"
AI_MODEL_NAME = "google:gemini-2.5-flash"
AGENT_RETRIES = 3
USE_NATIVE_OUTPUT = True
MODEL_SETTINGS: dict[str, Any] = {"temperature": 0.0}

if MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AGENT_RETRIES = 4

DOCUMENT_TIMEOUT_SECONDS = 600.0
MODEL_RUN_TIMEOUT_SECONDS = 240.0
MAX_MODEL_REQUESTS_PER_DOCUMENT = 6
DOCUMENT_VALIDATION_RETRY_ATTEMPTS = 1
TRANSIENT_RUN_ATTEMPTS = 2
TRANSIENT_RETRY_BASE_SECONDS = 2.0
CHECKPOINT_EVERY = 5

DOCUMENT_VALIDATION_VERSION = "25"
ENTITY_MODEL_POLICY = "generation_roots_components_event_targets_v3"
EVENT_GRANULARITY_POLICY = "canonical_split_independent_generation_projects_v3"
TEMPORAL_POLICY = "current_publication_object_only_v2"
DOCUMENTARY_MATCH_POLICY = "generation_titles_and_exact_source_spans_v5"
TARGET_SEMANTICS_POLICY = "contextual_entity_targets_v2"
COMPONENT_LINK_POLICY = "contextual_links_not_same_quote_required_v1"
QUALITY_WORKFLOW_POLICY = "auto_review_manual_precedence_v2"
SCOPE_CLASSIFICATION_POLICY = "binary_named_generation_pre_model_guard_v3"

MAX_DOCUMENT_CHARS: int | None = None
MIN_SUBSTANTIVE_TEXT_CHARS_BEFORE_ANNEX = 1_000
_AFFECTED_ASSETS_ANNEX_HEADING_RE = re.compile(
    r"(?im)^[ \t]*(?:ANEXO(?:\s+[A-Z0-9IVX.-]+)?\s*[:.-]?\s*)?"
    r"RELACI[ÓO]N(?:\s+CONCRETA\s+E\s+INDIVIDUALIZADA)?"
    r"\s+DE\s+BIENES\s+Y\s+DERECHOS\s+AFECTADOS.*$"
)

DOCUMENT_PROMPT_TEMPLATE = r"""
Analiza exclusivamente la publicación delimitada a continuación.
No reproduzcas boe_id ni publication_date en BOEAIExtraction.

BOE_ID: {boe_id}
FECHA_PUBLICACION: {publication_date}
TITULO: {title}
ESTRATEGIA_TEXTO: {input_selection_strategy}

<BOE_DOCUMENT>
{document_text}
</BOE_DOCUMENT>
""".strip()


def validate_runtime_configuration() -> None:
    """Valida únicamente invariantes operativas, sin acoplar pruebas a una versión concreta."""
    if MODEL_PROVIDER not in {"gemini", "ollama"}:
        raise ValueError(
            "MODEL_PROVIDER debe ser 'gemini' u 'ollama': "
            f"{MODEL_PROVIDER!r}."
        )

    boolean_settings = {
        "USE_NATIVE_OUTPUT": USE_NATIVE_OUTPUT,
    }
    for setting_name, setting_value in boolean_settings.items():
        if not isinstance(setting_value, bool):
            raise TypeError(
                f"{setting_name} debe ser bool, no {type(setting_value).__name__}."
            )

    positive_integer_settings = {
        "AGENT_RETRIES": AGENT_RETRIES,
        "MAX_MODEL_REQUESTS_PER_DOCUMENT": MAX_MODEL_REQUESTS_PER_DOCUMENT,
        "TRANSIENT_RUN_ATTEMPTS": TRANSIENT_RUN_ATTEMPTS,
        "CHECKPOINT_EVERY": CHECKPOINT_EVERY,
        "MIN_SUBSTANTIVE_TEXT_CHARS_BEFORE_ANNEX": (
            MIN_SUBSTANTIVE_TEXT_CHARS_BEFORE_ANNEX
        ),
    }
    for setting_name, setting_value in positive_integer_settings.items():
        if not isinstance(setting_value, int) or isinstance(setting_value, bool):
            raise TypeError(f"{setting_name} debe ser un entero positivo.")
        if setting_value <= 0:
            raise ValueError(f"{setting_name} debe ser mayor que cero.")

    nonnegative_integer_settings = {
        "DOCUMENT_VALIDATION_RETRY_ATTEMPTS": (
            DOCUMENT_VALIDATION_RETRY_ATTEMPTS
        ),
    }
    for setting_name, setting_value in nonnegative_integer_settings.items():
        if not isinstance(setting_value, int) or isinstance(setting_value, bool):
            raise TypeError(f"{setting_name} debe ser un entero no negativo.")
        if setting_value < 0:
            raise ValueError(f"{setting_name} no puede ser negativo.")

    positive_numeric_settings = {
        "DOCUMENT_TIMEOUT_SECONDS": DOCUMENT_TIMEOUT_SECONDS,
        "MODEL_RUN_TIMEOUT_SECONDS": MODEL_RUN_TIMEOUT_SECONDS,
        "TRANSIENT_RETRY_BASE_SECONDS": TRANSIENT_RETRY_BASE_SECONDS,
    }
    for setting_name, setting_value in positive_numeric_settings.items():
        if (
            not isinstance(setting_value, (int, float))
            or isinstance(setting_value, bool)
            or setting_value <= 0
        ):
            raise ValueError(f"{setting_name} debe ser un número mayor que cero.")

    if MODEL_RUN_TIMEOUT_SECONDS >= DOCUMENT_TIMEOUT_SECONDS:
        raise ValueError(
            "MODEL_RUN_TIMEOUT_SECONDS debe ser menor que DOCUMENT_TIMEOUT_SECONDS."
        )

    if MAX_DOCUMENT_CHARS is not None:
        if (
            not isinstance(MAX_DOCUMENT_CHARS, int)
            or isinstance(MAX_DOCUMENT_CHARS, bool)
            or MAX_DOCUMENT_CHARS <= 0
        ):
            raise ValueError(
                "MAX_DOCUMENT_CHARS debe ser None o un entero mayor que cero."
            )

    if not re.fullmatch(r"[1-9]\d*", DOCUMENT_VALIDATION_VERSION):
        raise ValueError(
            "DOCUMENT_VALIDATION_VERSION debe ser una cadena entera positiva."
        )

    for policy_name, policy_value in {
        "ENTITY_MODEL_POLICY": ENTITY_MODEL_POLICY,
        "EVENT_GRANULARITY_POLICY": EVENT_GRANULARITY_POLICY,
        "TEMPORAL_POLICY": TEMPORAL_POLICY,
        "DOCUMENTARY_MATCH_POLICY": DOCUMENTARY_MATCH_POLICY,
        "TARGET_SEMANTICS_POLICY": TARGET_SEMANTICS_POLICY,
        "COMPONENT_LINK_POLICY": COMPONENT_LINK_POLICY,
        "QUALITY_WORKFLOW_POLICY": QUALITY_WORKFLOW_POLICY,
        "SCOPE_CLASSIFICATION_POLICY": SCOPE_CLASSIFICATION_POLICY,
    }.items():
        if not isinstance(policy_value, str) or not policy_value.strip():
            raise ValueError(f"{policy_name} debe ser una cadena no vacía.")


validate_runtime_configuration()


def build_ollama_model(
    model_name: str,
    *,
    base_url: str = "http://localhost:11434/v1",
):
    if not PYDANTIC_AI_AVAILABLE:
        raise RuntimeError("pydantic-ai no está instalado en este entorno.")
    return OllamaModel(
        model_name,
        provider=OllamaProvider(base_url=base_url),
    )


def build_boe_extraction_agent():
    if not PYDANTIC_AI_AVAILABLE:
        return None
    model = (
        AI_MODEL_NAME
        if MODEL_PROVIDER == "gemini"
        else build_ollama_model(AI_MODEL_NAME)
    )
    output_type = (
        NativeOutput(BOEAIExtraction)
        if USE_NATIVE_OUTPUT
        else BOEAIExtraction
    )
    return Agent(
        model=model,
        output_type=output_type,
        instructions=AGENT_INSTRUCTIONS,
        retries=AGENT_RETRIES,
    )


agent = build_boe_extraction_agent()


def _stable_json_hash(value: Any) -> str:
    serialized = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return sha256(serialized.encode("utf-8")).hexdigest()


CONTRACT_SCHEMA_SHA256 = _stable_json_hash(BOEAIExtraction.model_json_schema())
INSTRUCTIONS_SHA256 = sha256(AGENT_INSTRUCTIONS.encode("utf-8")).hexdigest()
EXTRACTION_CONFIG = {
    "model_provider": MODEL_PROVIDER,
    "model_name": AI_MODEL_NAME,
    "agent_retries": AGENT_RETRIES,
    "use_native_output": USE_NATIVE_OUTPUT,
    "model_settings": MODEL_SETTINGS,
    "document_validation_version": DOCUMENT_VALIDATION_VERSION,
    "model_run_timeout_seconds": MODEL_RUN_TIMEOUT_SECONDS,
    "entity_model_policy": ENTITY_MODEL_POLICY,
    "event_granularity_policy": EVENT_GRANULARITY_POLICY,
    "temporal_policy": TEMPORAL_POLICY,
    "documentary_match_policy": DOCUMENTARY_MATCH_POLICY,
    "target_semantics_policy": TARGET_SEMANTICS_POLICY,
    "component_link_policy": COMPONENT_LINK_POLICY,
    "quality_workflow_policy": QUALITY_WORKFLOW_POLICY,
    "scope_classification_policy": SCOPE_CLASSIFICATION_POLICY,
    "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
    "instructions_sha256": INSTRUCTIONS_SHA256,
}
EXTRACTION_CONFIG_ID = _stable_json_hash(EXTRACTION_CONFIG)[:16]
BOE_ID_ADAPTER = TypeAdapter(BOEId)

print(f"{DOCUMENT_VALIDATION_VERSION=}")
print(f"{EXTRACTION_CONFIG_ID=}")
print(f"{PYDANTIC_AI_AVAILABLE=}")

DOCUMENT_VALIDATION_VERSION='25'
EXTRACTION_CONFIG_ID='db2bc8c3564ce062'
PYDANTIC_AI_AVAILABLE=True


## 4. Fuente documental y persistencia

In [5]:
@dataclass(frozen=True)
class BOESourceDocument:
    boe_id: str
    publication_date: date
    title: str
    text: str
    source_document_sha256: str


@dataclass(frozen=True)
class SelectedDocumentText:
    text: str
    strategy: str
    marker: str | None
    excluded_chars: int
    text_sha256: str


@dataclass(frozen=True)
class PreparedDocumentPrompt:
    prompt: str
    input_text_chars: int
    input_text_sha256: str
    input_selection_strategy: str
    input_selection_marker: str | None
    input_excluded_chars: int


def _required_text(
    value: Any,
    *,
    field_name: str,
    strip_value: bool = True,
) -> str:
    if value is None or value is pd.NA or bool(pd.isna(value)):
        raise ValueError(f"{field_name} no puede ser nulo.")
    text = str(value)
    text = text.strip() if strip_value else text
    if not text.strip():
        raise ValueError(f"{field_name} no puede estar vacío.")
    return text


def _required_date(value: Any, *, field_name: str) -> date:
    parsed = pd.to_datetime(value, errors="coerce")
    if pd.isna(parsed):
        raise ValueError(f"{field_name} no contiene una fecha válida.")
    return parsed.date()


def _source_document_hash(
    *,
    boe_id: str,
    publication_date: date,
    title: str,
    text: str,
) -> str:
    value = "\n".join([boe_id, publication_date.isoformat(), title, text])
    return sha256(value.encode("utf-8")).hexdigest()


def build_source_document(row: pd.Series) -> BOESourceDocument:
    boe_id = BOE_ID_ADAPTER.validate_python(
        _required_text(row["identificador"], field_name="identificador")
    )
    publication_date = _required_date(
        row["fecha_publicacion"],
        field_name="fecha_publicacion",
    )
    title = _required_text(row["titulo"], field_name="titulo")
    text = _required_text(
        row["texto_limpio"],
        field_name="texto_limpio",
        strip_value=False,
    )
    return BOESourceDocument(
        boe_id=boe_id,
        publication_date=publication_date,
        title=title,
        text=text,
        source_document_sha256=_source_document_hash(
            boe_id=boe_id,
            publication_date=publication_date,
            title=title,
            text=text,
        ),
    )


def select_document_text(text: str) -> SelectedDocumentText:
    if not text.strip():
        raise ValueError("El texto documental no puede estar vacío.")

    selected = text
    strategy = "full_text"
    marker: str | None = None
    for match in _AFFECTED_ASSETS_ANNEX_HEADING_RE.finditer(text):
        if match.start() < MIN_SUBSTANTIVE_TEXT_CHARS_BEFORE_ANNEX:
            continue
        candidate = text[: match.start()].rstrip()
        if candidate:
            selected = candidate
            strategy = "before_affected_assets_annex"
            marker = match.group(0).strip()
            break

    if MAX_DOCUMENT_CHARS is not None and len(selected) > MAX_DOCUMENT_CHARS:
        raise ValueError(
            f"El documento contiene {len(selected):,} caracteres y supera "
            f"MAX_DOCUMENT_CHARS={MAX_DOCUMENT_CHARS:,}; no se truncará."
        )

    return SelectedDocumentText(
        text=selected,
        strategy=strategy,
        marker=marker,
        excluded_chars=len(text) - len(selected),
        text_sha256=sha256(selected.encode("utf-8")).hexdigest(),
    )


def build_document_prompt(document: BOESourceDocument) -> PreparedDocumentPrompt:
    selected = select_document_text(document.text)
    prompt = DOCUMENT_PROMPT_TEMPLATE.format(
        boe_id=document.boe_id,
        publication_date=document.publication_date.isoformat(),
        title=document.title,
        input_selection_strategy=selected.strategy,
        document_text=selected.text,
    )
    return PreparedDocumentPrompt(
        prompt=prompt,
        input_text_chars=len(selected.text),
        input_text_sha256=selected.text_sha256,
        input_selection_strategy=selected.strategy,
        input_selection_marker=selected.marker,
        input_excluded_chars=selected.excluded_chars,
    )


_REQUIRED_INPUT_COLUMNS = {
    "identificador",
    "fecha_publicacion",
    "titulo",
    "xml_status",
    "texto_limpio",
}


def load_and_prepare_candidates(
    input_path: Path = BOE_CANDIDATES_DOCS_TEXT_PATH,
) -> pd.DataFrame:
    candidates = pd.read_parquet(input_path)
    validate_required_columns(candidates, _REQUIRED_INPUT_COLUMNS)
    candidates = candidates.loc[
        candidates["xml_status"].eq("ok")
        & candidates["texto_limpio"].notna()
        & candidates["texto_limpio"].astype("string").str.strip().ne("")
    ].copy()
    candidates["fecha_publicacion"] = pd.to_datetime(
        candidates["fecha_publicacion"], errors="coerce"
    )
    if candidates["fecha_publicacion"].isna().any():
        raise ValueError("Existen candidatos sin fecha_publicacion válida.")
    if candidates["identificador"].duplicated().any():
        duplicated = candidates.loc[
            candidates["identificador"].duplicated(keep=False),
            "identificador",
        ].astype(str).unique().tolist()
        raise ValueError(f"Identificadores BOE duplicados: {duplicated[:20]}")
    candidates["source_document_sha256"] = candidates.apply(
        lambda row: build_source_document(row).source_document_sha256,
        axis=1,
    )
    return candidates


AI_EXTRACTION_LOG_COLUMNS = [
    "attempt_id",
    "identificador_boe",
    "fecha_publicacion",
    "titulo",
    "source_document_sha256",
    "input_text_chars",
    "input_text_sha256",
    "input_selection_strategy",
    "input_selection_marker",
    "input_excluded_chars",
    "extraction_config_id",
    "contract_schema_sha256",
    "instructions_sha256",
    "model_provider",
    "model_name",
    "document_validation_version",
    "classification_status",
    "document_scope",
    "classification_reason",
    "n_publication_events",
    "n_generation_assets",
    "n_associated_components",
    "n_administrative_actions",
    "n_participants",
    "n_administrative_locations",
    "n_generation_relations",
    "n_technical_mentions",
    "extraction_json",
    "extracted_at",
    "duration_seconds",
    "usage_requests",
    "usage_input_tokens",
    "usage_output_tokens",
    "usage_total_tokens",
    "extraction_status",
    "error_type",
    "error_message",
    "processing_stage",
    "document_validation_status",
    "document_validation_issue_count",
    "validation_issues_json",
    "deterministic_adjustment_count",
    "deterministic_adjustments_json",
]


def empty_ai_extraction_attempts_log() -> pd.DataFrame:
    return pd.DataFrame(columns=AI_EXTRACTION_LOG_COLUMNS)


def normalise_ai_extraction_attempts_log(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Añade el contrato vigente sin eliminar columnas históricas."""

    dataframe = dataframe.copy()
    for column in AI_EXTRACTION_LOG_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA
    extra_columns = [
        column
        for column in dataframe.columns
        if column not in AI_EXTRACTION_LOG_COLUMNS
    ]
    return dataframe[AI_EXTRACTION_LOG_COLUMNS + extra_columns]


def save_parquet_atomic(dataframe: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    dataframe.to_parquet(temporary, index=False)
    temporary.replace(output_path)


def load_ai_extraction_attempts(
    path: Path = BOE_AI_EXTRACTION_ATTEMPTS_PATH,
) -> pd.DataFrame:
    if not path.exists():
        return empty_ai_extraction_attempts_log()
    return normalise_ai_extraction_attempts_log(pd.read_parquet(path))


def combine_ai_extraction_attempt_frames(
    existing: pd.DataFrame,
    new_attempts: pd.DataFrame,
) -> pd.DataFrame:
    """Combina lotes de intentos sin depender de ``pd.concat``.

    Los registros correctos tienen varias columnas completamente nulas
    (por ejemplo ``error_type`` y ``validation_issues_json``). Pandas 2.x
    emite un ``FutureWarning`` al concatenar repetidamente esos bloques y
    anuncia un cambio futuro en la inferencia de tipos. Reconstruir la tabla
    desde registros evita esa inferencia ambigua y preserva todas las
    columnas históricas.
    """

    existing = normalise_ai_extraction_attempts_log(existing)
    new_attempts = normalise_ai_extraction_attempts_log(new_attempts)

    if existing.empty:
        return new_attempts.copy()
    if new_attempts.empty:
        return existing.copy()

    column_order = list(
        dict.fromkeys([*existing.columns.tolist(), *new_attempts.columns.tolist()])
    )
    records = [
        *existing.to_dict(orient="records"),
        *new_attempts.to_dict(orient="records"),
    ]
    combined = pd.DataFrame.from_records(records, columns=column_order)
    return normalise_ai_extraction_attempts_log(combined)


def append_ai_extraction_attempts(
    new_attempts: pd.DataFrame,
    path: Path = BOE_AI_EXTRACTION_ATTEMPTS_PATH,
) -> pd.DataFrame:
    existing = load_ai_extraction_attempts(path)
    combined = combine_ai_extraction_attempt_frames(existing, new_attempts)
    combined = combined.drop_duplicates(subset=["attempt_id"], keep="last")
    save_parquet_atomic(combined, path)
    return combined


def current_successful_ai_extractions(
    attempts: pd.DataFrame,
    source_df: pd.DataFrame,
) -> pd.DataFrame:
    attempts = normalise_ai_extraction_attempts_log(attempts)
    successful = attempts.loc[
        attempts["extraction_status"].eq("ok").fillna(False)
        & attempts["document_validation_status"].eq("passed").fillna(False)
        & attempts["document_validation_version"]
        .eq(DOCUMENT_VALIDATION_VERSION)
        .fillna(False)
        & attempts["extraction_config_id"].eq(EXTRACTION_CONFIG_ID).fillna(False)
    ].copy()
    sources = source_df[
        ["identificador", "source_document_sha256"]
    ].rename(columns={"identificador": "identificador_boe"})
    successful = successful.merge(
        sources,
        on=["identificador_boe", "source_document_sha256"],
        how="inner",
        validate="many_to_one",
    )
    if successful.empty:
        return normalise_ai_extraction_attempts_log(successful)
    current = (
        successful.sort_values(
            ["extracted_at", "attempt_id"],
            na_position="first",
            kind="stable",
        )
        .drop_duplicates("identificador_boe", keep="last")
        .reset_index(drop=True)
    )
    for row in current.itertuples(index=False):
        BOEProjectExtraction.model_validate_json(str(row.extraction_json))
    return normalise_ai_extraction_attempts_log(current)


def build_pending_candidates(
    source_df: pd.DataFrame,
    attempts: pd.DataFrame,
    manual_reviews: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if manual_reviews is None:
        current = current_successful_ai_extractions(attempts, source_df)
    else:
        current = select_best_valid_extractions(
            attempts=attempts,
            source_df=source_df,
            manual_reviews=manual_reviews,
        )
    processed = set(current["identificador_boe"].astype(str))
    pending = source_df.loc[
        ~source_df["identificador"].astype(str).isin(processed)
    ].copy()
    return current, pending

## 5. Canonicalización y validación documental

In [6]:
class DocumentExtractionValidationError(ValueError):
    def __init__(self, issues: list[str]) -> None:
        self.issues = issues
        preview = "; ".join(issues[:10])
        if len(issues) > 10:
            preview += f"; ... ({len(issues) - 10} adicionales)"
        super().__init__(preview)


# -----------------------------------------------------------------------------
# Normalización documental y búsqueda de citas literales
# -----------------------------------------------------------------------------


def _canonical_documentary_text(value: str) -> str:
    value = unicodedata.normalize("NFKC", str(value))
    replacements = {
        "\u00a0": " ",
        "«": '"',
        "»": '"',
        "“": '"',
        "”": '"',
        "’": "'",
        "–": "-",
        "—": "-",
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def _documentary_contains(needle: str, haystack: str) -> bool:
    canonical_needle = _canonical_documentary_text(needle).casefold()
    canonical_haystack = _canonical_documentary_text(haystack).casefold()
    return bool(canonical_needle) and canonical_needle in canonical_haystack


def _evidence_segments(evidence: str) -> list[str]:
    return [
        segment.strip()
        for segment in re.split(r"\s*\[\.\.\.\]\s*", str(evidence))
        if segment.strip()
    ]


def _evidence_is_supported(evidence: str, source_text: str) -> bool:
    segments = _evidence_segments(evidence)
    return bool(segments) and all(
        _documentary_contains(segment, source_text) for segment in segments
    )


def _source_units(source_text: str, document_title: str | None = None) -> list[str]:
    """Devuelve fragmentos literales de la fuente, del más informativo al más local."""

    candidates: list[str] = []
    if document_title and document_title.strip():
        candidates.append(document_title.strip())

    for line in str(source_text).splitlines():
        line = line.strip()
        if line:
            candidates.append(line)

    # Las frases ayudan a reparar una evidence demasiado extensa o reconstruida.
    for part in re.split(r"(?<=[.;:])\s+|\n+", str(source_text)):
        part = part.strip()
        if part:
            candidates.append(part)

    result: list[str] = []
    seen: set[str] = set()
    for candidate in candidates:
        key = _canonical_documentary_text(candidate).casefold()
        if key and key not in seen:
            seen.add(key)
            result.append(candidate)
    return result


def _find_literal_span(
    source_text: str,
    *,
    anchors: list[str] | tuple[str, ...] = (),
    semantic_patterns: list[str] | tuple[str, ...] = (),
    document_title: str | None = None,
) -> str | None:
    """Selecciona el fragmento literal más corto que contiene los anclajes."""

    canonical_anchors = [
        _canonical_documentary_text(anchor).casefold()
        for anchor in anchors
        if anchor and _canonical_documentary_text(anchor)
    ]
    compiled = [re.compile(pattern, re.IGNORECASE) for pattern in semantic_patterns]

    candidates: list[tuple[int, int, str]] = []
    for position, unit in enumerate(_source_units(source_text, document_title)):
        key = _canonical_documentary_text(unit).casefold()
        if canonical_anchors and not all(anchor in key for anchor in canonical_anchors):
            continue
        if compiled and not all(pattern.search(key) for pattern in compiled):
            continue
        candidates.append((len(key), position, unit))

    if candidates:
        return min(candidates, key=lambda item: (item[0], item[1]))[2]

    # Segundo intento: basta con un anclaje y todos los patrones semánticos.
    if canonical_anchors:
        relaxed: list[tuple[int, int, str]] = []
        for position, unit in enumerate(_source_units(source_text, document_title)):
            key = _canonical_documentary_text(unit).casefold()
            if not any(anchor in key for anchor in canonical_anchors):
                continue
            if compiled and not all(pattern.search(key) for pattern in compiled):
                continue
            relaxed.append((len(key), position, unit))
        if relaxed:
            return min(relaxed, key=lambda item: (item[0], item[1]))[2]

    return None


def _repair_evidence(
    evidence: str,
    *,
    source_text: str,
    anchors: list[str] | tuple[str, ...] = (),
    semantic_patterns: list[str] | tuple[str, ...] = (),
    document_title: str | None = None,
) -> str | None:
    if _evidence_is_supported(evidence, source_text):
        if not anchors or all(_documentary_contains(anchor, evidence) for anchor in anchors):
            return evidence
    return _find_literal_span(
        source_text,
        anchors=anchors,
        semantic_patterns=semantic_patterns,
        document_title=document_title,
    )


# -----------------------------------------------------------------------------
# Taxonomía semántica mínima
# -----------------------------------------------------------------------------


def _dedupe_technical_mentions(
    mentions: list[TechnicalMention],
) -> list[TechnicalMention]:
    result: list[TechnicalMention] = []
    seen: set[tuple[str, str]] = set()
    for mention in mentions:
        key = (mention.attribute_type.value, _text_key(mention.value_raw))
        if key not in seen:
            seen.add(key)
            result.append(mention)
    return result


def _infer_generation_type(text: str) -> GenerationType | None:
    key = _canonical_documentary_text(text).casefold()
    patterns = {
        GenerationType.WIND: r"\be[oó]lic|aerogenerador|parque\s+e[oó]lico",
        GenerationType.PHOTOVOLTAIC: r"fotovolta|\bplanta\s+solar\b|\bpsfv\b|\bfv\b",
        GenerationType.CONCENTRATED_SOLAR_POWER: r"termosolar|solar\s+termoel[eé]ctr",
        GenerationType.HYDROPOWER: r"hidroel[eé]ctr|central\s+hidr[aá]ul",
        GenerationType.BIOMASS: r"biomasa",
        GenerationType.BIOGAS: r"biog[aá]s",
        GenerationType.GEOTHERMAL: r"geot[eé]rm",
    }
    matches = [item for item, pattern in patterns.items() if re.search(pattern, key)]
    return matches[0] if len(matches) == 1 else None


_GENERATION_DESCRIPTOR_PATTERN = (
    r"(?:"
    r"(?:(?:la|el)\s+)?(?:planta|parque|central|instalaci[oó]n|proyecto)"
    r"(?:\s+(?:de\s+generaci[oó]n(?:\s+de\s+energ[ií]a\s+el[eé]ctrica)?|"
    r"solar|fotovoltaic[ao]s?|e[oó]lic[ao]s?|termosolar|"
    r"hidroel[eé]ctric[ao]s?|h[ií]brid[ao]s?))*"
    r"|panel(?:es)?\s+fotovoltaic[ao]s?(?:\s+flotantes?)?"
    r"|m[oó]dulos?\s+fotovoltaic[ao]s?"
    r"|aerogeneradores?"
    r"|aprovechamiento\s+hidroel[eé]ctrico"
    r")"
)

_COMPONENT_PATTERNS: dict[AssociatedComponentType, str] = {
    AssociatedComponentType.ENERGY_STORAGE: (
        r"(?:m[oó]dulo|sistema|instalaci[oó]n)?\s*(?:de\s+)?"
        r"(?:almacenamiento|bater[ií]as|bess)"
    ),
    AssociatedComponentType.EVACUATION_SYSTEM: (
        r"infraestructuras?\s+(?:el[eé]ctricas?\s+)?de\s+evacuaci[oó]n|"
        r"sistema\s+de\s+evacuaci[oó]n"
    ),
    AssociatedComponentType.ELECTRICAL_SUBSTATION: (
        r"subestaci[oó]n(?:\s+el[eé]ctrica)?|\bset\b"
    ),
    AssociatedComponentType.POWER_LINE: (
        r"l[ií]nea(?:\s+el[eé]ctrica)?|l[ií]nea\s+de\s+evacuaci[oó]n"
    ),
    AssociatedComponentType.GRID_CONNECTION: (
        r"(?:punto|posici[oó]n|permiso)\s+de\s+(?:acceso\s+y\s+)?conexi[oó]n"
    ),
}

_AUXILIARY_COMPONENT_TYPES = {
    AssociatedComponentType.EVACUATION_SYSTEM,
    AssociatedComponentType.ELECTRICAL_SUBSTATION,
    AssociatedComponentType.POWER_LINE,
    AssociatedComponentType.GRID_CONNECTION,
    AssociatedComponentType.OTHER_ASSOCIATED_COMPONENT,
}


def _component_pattern(component_type: AssociatedComponentType) -> str:
    if component_type in _AUXILIARY_COMPONENT_TYPES:
        return (
            r"infraestructuras?\s+(?:el[eé]ctricas?\s+)?de\s+evacuaci[oó]n|"
            r"sistema\s+de\s+evacuaci[oó]n|l[ií]nea(?:\s+el[eé]ctrica)?|"
            r"subestaci[oó]n(?:\s+el[eé]ctrica)?|\bset\b|"
            r"(?:punto|posici[oó]n|permiso)\s+de\s+(?:acceso\s+y\s+)?conexi[oó]n"
        )
    return _COMPONENT_PATTERNS[component_type]


def _generation_name_is_direct_target(name: str, text: str) -> bool:
    """Distingue el destinatario jurídico de una referencia contextual.

    Un nombre de planta contenido en expresiones como «infraestructura de
    evacuación de la planta X» o «almacenamiento para su hibridación con X»
    identifica la planta relacionada, pero no implica que la actuación recaiga
    también sobre ella.
    """

    canonical_text = _canonical_documentary_text(text).casefold()
    canonical_name = _canonical_documentary_text(name).casefold()
    if not canonical_name:
        return False

    component_or_integration_pattern = (
        rf"(?:{_component_pattern(AssociatedComponentType.EVACUATION_SYSTEM)}|"
        rf"{_component_pattern(AssociatedComponentType.ENERGY_STORAGE)})"
    )
    contextual_link_pattern = re.compile(
        r"(?:"
        r"\bde(?:l|\s+la)?\b|"
        r"\basociad[ao]s?\s+a(?:l|\s+la)?\b|"
        r"\bpara\s+su\s+hibridaci[oó]n\s+con\b|"
        r"\bhibridaci[oó]n\s+(?:de|con)\b"
        r")",
        re.IGNORECASE,
    )

    start = 0
    found_neutral = False
    while True:
        index = canonical_text.find(canonical_name, start)
        if index < 0:
            break

        before = canonical_text[max(0, index - 360):index]
        around = canonical_text[
            max(0, index - 160):index + len(canonical_name) + 100
        ]

        component_matches = list(
            re.finditer(component_or_integration_pattern, before, re.IGNORECASE)
        )
        is_contextual = False
        if component_matches:
            last_component = component_matches[-1]
            link_text = before[last_component.end():]
            if len(link_text) <= 280 and contextual_link_pattern.search(link_text):
                is_contextual = True

        if is_contextual:
            start = index + len(canonical_name)
            continue

        if re.search(_GENERATION_DESCRIPTOR_PATTERN, around, re.IGNORECASE):
            return True

        found_neutral = True
        start = index + len(canonical_name)

    return found_neutral


def _generation_refs_mentioned(
    event: PublicationEvent,
    text: str,
    *,
    direct_only: bool = False,
) -> list[str]:
    refs: list[str] = []
    for asset in event.generation_assets:
        for name in asset.names_raw:
            matches = (
                _generation_name_is_direct_target(name, text)
                if direct_only
                else _documentary_contains(name, text)
            )
            if matches:
                refs.append(asset.local_generation_asset_ref)
                break
    return _canonicalize_refs(refs)


def _component_refs_mentioned(event: PublicationEvent, text: str) -> list[str]:
    canonical = _canonical_documentary_text(text).casefold()
    refs: list[str] = []
    by_type: dict[AssociatedComponentType, list[AssociatedComponent]] = {}
    for component in event.associated_components:
        by_type.setdefault(component.component_type, []).append(component)
        if any(_documentary_contains(name, text) for name in component.names_raw):
            refs.append(component.local_component_ref)
            continue
        if component.description_raw and _documentary_contains(component.description_raw, text):
            refs.append(component.local_component_ref)

    for component_type, components in by_type.items():
        if len(components) == 1 and re.search(_component_pattern(component_type), canonical):
            refs.append(components[0].local_component_ref)
    return _canonicalize_refs(refs)


def _entity_refs_mentioned(event: PublicationEvent, text: str) -> list[str]:
    return _canonicalize_refs(
        _generation_refs_mentioned(event, text, direct_only=True)
        + _component_refs_mentioned(event, text)
    )



# -----------------------------------------------------------------------------
# Alcance operativo del TFM
# -----------------------------------------------------------------------------

class ScopeGuardDecision(str, Enum):
    FORCE_NOT_RELEVANT = "force_not_relevant"
    REQUIRE_PROJECT_REVIEW = "require_project_review"
    NO_DETERMINISTIC_DECISION = "no_deterministic_decision"


_PROCUREMENT_TITLE_RE = re.compile(
    r"formalizaci[oó]n\s+de\s+contratos?|anuncio\s+de\s+(?:licitaci[oó]n|"
    r"adjudicaci[oó]n)|contrataci[oó]n\s+del?\s+(?:suministro|servicio|obra)|"
    r"contrato\s+de\s+suministro",
    re.IGNORECASE,
)

_NON_GENERATION_MAIN_OBJECT_RE = re.compile(
    r"abastecimiento\s+de\s+agua|mejora\s+del\s+abastecimiento|"
    r"estaci[oó]n\s+depuradora|\bedar\b|saneamiento|depuraci[oó]n\s+de\s+aguas|"
    r"modernizaci[oó]n\s+de\s+regad[ií]os?|comunidad\s+de\s+regantes|"
    r"equipos?\s+de\s+bombeo|obras?\s+de\s+regad[ií]o|"
    r"proyecto\s+de\s+construcci[oó]n\s+de\s+(?:carretera|ferrocarril)|"
    r"vertederos?\s+asociados?\s+al\s+proyecto\s+de\s+construcci[oó]n",
    re.IGNORECASE,
)

_EXPLICIT_ELECTRIC_GENERATION_RE = re.compile(
    r"(?:planta|parque|central|instalaci[oó]n|m[oó]dulo\s+de\s+generaci[oó]n)"
    r"[^.;]{0,180}(?:fotovolta|solar|e[oó]lic|hidroel[eé]ct|termosolar|biomasa|"
    r"biog[aá]s|generaci[oó]n\s+de\s+energ[ií]a\s+el[eé]ctrica)|"
    r"(?:fotovolta|e[oó]lic|hidroel[eé]ct|termosolar)[^.;]{0,100}"
    r"(?:planta|parque|central|instalaci[oó]n)|"
    r"(?:aprovechamiento|central)[^.;]{0,100}producci[oó]n\s+de\s+energ[ií]a\s+el[eé]ctrica|"
    r"\b(?:psfv|pfv|fv|pe)\s+[A-ZÁÉÍÓÚÑ][\wÁÉÍÓÚÜÑáéíóúüñ'-]+",
    re.IGNORECASE,
)

_AUXILIARY_RENEWABLE_RE = re.compile(
    r"paneles?\s+fotovoltaicos?|placas?\s+fotovoltaicas?|"
    r"instalaci[oó]n\s+solar\s+fotovoltaica|energ[ií]as?\s+renovables?",
    re.IGNORECASE,
)


_NON_ELECTRIC_GAS_INFRASTRUCTURE_RE = re.compile(
    r"\bgasoductos?\b|instalaciones?\s+gasistas?|"
    r"estaci[oó]n\s+(?:de\s+)?(?:medida|regulaci[oó]n|compresi[oó]n)|"
    r"posici[oó]n\s+[A-Z0-9.-]+[^.;]{0,120}(?:gas|biometano)|"
    r"inyecci[oó]n\s+de\s+(?:biometano|hidr[oó]geno)[^.;]{0,120}"
    r"(?:gasoducto|red\s+gasista|red\s+de\s+gas)",
    re.IGNORECASE,
)


_STANDALONE_STORAGE_MAIN_OBJECT_RE = re.compile(
    r"(?:planta|sistema|instalaci[oó]n|m[oó]dulo)\s+de\s+"
    r"almacenamiento(?:\s+de\s+energ[ií]a)?|"
    r"\b(?:bess|sistema\s+de\s+bater[ií]as)\b",
    re.IGNORECASE,
)

_STORAGE_LINKED_TO_GENERATION_RE = re.compile(
    r"hibridaci[oó]n|hibridad[oa]|hibrida\s+con|"
    r"(?:asociad[oa]|vinculad[oa]|integrado|incorporado)\s+(?:a|al|en)\s+"
    r"(?:la\s+|el\s+)?(?:planta|parque|central|instalaci[oó]n)[^.;]{0,160}"
    r"(?:fotovolta|solar|e[oó]lic|hidroel[eé]ct|termosolar|biomasa|biog[aá]s)|"
    r"(?:planta|parque|central|instalaci[oó]n)[^.;]{0,160}"
    r"(?:fotovolta|e[oó]lic|hidroel[eé]ct|termosolar|biomasa|biog[aá]s)",
    re.IGNORECASE,
)


def _scope_guard_from_document(
    *,
    document_title: str,
    source_text: str,
) -> tuple[ScopeGuardDecision, str | None]:
    """Aplica solo decisiones de alcance de alta precisión.

    No determina la categoría de todos los BOE. Su objetivo es impedir dos errores
    graves: convertir contratación/obras sectoriales en plantas y aceptar como no
    relevante un título que identifica inequívocamente una planta y un acto actual.
    """

    title = _canonical_documentary_text(document_title)
    source = _canonical_documentary_text(source_text)

    if _PROCUREMENT_TITLE_RE.search(title):
        return (
            ScopeGuardDecision.FORCE_NOT_RELEVANT,
            "Anuncio de contratación pública; no constituye un acto administrativo del ciclo de una planta de generación.",
        )

    explicit_generation_in_title = bool(_EXPLICIT_ELECTRIC_GENERATION_RE.search(title))
    non_generation_main_object = bool(_NON_GENERATION_MAIN_OBJECT_RE.search(title))
    non_electric_gas_infrastructure = bool(
        _NON_ELECTRIC_GAS_INFRASTRUCTURE_RE.search(title)
    )
    standalone_storage_main_object = bool(
        _STANDALONE_STORAGE_MAIN_OBJECT_RE.search(title)
    )
    storage_linked_to_generation = bool(
        _STORAGE_LINKED_TO_GENERATION_RE.search(title)
    )
    auxiliary_renewable = bool(_AUXILIARY_RENEWABLE_RE.search(source))
    explicit_hydroelectric_use = bool(re.search(
        r"producci[oó]n\s+de\s+energ[ií]a\s+el[eé]ctrica|aprovechamiento\s+hidroel[eé]ctrico",
        title,
        re.IGNORECASE,
    ))

    if non_electric_gas_infrastructure and not explicit_generation_in_title:
        return (
            ScopeGuardDecision.FORCE_NOT_RELEVANT,
            "Infraestructura gasista o de inyección a la red de gas sin una planta de generación eléctrica identificable.",
        )

    if (
        standalone_storage_main_object
        and not storage_linked_to_generation
        and not explicit_generation_in_title
    ):
        return (
            ScopeGuardDecision.FORCE_NOT_RELEVANT,
            "Instalación autónoma de almacenamiento: el objeto actual no identifica una planta de generación eléctrica asociada.",
        )

    if (
        non_generation_main_object
        and not explicit_generation_in_title
        and not explicit_hydroelectric_use
    ):
        detail = (
            "La generación aparece como elemento auxiliar de un proyecto sectorial cuyo objeto principal no es una planta de generación."
            if auxiliary_renewable
            else "El objeto principal del título es un proyecto sectorial no perteneciente a la generación eléctrica."
        )
        return ScopeGuardDecision.FORCE_NOT_RELEVANT, detail

    title_actions = _action_types_from_title(document_title)
    if explicit_generation_in_title and title_actions:
        return (
            ScopeGuardDecision.REQUIRE_PROJECT_REVIEW,
            "El título identifica una planta de generación y una actuación administrativa actual.",
        )

    return ScopeGuardDecision.NO_DETERMINISTIC_DECISION, None


def _force_non_relevant_extraction(
    extraction: BOEProjectExtraction,
    *,
    reason: str,
) -> BOEProjectExtraction:
    extraction = extraction.model_copy(deep=True)
    extraction.classification_status = ClassificationStatus.CLASSIFIED
    extraction.document_scope = DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
    extraction.classification_reason = reason
    extraction.publication_events = []
    extraction.extraction_notes = None
    return BOEProjectExtraction.model_validate(extraction.model_dump())


def preclassify_document_without_model(
    document: BOESourceDocument,
) -> tuple[BOEProjectExtraction | None, list[str]]:
    """Descarta antes de la IA solo documentos inequívocamente fuera de alcance.

    Devuelve ``None`` cuando la decisión requiere comprensión generativa. Esta
    función no intenta clasificar todos los BOE y prioriza evitar falsos negativos.
    """

    decision, reason = _scope_guard_from_document(
        document_title=document.title,
        source_text=f"{document.title}\n{document.text}",
    )
    if decision != ScopeGuardDecision.FORCE_NOT_RELEVANT:
        return None, []

    resolved_reason = reason or "Documento inequívocamente fuera del alcance del TFM."
    extraction = BOEProjectExtraction(
        classification_status=ClassificationStatus.CLASSIFIED,
        document_scope=DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS,
        classification_reason=resolved_reason,
        publication_events=[],
        extraction_notes=None,
        boe_id=document.boe_id,
        publication_date=document.publication_date,
    )
    return extraction, [
        "Alcance resuelto antes de llamar al modelo: " + resolved_reason
    ]

# -----------------------------------------------------------------------------
# Actuaciones actuales expresadas en el título
# -----------------------------------------------------------------------------


_ACTION_PATTERNS: dict[AdministrativeActionType, str] = {
    AdministrativeActionType.ERROR_CORRECTION: r"correcci[oó]n\s+de\s+errores",
    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION: (
        r"autorizaci[oó]n\s+administrativa\s+previa|\baap\b"
    ),
    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION: (
        r"autorizaci[oó]n\s+administrativa\s+(?:de\s+)?construcci[oó]n|autorizaci[oó]n\s+(?:de\s+)?construcci[oó]n|autoriza[^.;]{0,50}construcci[oó]n|\baac\b"
    ),
    AdministrativeActionType.OPERATING_AUTHORIZATION: (
        r"autorizaci[oó]n\s+(?:de\s+)?explotaci[oó]n"
    ),
    AdministrativeActionType.WATER_CONCESSION: (
        r"(?:otorga|concede|solicitud\s+de)\s+(?:la\s+)?concesi[oó]n|"
        r"concesi[oó]n\s+para\s+el\s+aprovechamiento"
    ),
    AdministrativeActionType.PUBLIC_UTILITY_DECLARATION: (
        r"declaraci[oó]n(?:,\s*en\s+concreto,)?\s+de\s+utilidad\s+p[uú]blica|\bdup\b"
    ),
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT: (
        r"declaraci[oó]n\s+de\s+impacto\s+ambiental|\bdia\b"
    ),
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT: (
        r"informe\s+de\s+impacto\s+ambiental"
    ),
    AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT: (
        r"informe\s+de\s+determinaci[oó]n\s+de\s+afecci[oó]n\s+ambiental|\bidaa\b"
    ),
    AdministrativeActionType.PRIOR_OCCUPATION_RECORDS: (
        r"levantamiento\s+de\s+actas\s+previas\s+a\s+la\s+ocupaci[oó]n"
    ),
    AdministrativeActionType.OCCUPATION_RECORDS: r"actas\s+de\s+ocupaci[oó]n",
    AdministrativeActionType.AFFECTED_ASSETS_AND_RIGHTS_LIST: (
        r"relaci[oó]n(?:\s+concreta\s+e\s+individualizada)?\s+de\s+bienes\s+y\s+derechos\s+afectados"
    ),
    AdministrativeActionType.OWNERSHIP_CHANGE: r"cambio\s+de\s+titularidad|transmisi[oó]n\s+de\s+titularidad",
    AdministrativeActionType.DEADLINE_EXTENSION: r"pr[oó]rroga",
    AdministrativeActionType.PROCEDURE_TERMINATION: r"archivo|desistimiento|inadmisi[oó]n",
}


def _action_types_from_title(title: str) -> set[AdministrativeActionType]:
    key = _canonical_documentary_text(title).casefold()
    result = {
        action_type
        for action_type, pattern in _ACTION_PATTERNS.items()
        if re.search(pattern, key)
    }

    # Una publicación de corrección no vuelve a publicar el acto corregido.
    if AdministrativeActionType.ERROR_CORRECTION in result:
        return {AdministrativeActionType.ERROR_CORRECTION}

    is_public_information = bool(re.search(
        r"(?:somete|sometimiento|anuncio)[^.;]{0,180}informaci[oó]n\s+p[uú]blica",
        key,
    ))
    if is_public_information:
        environmental_products = {
            AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
            AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
            AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT,
        }
        explicit_environmental_process = bool(re.search(
            r"estudio\s+de\s+impacto\s+ambiental|"
            r"evaluaci[oó]n\s+de\s+impacto\s+ambiental",
            key,
        ))
        if result & environmental_products or explicit_environmental_process:
            result -= environmental_products
            result.add(AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT)
        elif not result:
            result.add(AdministrativeActionType.PUBLIC_INFORMATION)
    return result


def _decision_from_title(
    title: str,
    action_type: AdministrativeActionType,
) -> AdministrativeDecision:
    key = _canonical_documentary_text(title).casefold()
    if re.search(r"informaci[oó]n\s+p[uú]blica", key):
        return AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
    if action_type == AdministrativeActionType.ERROR_CORRECTION:
        return AdministrativeDecision.RECTIFIED
    if action_type in {
        AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
        AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
        AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT,
    }:
        if re.search(r"desfavorable|no\s+favorable", key):
            return AdministrativeDecision.UNFAVORABLE
        if re.search(r"favorable", key):
            return AdministrativeDecision.FAVORABLE
        return AdministrativeDecision.FORMULATED
    if action_type in {
        AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
        AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
        AdministrativeActionType.OPERATING_AUTHORIZATION,
        AdministrativeActionType.OWNERSHIP_CHANGE,
    }:
        if re.search(r"deniega|denegaci[oó]n", key):
            return AdministrativeDecision.DENIED
        if re.search(r"solicitud|solicita", key) and not re.search(r"otorga|concede|autoriza", key):
            return AdministrativeDecision.REQUESTED
        return AdministrativeDecision.AUTHORIZED
    if action_type == AdministrativeActionType.WATER_CONCESSION:
        if re.search(r"deniega|denegaci[oó]n", key):
            return AdministrativeDecision.DENIED
        if re.search(r"informaci[oó]n\s+p[uú]blica", key):
            return AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
        if re.search(r"solicitud|solicita", key) and not re.search(r"otorga|concede", key):
            return AdministrativeDecision.REQUESTED
        return AdministrativeDecision.AUTHORIZED
    if action_type == AdministrativeActionType.PUBLIC_UTILITY_DECLARATION:
        if re.search(r"solicitud|informaci[oó]n\s+p[uú]blica", key):
            return AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
        return AdministrativeDecision.DECLARED
    if action_type in {
        AdministrativeActionType.PRIOR_OCCUPATION_RECORDS,
        AdministrativeActionType.OCCUPATION_RECORDS,
        AdministrativeActionType.AFFECTED_ASSETS_AND_RIGHTS_LIST,
    }:
        return AdministrativeDecision.ANNOUNCED
    if action_type == AdministrativeActionType.DEADLINE_EXTENSION:
        return AdministrativeDecision.EXTENDED
    if action_type == AdministrativeActionType.PROCEDURE_TERMINATION:
        if re.search(r"desist", key):
            return AdministrativeDecision.WITHDRAWN
        if re.search(r"inadmis", key):
            return AdministrativeDecision.INADMISSIBLE
        return AdministrativeDecision.CLOSED
    return AdministrativeDecision.OTHER


_MODIFIABLE_ACTION_TYPES = {
    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
    AdministrativeActionType.OPERATING_AUTHORIZATION,
    AdministrativeActionType.PUBLIC_UTILITY_DECLARATION,
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
    AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT,
}


def _modification_expectation(
    action: AdministrativeAction,
    *,
    document_title: str,
) -> bool | None:
    if action.action_type not in _MODIFIABLE_ACTION_TYPES:
        return False
    key = _canonical_documentary_text(document_title).casefold()
    pattern = _ACTION_PATTERNS.get(action.action_type)
    if pattern is None or not re.search(pattern, key):
        return None

    coordinated_prior_only = re.search(
        r"autorizaci[oó]n\s+administrativa\s+previa[^.;]{0,100}"
        r"modificaci[oó]n(?:es)?[^.;]{0,100}\by\s+"
        r"autorizaci[oó]n\s+administrativa\s+(?:de\s+)?construcci[oó]n",
        key,
    )
    if coordinated_prior_only:
        if action.action_type == AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION:
            return True
        if action.action_type == AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION:
            construction_tail = key[coordinated_prior_only.end():]
            return bool(re.search(r"modificaci[oó]n|modificad[ao]|\bmodifica(?:n|da|do)?\b", construction_tail))

    # Busca «modificación» en la cláusula local del acto, no en todo el título.
    match = re.search(pattern, key)
    assert match is not None
    left = max(key.rfind(";", 0, match.start()), key.rfind(".", 0, match.start()))
    right_positions = [
        position
        for token in (";", ".")
        if (position := key.find(token, match.end())) >= 0
    ]
    right = min(right_positions) if right_positions else len(key)
    clause = key[left + 1:right]
    return bool(re.search(r"modificaci[oó]n|modificad[ao]|\bmodifica(?:n|da|do)?\b", clause))


def _is_clearly_historical(evidence: str) -> bool:
    key = _canonical_documentary_text(evidence).casefold()
    return bool(re.search(
        r"^(?:mediante|por)\s+(?:resoluci[oó]n|orden)|"
        r"^con\s+fecha\s+\d|^en\s+fecha\s+\d|"
        r"antecedentes?|previamente|con\s+anterioridad|"
        r"hab[ií]a\s+(?:sido|obtenido|solicitado|autorizado)",
        key,
    ))


# -----------------------------------------------------------------------------
# Canonicalización de entidades
# -----------------------------------------------------------------------------


def _repair_technical_mentions(
    mentions: list[TechnicalMention],
    *,
    source_text: str,
    document_title: str,
    storage: bool,
) -> tuple[list[TechnicalMention], list[str]]:
    repaired: list[TechnicalMention] = []
    adjustments: list[str] = []
    for mention in _dedupe_technical_mentions(mentions):
        if not _documentary_contains(mention.value_raw, source_text):
            adjustments.append(
                f"Magnitud opcional descartada por no ser literal: {mention.value_raw!r}."
            )
            continue
        evidence = _repair_evidence(
            mention.evidence,
            source_text=source_text,
            anchors=[mention.value_raw],
            document_title=document_title,
        )
        if evidence is None:
            adjustments.append(
                f"Magnitud opcional descartada por carecer de cita: {mention.value_raw!r}."
            )
            continue
        attribute_type = mention.attribute_type
        value_key = _canonical_documentary_text(mention.value_raw).casefold()
        if storage and attribute_type == TechnicalAttributeType.INSTALLED_POWER:
            attribute_type = (
                TechnicalAttributeType.STORAGE_CAPACITY
                if re.search(r"\b(?:mwh|kwh|gwh)\b", value_key)
                else TechnicalAttributeType.STORAGE_POWER
            )
        repaired.append(mention.model_copy(update={
            "attribute_type": attribute_type,
            "evidence": evidence,
        }))
    return _dedupe_technical_mentions(repaired), adjustments


def _salvage_generation_names(
    asset: GenerationAssetMention,
    *,
    source_text: str,
) -> tuple[list[str], list[str]]:
    names: list[str] = []
    adjustments: list[str] = []
    for name in _deduplicate_strings(asset.names_raw):
        if _documentary_contains(name, source_text):
            names.append(name)
            continue
        stripped = re.sub(
            r"\s+(?:e[oó]lic[oa]|fotovoltaic[oa]|solar|fv)$",
            "",
            name,
            flags=re.IGNORECASE,
        ).strip(" \"'«»")
        if stripped and _documentary_contains(stripped, source_text):
            names.append(stripped)
            adjustments.append(
                f"Nombre de planta reparado eliminando un sufijo no documental: {name!r}."
            )
        else:
            adjustments.append(f"Nombre de planta no documental descartado: {name!r}.")
    return _deduplicate_strings(names), adjustments


def _repair_generation_asset(
    asset: GenerationAssetMention,
    *,
    source_text: str,
    document_title: str,
) -> tuple[GenerationAssetMention | None, list[str]]:
    adjustments: list[str] = []
    names, current = _salvage_generation_names(asset, source_text=source_text)
    adjustments.extend(current)
    if not names:
        return None, adjustments + [
            f"{asset.local_generation_asset_ref}: planta descartada al no conservar un nombre literal."
        ]

    evidence = _repair_evidence(
        asset.evidence,
        source_text=source_text,
        anchors=[names[0]],
        semantic_patterns=[_GENERATION_DESCRIPTOR_PATTERN],
        document_title=document_title,
    )
    if evidence is None:
        evidence = _repair_evidence(
            asset.evidence,
            source_text=source_text,
            anchors=[names[0]],
            document_title=document_title,
        )
    if evidence is None:
        return None, adjustments + [
            f"{asset.local_generation_asset_ref}: planta descartada al no localizar cita literal."
        ]

    generation_type = asset.generation_type
    if generation_type == GenerationType.OTHER_GENERATION:
        inferred = _infer_generation_type(f"{evidence} {' '.join(names)}")
        if inferred is not None:
            generation_type = inferred
            adjustments.append(
                f"{asset.local_generation_asset_ref}: tipo de generación inferido como {inferred.value}."
            )

    mentions, current = _repair_technical_mentions(
        asset.technical_mentions,
        source_text=source_text,
        document_title=document_title,
        storage=False,
    )
    adjustments.extend(current)
    return asset.model_copy(update={
        "names_raw": names,
        "generation_type": generation_type,
        "technical_mentions": mentions,
        "evidence": evidence,
    }), adjustments


def _expand_multitechnology_generation_assets(
    event: PublicationEvent,
    *,
    source_text: str,
) -> tuple[PublicationEvent, list[str]]:
    """Desdobla una instalación híbrida condensada solo con evidencia inequívoca."""

    assets: list[GenerationAssetMention] = []
    ref_mapping: dict[str, list[str]] = {}
    adjustments: list[str] = []
    relation_evidence = _find_literal_span(
        source_text,
        semantic_patterns=[r"h[ií]brid"],
    )

    for asset in event.generation_assets:
        by_type: dict[GenerationType, list[TechnicalMention]] = {}
        for mention in asset.technical_mentions:
            inferred = _infer_generation_type(f"{mention.evidence} {mention.value_raw}")
            if inferred is not None:
                by_type.setdefault(inferred, []).append(mention)
        should_split = (
            asset.generation_type == GenerationType.OTHER_GENERATION
            and len(by_type) >= 2
            and relation_evidence is not None
        )
        if not should_split:
            new_ref = f"generation_asset_{len(assets) + 1}"
            assets.append(asset.model_copy(update={"local_generation_asset_ref": new_ref}))
            ref_mapping[asset.local_generation_asset_ref] = [new_ref]
            continue

        new_refs: list[str] = []
        for generation_type in sorted(by_type, key=lambda item: item.value):
            new_ref = f"generation_asset_{len(assets) + 1}"
            new_refs.append(new_ref)
            assets.append(asset.model_copy(update={
                "local_generation_asset_ref": new_ref,
                "generation_type": generation_type,
                "technical_mentions": by_type[generation_type],
            }))
        ref_mapping[asset.local_generation_asset_ref] = new_refs
        adjustments.append(
            f"{asset.local_generation_asset_ref}: instalación híbrida multitecnología desdoblada en {new_refs}."
        )

    if not any(len(values) > 1 for values in ref_mapping.values()):
        return event.model_copy(update={"generation_assets": assets}), adjustments

    components: list[AssociatedComponent] = []
    for component in event.associated_components:
        refs: list[str] = []
        for ref in component.related_generation_asset_refs:
            refs.extend(ref_mapping.get(ref, [ref]))
        components.append(component.model_copy(update={
            "related_generation_asset_refs": _canonicalize_refs(refs),
        }))

    actions: list[AdministrativeAction] = []
    for action in event.administrative_actions:
        targets: list[str] = []
        for target in action.targets:
            if target.startswith("generation_asset_"):
                targets.extend(ref_mapping.get(target, [target]))
            else:
                targets.append(target)
        actions.append(action.model_copy(update={"targets": _canonicalize_refs(targets)}))

    relations = list(event.generation_relations)
    for old_ref, refs in ref_mapping.items():
        if len(refs) < 2 or relation_evidence is None:
            continue
        for index, source in enumerate(refs):
            for target in refs[index + 1:]:
                relations.append(GenerationAssetRelation(
                    source_generation_asset_ref=source,
                    target_generation_asset_ref=target,
                    relation_type=GenerationRelationType.HYBRIDIZED_WITH,
                    evidence=relation_evidence,
                ))

    return event.model_copy(update={
        "generation_assets": assets,
        "associated_components": components,
        "administrative_actions": actions,
        "generation_relations": relations,
    }), adjustments


def _exact_component_description(
    component_type: AssociatedComponentType,
    *,
    source_text: str,
    evidence: str,
) -> str | None:
    pattern = _component_pattern(component_type)
    for text in (evidence, source_text):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0).strip(" ,.;:")
    return None


def _repair_component(
    component: AssociatedComponent,
    *,
    source_text: str,
    document_title: str,
) -> tuple[AssociatedComponent | None, list[str]]:
    adjustments: list[str] = []
    component_type = (
        AssociatedComponentType.EVACUATION_SYSTEM
        if component.component_type in _AUXILIARY_COMPONENT_TYPES
        else component.component_type
    )
    if component_type != component.component_type:
        adjustments.append(
            f"{component.local_component_ref}: infraestructura auxiliar agregada como sistema_evacuacion."
        )

    names = [
        name
        for name in _deduplicate_strings(component.names_raw)
        if _documentary_contains(name, source_text)
    ]
    if len(names) != len(component.names_raw):
        adjustments.append(
            f"{component.local_component_ref}: nombres no documentales del componente descartados."
        )

    pattern = _component_pattern(component_type)
    evidence = _repair_evidence(
        component.evidence,
        source_text=source_text,
        anchors=names[:1],
        semantic_patterns=[pattern],
        document_title=document_title,
    )
    if evidence is None:
        evidence = _find_literal_span(
            source_text,
            semantic_patterns=[pattern],
            document_title=document_title,
        )
    if evidence is None:
        return None, adjustments + [
            f"{component.local_component_ref}: componente opcional descartado al no localizar cita literal."
        ]

    description = component.description_raw
    if description and not _documentary_contains(description, source_text):
        description = None
    if description is None:
        description = _exact_component_description(
            component_type,
            source_text=source_text,
            evidence=evidence,
        )

    mentions, current = _repair_technical_mentions(
        component.technical_mentions,
        source_text=source_text,
        document_title=document_title,
        storage=component_type == AssociatedComponentType.ENERGY_STORAGE,
    )
    adjustments.extend(current)
    return component.model_copy(update={
        "component_type": component_type,
        "names_raw": names,
        "description_raw": description,
        "technical_mentions": mentions,
        "evidence": evidence,
    }), adjustments


def _merge_auxiliary_components(
    components: list[AssociatedComponent],
) -> tuple[list[AssociatedComponent], dict[str, str], list[str]]:
    """Agrega línea/subestación/conexión en un único sistema de evacuación."""

    storage: list[AssociatedComponent] = []
    auxiliary: list[AssociatedComponent] = []
    for component in components:
        if component.component_type == AssociatedComponentType.ENERGY_STORAGE:
            storage.append(component)
        else:
            auxiliary.append(component)

    merged: list[AssociatedComponent] = []
    mapping: dict[str, str] = {}
    adjustments: list[str] = []

    # Almacenamientos con nombre/evidencia distinta se conservan separados.
    for component in storage:
        new_ref = f"component_{len(merged) + 1}"
        mapping[component.local_component_ref] = new_ref
        merged.append(component.model_copy(update={"local_component_ref": new_ref}))

    if auxiliary:
        base = auxiliary[0]
        new_ref = f"component_{len(merged) + 1}"
        for component in auxiliary:
            mapping[component.local_component_ref] = new_ref
        names = _deduplicate_strings([
            name for component in auxiliary for name in component.names_raw
        ])
        related = _canonicalize_refs([
            ref
            for component in auxiliary
            for ref in component.related_generation_asset_refs
        ])
        technical = _dedupe_technical_mentions([
            mention
            for component in auxiliary
            for mention in component.technical_mentions
        ])
        evidence = min(
            (component.evidence for component in auxiliary),
            key=lambda value: len(_canonical_documentary_text(value)),
        )
        description = next(
            (component.description_raw for component in auxiliary if component.description_raw),
            None,
        )
        merged.append(base.model_copy(update={
            "local_component_ref": new_ref,
            "component_type": AssociatedComponentType.EVACUATION_SYSTEM,
            "names_raw": names,
            "description_raw": description,
            "related_generation_asset_refs": related,
            "technical_mentions": technical,
            "evidence": evidence,
        }))
        if len(auxiliary) > 1:
            adjustments.append(
                f"{len(auxiliary)} elementos auxiliares agregados en {new_ref} como sistema de evacuación."
            )

    return merged, mapping, adjustments


def _infer_component_links(
    event: PublicationEvent,
    *,
    source_text: str,
) -> tuple[PublicationEvent, list[str]]:
    event = event.model_copy(deep=True)
    all_refs = [asset.local_generation_asset_ref for asset in event.generation_assets]
    adjustments: list[str] = []
    hybrid_context = bool(re.search(
        r"h[ií]brid|incorporaci[oó]n\s+de\s+almacenamiento",
        _canonical_documentary_text(source_text).casefold(),
    ))

    for component in event.associated_components:
        valid = [ref for ref in component.related_generation_asset_refs if ref in all_refs]
        if len(all_refs) == 1:
            inferred = all_refs
        else:
            context = " ".join(filter(None, [
                component.evidence,
                component.description_raw,
                " ".join(component.names_raw),
            ]))
            mentioned = _generation_refs_mentioned(event, context)
            if mentioned:
                inferred = mentioned
            elif (
                component.component_type == AssociatedComponentType.ENERGY_STORAGE
                and hybrid_context
            ):
                inferred = all_refs
            else:
                inferred = valid
        inferred = _canonicalize_refs(inferred)
        if inferred != component.related_generation_asset_refs:
            adjustments.append(
                f"{component.local_component_ref}: enlaces a plantas canonicalizados como {inferred}."
            )
        component.related_generation_asset_refs = inferred
    return event, adjustments


# -----------------------------------------------------------------------------
# Canonicalización de actuaciones y targets
# -----------------------------------------------------------------------------


def _normalize_action_type(action: AdministrativeAction) -> AdministrativeActionType:
    if (
        action.decision == AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
        and action.action_type in {
            AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
            AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
            AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT,
        }
    ):
        return AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT
    return action.action_type



def _normalize_action_type_from_context(
    action: AdministrativeAction,
    *,
    document_title: str,
) -> AdministrativeActionType:
    normalized = _normalize_action_type(action)
    title_types = _action_types_from_title(document_title)
    title_key = _canonical_documentary_text(document_title).casefold()

    if (
        AdministrativeActionType.WATER_CONCESSION in title_types
        and normalized == AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION
        and not re.search(_ACTION_PATTERNS[AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION], title_key)
    ):
        return AdministrativeActionType.WATER_CONCESSION
    return normalized


def _action_pattern(action_type: AdministrativeActionType) -> str:
    if action_type == AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT:
        return r"evaluaci[oó]n\s+de\s+impacto\s+ambiental|impacto\s+ambiental"
    if action_type == AdministrativeActionType.PUBLIC_INFORMATION:
        return r"informaci[oó]n\s+p[uú]blica"
    return _ACTION_PATTERNS.get(action_type, re.escape(action_type.value.replace("_", " ")))


def _repair_action_evidence(
    action: AdministrativeAction,
    *,
    source_text: str,
    document_title: str,
    title_types: set[AdministrativeActionType],
) -> str | None:
    effective = _normalize_action_type(action)
    if effective in title_types and _documentary_contains(document_title, source_text):
        return document_title
    return _repair_evidence(
        action.evidence,
        source_text=source_text,
        semantic_patterns=[_action_pattern(effective)],
        document_title=document_title,
    )


def _infer_action_targets(
    event: PublicationEvent,
    *,
    action: AdministrativeAction,
    context: str,
) -> list[str]:
    generation_refs = _generation_refs_mentioned(event, context, direct_only=True)
    component_refs = _component_refs_mentioned(event, context)
    mentioned = _canonicalize_refs(generation_refs + component_refs)
    all_entities = _canonicalize_refs(
        [asset.local_generation_asset_ref for asset in event.generation_assets]
        + [component.local_component_ref for component in event.associated_components]
    )

    # Si la cita enumera todos los elementos del proyecto, «event» expresa la
    # semántica con menor ambigüedad y evita duplicar la lista.
    if mentioned and set(mentioned) == set(all_entities):
        return ["event"]
    if mentioned:
        return mentioned

    valid_existing = [
        target
        for target in action.targets
        if target == "event" or target in all_entities
    ]
    if valid_existing:
        if set(valid_existing) == set(all_entities):
            return ["event"]
        return _canonicalize_refs(valid_existing)
    return ["event"]


def _canonicalize_actions(
    event: PublicationEvent,
    *,
    source_text: str,
    document_title: str,
) -> tuple[PublicationEvent, list[str]]:
    event = event.model_copy(deep=True)
    title_types = _action_types_from_title(document_title)
    adjustments: list[str] = []
    actions: list[AdministrativeAction] = []

    for index, original in enumerate(event.administrative_actions, start=1):
        action = original.model_copy(deep=True)
        normalized_type = _normalize_action_type_from_context(
            action,
            document_title=document_title,
        )
        if normalized_type != action.action_type:
            adjustments.append(
                f"action_{index}: producto ambiental no final normalizado a evaluación_impacto_ambiental."
            )
            action.action_type = normalized_type

        if (
            action.action_type == AdministrativeActionType.AUTHORIZATION_MODIFICATION
            and title_types & {
                AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
                AdministrativeActionType.OPERATING_AUTHORIZATION,
            }
        ):
            adjustments.append(
                f"action_{index}: modificación genérica omitida; se conserva en is_modification de la autorización concreta."
            )
            continue

        # Cuando el título contiene el acto actual, normaliza también su decisión.
        # Evita conservar 'solicitado' cuando el objeto publicado es el sometimiento
        # a información pública de esa solicitud.
        if action.action_type in title_types:
            title_decision = _decision_from_title(document_title, action.action_type)
            if title_decision != AdministrativeDecision.OTHER:
                action.decision = title_decision

        # Elimina antecedentes cuando el título define el objeto actual y el
        # tipo de la cita no forma parte de ese objeto.
        if (
            title_types
            and action.action_type not in title_types
            and _is_clearly_historical(action.evidence)
        ):
            adjustments.append(
                f"action_{index}: antecedente histórico omitido ({action.action_type.value})."
            )
            continue

        evidence = _repair_action_evidence(
            action,
            source_text=source_text,
            document_title=document_title,
            title_types=title_types,
        )
        if evidence is None:
            adjustments.append(
                f"action_{index}: actuación descartada al no localizar evidencia literal."
            )
            continue
        action.evidence = evidence

        expectation = _modification_expectation(action, document_title=document_title)
        if expectation is not None:
            action.is_modification = expectation

        action.targets = _infer_action_targets(
            event,
            action=action,
            context=evidence,
        )
        actions.append(action)

    existing_types = {action.action_type for action in actions}
    for action_type in sorted(title_types - existing_types, key=lambda item: item.value):
        decision = _decision_from_title(document_title, action_type)
        action = AdministrativeAction(
            action_type=action_type,
            decision=decision,
            is_modification=False,
            targets=["event"],
            evidence=document_title,
        )
        expectation = _modification_expectation(action, document_title=document_title)
        if expectation is not None:
            action.is_modification = expectation
        action.targets = _infer_action_targets(
            event,
            action=action,
            context=document_title,
        )
        actions.append(action)
        adjustments.append(
            f"Actuación actual añadida determinísticamente desde el título: {action_type.value}."
        )

    has_specific_public_info = any(
        action.decision == AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
        and action.action_type != AdministrativeActionType.PUBLIC_INFORMATION
        for action in actions
    )
    deduped: list[AdministrativeAction] = []
    seen: set[tuple[Any, ...]] = set()
    for action in actions:
        if (
            has_specific_public_info
            and action.action_type == AdministrativeActionType.PUBLIC_INFORMATION
        ):
            adjustments.append("Información pública genérica omitida por redundancia.")
            continue
        key = (
            action.action_type,
            action.decision,
            action.is_modification,
            tuple(action.targets),
        )
        if key in seen:
            continue
        seen.add(key)
        deduped.append(action)

    event.administrative_actions = deduped
    return event, adjustments


def _remap_component_targets(
    actions: list[AdministrativeAction],
    mapping: dict[str, str],
) -> list[AdministrativeAction]:
    result: list[AdministrativeAction] = []
    for action in actions:
        targets = [mapping.get(target, target) for target in action.targets]
        targets = _canonicalize_refs(targets)
        if "event" in targets:
            targets = ["event"]
        result.append(action.model_copy(update={"targets": targets}))
    return result


# -----------------------------------------------------------------------------
# Relaciones y granularidad de eventos
# -----------------------------------------------------------------------------


def _canonicalize_generation_relations(
    event: PublicationEvent,
    *,
    source_text: str,
    document_title: str,
) -> tuple[PublicationEvent, list[str]]:
    event = event.model_copy(deep=True)
    valid_refs = {asset.local_generation_asset_ref for asset in event.generation_assets}
    adjustments: list[str] = []
    relations: list[GenerationAssetRelation] = []

    for relation in event.generation_relations:
        if {
            relation.source_generation_asset_ref,
            relation.target_generation_asset_ref,
        } - valid_refs:
            continue
        semantic = (
            r"h[ií]brid"
            if relation.relation_type == GenerationRelationType.HYBRIDIZED_WITH
            else r"sustitu|reemplaz"
        )
        source_asset = next(
            asset for asset in event.generation_assets
            if asset.local_generation_asset_ref == relation.source_generation_asset_ref
        )
        target_asset = next(
            asset for asset in event.generation_assets
            if asset.local_generation_asset_ref == relation.target_generation_asset_ref
        )
        shared_names = {
            _text_key(name) for name in source_asset.names_raw
        } & {
            _text_key(name) for name in target_asset.names_raw
        }
        anchors = (
            [source_asset.names_raw[0]]
            if shared_names
            else [source_asset.names_raw[0], target_asset.names_raw[0]]
        )
        evidence = _repair_evidence(
            relation.evidence,
            source_text=source_text,
            anchors=anchors,
            semantic_patterns=[semantic],
            document_title=document_title,
        )
        if evidence is None:
            continue
        source_ref = relation.source_generation_asset_ref
        target_ref = relation.target_generation_asset_ref
        if (
            relation.relation_type == GenerationRelationType.HYBRIDIZED_WITH
            and _reference_sort_key(source_ref) > _reference_sort_key(target_ref)
        ):
            source_ref, target_ref = target_ref, source_ref
        relations.append(relation.model_copy(update={
            "source_generation_asset_ref": source_ref,
            "target_generation_asset_ref": target_ref,
            "evidence": evidence,
        }))

    # Caso común: dos componentes tecnológicos con el mismo nombre híbrido.
    if len(event.generation_assets) == 2 and not relations:
        first_asset, second_asset = event.generation_assets
        shared_names = {
            _text_key(name) for name in first_asset.names_raw
        } & {
            _text_key(name) for name in second_asset.names_raw
        }
        hybrid_anchors = (
            [first_asset.names_raw[0]]
            if shared_names
            else [first_asset.names_raw[0], second_asset.names_raw[0]]
        )
        hybrid_evidence = _find_literal_span(
            source_text,
            anchors=hybrid_anchors,
            semantic_patterns=[r"h[ií]brid"],
            document_title=document_title,
        )
        if hybrid_evidence is not None:
            relations.append(GenerationAssetRelation(
                source_generation_asset_ref=event.generation_assets[0].local_generation_asset_ref,
                target_generation_asset_ref=event.generation_assets[1].local_generation_asset_ref,
                relation_type=GenerationRelationType.HYBRIDIZED_WITH,
                evidence=hybrid_evidence,
            ))
            adjustments.append("Relación de hibridación completada desde evidencia literal.")

    unique: dict[tuple[str, str, str], GenerationAssetRelation] = {}
    for relation in relations:
        key = (
            relation.source_generation_asset_ref,
            relation.target_generation_asset_ref,
            relation.relation_type.value,
        )
        unique.setdefault(key, relation)
    event.generation_relations = list(unique.values())
    return event, adjustments


def _event_is_integrated(event: PublicationEvent, source_text: str) -> bool:
    if len(event.generation_assets) <= 1:
        return True
    if event.generation_relations:
        return True
    if any(
        component.component_type == AssociatedComponentType.ENERGY_STORAGE
        and len(component.related_generation_asset_refs) > 1
        for component in event.associated_components
    ):
        return True

    assets = event.generation_assets
    for index, source_asset in enumerate(assets):
        for target_asset in assets[index + 1:]:
            shared_names = {
                _text_key(name) for name in source_asset.names_raw
            } & {
                _text_key(name) for name in target_asset.names_raw
            }
            anchors = (
                [source_asset.names_raw[0]]
                if shared_names
                else [source_asset.names_raw[0], target_asset.names_raw[0]]
            )
            if _find_literal_span(
                source_text,
                anchors=anchors,
                semantic_patterns=[r"h[ií]brid|sustitu|reemplaz"],
            ) is not None:
                return True
    return False


def _renumber_event(event: PublicationEvent) -> PublicationEvent:
    event = event.model_copy(deep=True)
    generation_mapping = {
        asset.local_generation_asset_ref: f"generation_asset_{index}"
        for index, asset in enumerate(event.generation_assets, start=1)
    }
    for asset in event.generation_assets:
        asset.local_generation_asset_ref = generation_mapping[asset.local_generation_asset_ref]

    component_mapping = {
        component.local_component_ref: f"component_{index}"
        for index, component in enumerate(event.associated_components, start=1)
    }
    for component in event.associated_components:
        component.local_component_ref = component_mapping[component.local_component_ref]
        component.related_generation_asset_refs = _canonicalize_refs([
            generation_mapping[ref]
            for ref in component.related_generation_asset_refs
            if ref in generation_mapping
        ])

    for action in event.administrative_actions:
        targets: list[str] = []
        for target in action.targets:
            if target == "event":
                targets = ["event"]
                break
            if target in generation_mapping:
                targets.append(generation_mapping[target])
            elif target in component_mapping:
                targets.append(component_mapping[target])
        action.targets = _canonicalize_refs(targets) if targets else ["event"]

    for relation in event.generation_relations:
        relation.source_generation_asset_ref = generation_mapping[
            relation.source_generation_asset_ref
        ]
        relation.target_generation_asset_ref = generation_mapping[
            relation.target_generation_asset_ref
        ]
    return event


def _split_independent_generation_event(
    event: PublicationEvent,
    *,
    source_text: str,
) -> tuple[list[PublicationEvent], list[str]]:
    if _event_is_integrated(event, source_text):
        return [_renumber_event(event)], []

    split_events: list[PublicationEvent] = []
    adjustments = [
        f"Evento con {len(event.generation_assets)} plantas independientes dividido por planta."
    ]
    for asset in event.generation_assets:
        old_generation_ref = asset.local_generation_asset_ref
        new_asset = asset.model_copy(update={
            "local_generation_asset_ref": "generation_asset_1"
        })

        included_original = [
            component
            for component in event.associated_components
            if not component.related_generation_asset_refs
            or old_generation_ref in component.related_generation_asset_refs
        ]
        component_mapping: dict[str, str] = {}
        included_components: list[AssociatedComponent] = []
        for index, component in enumerate(included_original, start=1):
            new_ref = f"component_{index}"
            component_mapping[component.local_component_ref] = new_ref
            included_components.append(component.model_copy(update={
                "local_component_ref": new_ref,
                "related_generation_asset_refs": ["generation_asset_1"],
            }))

        actions: list[AdministrativeAction] = []
        for action in event.administrative_actions:
            if action.targets == ["event"]:
                actions.append(action.model_copy(deep=True))
                continue
            mapped_targets: list[str] = []
            if old_generation_ref in action.targets:
                mapped_targets.append("generation_asset_1")
            for target in action.targets:
                if target in component_mapping:
                    mapped_targets.append(component_mapping[target])
            if mapped_targets:
                all_entities = {"generation_asset_1"} | set(component_mapping.values())
                canonical_targets = _canonicalize_refs(mapped_targets)
                if set(canonical_targets) == all_entities:
                    canonical_targets = ["event"]
                actions.append(action.model_copy(update={"targets": canonical_targets}))

        if not actions:
            actions = [
                action.model_copy(update={"targets": ["event"]})
                for action in event.administrative_actions
            ]

        split_events.append(PublicationEvent(
            generation_assets=[new_asset],
            associated_components=included_components,
            administrative_actions=actions,
            participants=[item.model_copy(deep=True) for item in event.participants],
            administrative_locations=[
                item.model_copy(deep=True) for item in event.administrative_locations
            ],
            generation_relations=[],
            case_file_references=list(event.case_file_references),
            event_summary=event.event_summary,
        ))
    return split_events, adjustments


def _canonicalize_optional_mentions(
    event: PublicationEvent,
    *,
    source_text: str,
    document_title: str,
) -> tuple[PublicationEvent, list[str]]:
    event = event.model_copy(deep=True)
    adjustments: list[str] = []

    participants: list[ParticipantMention] = []
    for participant in event.participants:
        if not _documentary_contains(participant.participant_name_raw, source_text):
            adjustments.append(
                f"Participante no documental descartado: {participant.participant_name_raw!r}."
            )
            continue
        evidence = _repair_evidence(
            participant.evidence,
            source_text=source_text,
            anchors=[participant.participant_name_raw],
            document_title=document_title,
        )
        if evidence is not None:
            participants.append(participant.model_copy(update={"evidence": evidence}))
    event.participants = participants

    locations: list[AdministrativeLocationMention] = []
    for location in event.administrative_locations:
        if not _documentary_contains(location.location_name_raw, source_text):
            adjustments.append(
                f"Localización no documental descartada: {location.location_name_raw!r}."
            )
            continue
        evidence = _repair_evidence(
            location.evidence,
            source_text=source_text,
            anchors=[location.location_name_raw],
            document_title=document_title,
        )
        if evidence is not None:
            locations.append(location.model_copy(update={"evidence": evidence}))
    event.administrative_locations = locations
    event.case_file_references = [
        value
        for value in _deduplicate_strings(event.case_file_references)
        if _documentary_contains(value, source_text)
    ]
    return event, adjustments


def canonicalize_project_extraction(
    extraction: BOEProjectExtraction,
    *,
    source_text: str,
    document_title: str,
) -> tuple[BOEProjectExtraction, list[str]]:
    extraction = extraction.model_copy(deep=True)
    adjustments: list[str] = []

    scope_decision, scope_reason = _scope_guard_from_document(
        document_title=document_title,
        source_text=source_text,
    )
    if scope_decision == ScopeGuardDecision.FORCE_NOT_RELEVANT:
        if (
            extraction.document_scope != DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
            or extraction.publication_events
        ):
            adjustments.append(
                "Alcance corregido determinísticamente a no relevante: "
                f"{scope_reason}"
            )
        return (
            _force_non_relevant_extraction(
                extraction,
                reason=scope_reason or "Documento fuera del alcance del TFM.",
            ),
            adjustments,
        )

    canonical_events: list[PublicationEvent] = []

    for original_event in extraction.publication_events:
        event = original_event.model_copy(deep=True)

        repaired_assets: list[GenerationAssetMention] = []
        old_to_new: dict[str, str] = {}
        for asset in event.generation_assets:
            repaired, current = _repair_generation_asset(
                asset,
                source_text=source_text,
                document_title=document_title,
            )
            adjustments.extend(current)
            if repaired is None:
                continue
            new_ref = f"generation_asset_{len(repaired_assets) + 1}"
            old_to_new[asset.local_generation_asset_ref] = new_ref
            repaired_assets.append(repaired.model_copy(update={
                "local_generation_asset_ref": new_ref,
            }))
        event.generation_assets = repaired_assets
        if not event.generation_assets:
            adjustments.append("Evento descartado al no conservar ninguna planta de generación identificable.")
            continue

        # Reasigna referencias del modelo a las plantas conservadas.
        for component in event.associated_components:
            component.related_generation_asset_refs = _canonicalize_refs([
                old_to_new[ref]
                for ref in component.related_generation_asset_refs
                if ref in old_to_new
            ])
        for action in event.administrative_actions:
            remapped: list[str] = []
            for target in action.targets:
                if target.startswith("generation_asset_"):
                    if target in old_to_new:
                        remapped.append(old_to_new[target])
                else:
                    remapped.append(target)
            action.targets = _canonicalize_refs(remapped) if remapped else []
        for relation in event.generation_relations:
            relation.source_generation_asset_ref = old_to_new.get(
                relation.source_generation_asset_ref,
                relation.source_generation_asset_ref,
            )
            relation.target_generation_asset_ref = old_to_new.get(
                relation.target_generation_asset_ref,
                relation.target_generation_asset_ref,
            )

        event, current = _expand_multitechnology_generation_assets(
            event,
            source_text=source_text,
        )
        adjustments.extend(current)

        repaired_components: list[AssociatedComponent] = []
        for component in event.associated_components:
            repaired, current = _repair_component(
                component,
                source_text=source_text,
                document_title=document_title,
            )
            adjustments.extend(current)
            if repaired is not None:
                repaired_components.append(repaired)
        event.associated_components = repaired_components

        # Si el título menciona evacuación y no existe componente, se crea una
        # única entidad secundaria con evidencia literal. El almacenamiento no
        # se fabrica porque requiere identidad y magnitudes propias.
        title_key = _canonical_documentary_text(document_title).casefold()
        if (
            re.search(_component_pattern(AssociatedComponentType.EVACUATION_SYSTEM), title_key)
            and not any(
                component.component_type in _AUXILIARY_COMPONENT_TYPES
                or component.component_type == AssociatedComponentType.EVACUATION_SYSTEM
                for component in event.associated_components
            )
        ):
            description = _exact_component_description(
                AssociatedComponentType.EVACUATION_SYSTEM,
                source_text=source_text,
                evidence=document_title,
            )
            event.associated_components.append(AssociatedComponent(
                local_component_ref=f"component_{len(event.associated_components) + 1}",
                component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                names_raw=[],
                description_raw=description,
                related_generation_asset_refs=[],
                technical_mentions=[],
                evidence=document_title,
            ))
            adjustments.append("Sistema de evacuación materializado desde el título literal.")

        merged_components, component_mapping, current = _merge_auxiliary_components(
            event.associated_components
        )
        adjustments.extend(current)
        event.associated_components = merged_components
        event.administrative_actions = _remap_component_targets(
            event.administrative_actions,
            component_mapping,
        )

        event, current = _infer_component_links(event, source_text=source_text)
        adjustments.extend(current)
        event, current = _canonicalize_actions(
            event,
            source_text=source_text,
            document_title=document_title,
        )
        adjustments.extend(current)
        event, current = _canonicalize_generation_relations(
            event,
            source_text=source_text,
            document_title=document_title,
        )
        adjustments.extend(current)
        event, current = _canonicalize_optional_mentions(
            event,
            source_text=source_text,
            document_title=document_title,
        )
        adjustments.extend(current)

        split_events, current = _split_independent_generation_event(
            event,
            source_text=source_text,
        )
        adjustments.extend(current)
        canonical_events.extend(split_events)

    extraction.publication_events = canonical_events
    if (
        extraction.document_scope == DocumentScope.GENERATION_PROJECT_SPECIFIC
        and not canonical_events
    ):
        extraction.classification_status = ClassificationStatus.UNCERTAIN
        extraction.document_scope = None
        extraction.classification_reason = (
            "No se pudo conservar una planta de generación con nombre literal y evidencia suficiente."
        )
        extraction.extraction_notes = (
            "La publicación requiere revisión porque el modelo no identificó de forma inequívoca la raíz de generación."
        )

    canonical = BOEProjectExtraction.model_validate(extraction.model_dump())
    return canonical, adjustments


# -----------------------------------------------------------------------------
# Validación final: estricta en identidad y literalidad, no en coincidencias
# artificialmente locales entre evidencia, enlaces y targets.
# -----------------------------------------------------------------------------


def _generation_name_has_documentary_context(
    *,
    asset: GenerationAssetMention,
    name: str,
    source_text: str,
    document_title: str,
) -> bool:
    """Comprueba que el nombre identifica una planta sin imponer una sintaxis única."""

    inferred_from_name = _infer_generation_type(name)
    if (
        inferred_from_name is not None
        and inferred_from_name == asset.generation_type
        and _documentary_contains(name, source_text)
    ):
        return True

    return _find_literal_span(
        source_text,
        anchors=[name],
        semantic_patterns=[_GENERATION_DESCRIPTOR_PATTERN],
        document_title=document_title,
    ) is not None


def validate_extraction_against_document(
    *,
    document: BOESourceDocument,
    extraction: BOEProjectExtraction,
) -> None:
    source = f"{document.title}\n{document.text}"
    issues: list[str] = []

    scope_decision, scope_reason = _scope_guard_from_document(
        document_title=document.title,
        source_text=source,
    )
    if (
        scope_decision == ScopeGuardDecision.FORCE_NOT_RELEVANT
        and extraction.document_scope == DocumentScope.GENERATION_PROJECT_SPECIFIC
    ):
        issues.append(
            "Falso positivo de alcance: "
            + (scope_reason or "el objeto principal no es una planta de generación.")
        )
    if (
        scope_decision == ScopeGuardDecision.REQUIRE_PROJECT_REVIEW
        and extraction.document_scope != DocumentScope.GENERATION_PROJECT_SPECIFIC
    ):
        issues.append(
            "Posible falso negativo de alcance: "
            + (scope_reason or "el título identifica un proyecto de generación.")
        )

    if extraction.boe_id != document.boe_id:
        issues.append("boe_id no coincide con la fuente.")
    if extraction.publication_date != document.publication_date:
        issues.append("publication_date no coincide con la fuente.")

    for event_index, event in enumerate(extraction.publication_events, start=1):
        event_path = f"publication_events[{event_index}]"
        generation_refs = {
            asset.local_generation_asset_ref for asset in event.generation_assets
        }
        component_refs = {
            component.local_component_ref for component in event.associated_components
        }
        all_entities = generation_refs | component_refs

        if not event.generation_assets:
            issues.append(f"{event_path}: no contiene ninguna planta de generación.")

        for asset in event.generation_assets:
            path = f"{event_path}.{asset.local_generation_asset_ref}"
            if not asset.names_raw:
                issues.append(f"{path}: names_raw está vacío.")
            for name in asset.names_raw:
                if not _documentary_contains(name, source):
                    issues.append(f"{path}: nombre no documental {name!r}.")
            if not _evidence_is_supported(asset.evidence, source):
                issues.append(f"{path}.evidence no es literal.")
            elif not any(
                _generation_name_has_documentary_context(
                    asset=asset,
                    name=name,
                    source_text=source,
                    document_title=document.title,
                )
                for name in asset.names_raw
            ):
                issues.append(
                    f"{path}: la fuente no identifica la denominación como planta de generación."
                )
            for mention in asset.technical_mentions:
                if not _documentary_contains(mention.value_raw, source):
                    issues.append(f"{path}: magnitud no documental {mention.value_raw!r}.")
                if not _evidence_is_supported(mention.evidence, source):
                    issues.append(f"{path}: evidence técnica no literal.")

        for component in event.associated_components:
            path = f"{event_path}.{component.local_component_ref}"
            if not _evidence_is_supported(component.evidence, source):
                issues.append(f"{path}.evidence no es literal.")
            if component.description_raw and not _documentary_contains(
                component.description_raw, source
            ):
                issues.append(f"{path}.description_raw no es literal.")
            for name in component.names_raw:
                if not _documentary_contains(name, source):
                    issues.append(f"{path}: nombre no documental {name!r}.")
            if not component.related_generation_asset_refs:
                issues.append(f"{path}: no está vinculado a ninguna planta.")
            missing = set(component.related_generation_asset_refs) - generation_refs
            if missing:
                issues.append(f"{path}: referencias de planta inexistentes {sorted(missing)}.")
            for mention in component.technical_mentions:
                if not _documentary_contains(mention.value_raw, source):
                    issues.append(f"{path}: magnitud no documental {mention.value_raw!r}.")
                if not _evidence_is_supported(mention.evidence, source):
                    issues.append(f"{path}: evidence técnica no literal.")

        for action_index, action in enumerate(event.administrative_actions, start=1):
            path = f"{event_path}.administrative_actions[{action_index}]"
            if not _evidence_is_supported(action.evidence, source):
                issues.append(f"{path}.evidence no es literal.")
            if not action.targets:
                issues.append(f"{path}: targets está vacío.")
            if "event" in action.targets and len(action.targets) != 1:
                issues.append(f"{path}: event no puede combinarse con otros targets.")
            missing = set(action.targets) - {"event"} - all_entities
            if missing:
                issues.append(f"{path}: targets inexistentes {sorted(missing)}.")
            expectation = _modification_expectation(action, document_title=document.title)
            if expectation is not None and action.is_modification != expectation:
                issues.append(f"{path}: is_modification no coincide con el título.")

        for participant in event.participants:
            if not _documentary_contains(participant.participant_name_raw, source):
                issues.append(f"{event_path}: participante no documental.")
            if not _evidence_is_supported(participant.evidence, source):
                issues.append(f"{event_path}: evidence de participante no literal.")

        for location in event.administrative_locations:
            if not _documentary_contains(location.location_name_raw, source):
                issues.append(f"{event_path}: localización no documental.")
            if not _evidence_is_supported(location.evidence, source):
                issues.append(f"{event_path}: evidence de localización no literal.")

        for relation in event.generation_relations:
            path = f"{event_path}.generation_relations"
            if {
                relation.source_generation_asset_ref,
                relation.target_generation_asset_ref,
            } - generation_refs:
                issues.append(f"{path}: referencias inexistentes.")
            if not _evidence_is_supported(relation.evidence, source):
                issues.append(f"{path}: evidence no literal.")
            relation_key = _canonical_documentary_text(relation.evidence).casefold()
            supported = (
                relation.relation_type == GenerationRelationType.HYBRIDIZED_WITH
                and bool(re.search(r"h[ií]brid", relation_key))
            ) or (
                relation.relation_type == GenerationRelationType.REPLACES
                and bool(re.search(r"sustitu|reemplaz", relation_key))
            )
            if not supported:
                issues.append(f"{path}: semántica no respaldada.")

        if len(event.generation_assets) > 1 and not _event_is_integrated(event, source):
            issues.append(
                f"{event_path}: varias plantas independientes deben estar en eventos distintos."
            )

    title_types = _action_types_from_title(document.title)
    extracted_types = {
        action.action_type
        for event in extraction.publication_events
        for action in event.administrative_actions
    }
    missing_title_types = title_types - extracted_types
    if (
        extraction.document_scope == DocumentScope.GENERATION_PROJECT_SPECIFIC
        and missing_title_types
    ):
        issues.append(
            "Faltan actuaciones actuales explícitas del título: "
            f"{sorted(item.value for item in missing_title_types)}."
        )

    if issues:
        raise DocumentExtractionValidationError(issues)


def build_document_validation_retry_prompt(
    *,
    original_prompt: str,
    validation_error: DocumentExtractionValidationError,
) -> str:
    issues = "\n".join(f"- {issue}" for issue in validation_error.issues)
    return (
        f"{original_prompt}\n\n"
        "La salida anterior cumplió el esquema, pero no las invariantes documentales. "
        "Genera una salida completa nueva y corrige estas incidencias:\n"
        f"{issues}\n\n"
        "Recuerda: las únicas raíces son plantas de generación con nombres literales; "
        "almacenamiento y evacuación son componentes asociados. Usa target='event' "
        "cuando la actuación recae sobre todo el proyecto y targets concretos solo "
        "cuando afecta a un subconjunto inequívoco."
    )


## 6. Ejecución, reintentos y registros

In [7]:
_RETRYABLE_HTTP_STATUS_CODES = {408, 425, 429, 500, 502, 503, 504}


def _usage_values(usage: RunUsage) -> dict[str, int | None]:
    return {
        "usage_requests": getattr(usage, "requests", None),
        "usage_input_tokens": getattr(usage, "input_tokens", None),
        "usage_output_tokens": getattr(usage, "output_tokens", None),
        "usage_total_tokens": getattr(usage, "total_tokens", None),
    }


def _count_extracted_nodes(extraction: BOEProjectExtraction) -> dict[str, int]:
    events = extraction.publication_events
    return {
        "n_publication_events": len(events),
        "n_generation_assets": sum(len(event.generation_assets) for event in events),
        "n_associated_components": sum(len(event.associated_components) for event in events),
        "n_administrative_actions": sum(len(event.administrative_actions) for event in events),
        "n_participants": sum(len(event.participants) for event in events),
        "n_administrative_locations": sum(len(event.administrative_locations) for event in events),
        "n_generation_relations": sum(len(event.generation_relations) for event in events),
        "n_technical_mentions": sum(
            sum(len(asset.technical_mentions) for asset in event.generation_assets)
            + sum(
                len(component.technical_mentions)
                for component in event.associated_components
            )
            for event in events
        ),
    }


def _is_retryable_model_error(error: Exception) -> bool:
    if isinstance(error, (TimeoutError, asyncio.TimeoutError)):
        return True
    status_code = getattr(error, "status_code", None)
    return status_code in _RETRYABLE_HTTP_STATUS_CODES


async def run_agent_with_transient_retries(
    *,
    agent: Agent,
    prompt: str,
    usage: RunUsage,
):
    for attempt_number in range(1, TRANSIENT_RUN_ATTEMPTS + 1):
        try:
            return await asyncio.wait_for(
                agent.run(
                    prompt,
                    model_settings=MODEL_SETTINGS,
                    usage_limits=UsageLimits(
                        request_limit=MAX_MODEL_REQUESTS_PER_DOCUMENT
                    ),
                    usage=usage,
                ),
                timeout=MODEL_RUN_TIMEOUT_SECONDS,
            )
        except Exception as error:
            if (
                attempt_number == TRANSIENT_RUN_ATTEMPTS
                or not _is_retryable_model_error(error)
            ):
                raise
            await asyncio.sleep(
                TRANSIENT_RETRY_BASE_SECONDS * (2 ** (attempt_number - 1))
            )
    raise RuntimeError("Flujo de reintentos inalcanzable.")


def _base_record(
    *,
    document: BOESourceDocument,
    prepared: PreparedDocumentPrompt | None,
) -> dict[str, Any]:
    return {
        "attempt_id": uuid4().hex,
        "identificador_boe": document.boe_id,
        "fecha_publicacion": pd.Timestamp(document.publication_date),
        "titulo": document.title,
        "source_document_sha256": document.source_document_sha256,
        "input_text_chars": prepared.input_text_chars if prepared else None,
        "input_text_sha256": prepared.input_text_sha256 if prepared else None,
        "input_selection_strategy": prepared.input_selection_strategy if prepared else None,
        "input_selection_marker": prepared.input_selection_marker if prepared else None,
        "input_excluded_chars": prepared.input_excluded_chars if prepared else None,
        "extraction_config_id": EXTRACTION_CONFIG_ID,
        "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
        "instructions_sha256": INSTRUCTIONS_SHA256,
        "model_provider": MODEL_PROVIDER,
        "model_name": AI_MODEL_NAME,
        "document_validation_version": DOCUMENT_VALIDATION_VERSION,
    }


def build_success_record(
    *,
    document: BOESourceDocument,
    prepared: PreparedDocumentPrompt | None,
    extraction: BOEProjectExtraction,
    duration_seconds: float,
    usage: RunUsage,
    adjustments: list[str],
    processing_stage: str = "completed",
) -> dict[str, Any]:
    counts = _count_extracted_nodes(extraction)
    record = {
        **_base_record(document=document, prepared=prepared),
        "classification_status": extraction.classification_status.value,
        "document_scope": extraction.document_scope.value if extraction.document_scope else None,
        "classification_reason": extraction.classification_reason,
        **counts,
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc),
        "duration_seconds": duration_seconds,
        **_usage_values(usage),
        "extraction_status": "ok",
        "error_type": None,
        "error_message": None,
        "processing_stage": processing_stage,
        "document_validation_status": "passed",
        "document_validation_issue_count": 0,
        "validation_issues_json": None,
        "deterministic_adjustment_count": len(adjustments),
        "deterministic_adjustments_json": (
            json.dumps(adjustments, ensure_ascii=False) if adjustments else None
        ),
    }
    return normalise_ai_extraction_attempts_log(pd.DataFrame([record])).iloc[0].to_dict()


def build_error_record(
    *,
    document: BOESourceDocument,
    prepared: PreparedDocumentPrompt | None,
    error: Exception,
    processing_stage: str,
    duration_seconds: float,
    usage: RunUsage,
    extraction: BOEProjectExtraction | None,
    adjustments: list[str],
) -> dict[str, Any]:
    issues = error.issues if isinstance(error, DocumentExtractionValidationError) else []
    counts = _count_extracted_nodes(extraction) if extraction else {
        "n_publication_events": None,
        "n_generation_assets": None,
        "n_associated_components": None,
        "n_administrative_actions": None,
        "n_participants": None,
        "n_administrative_locations": None,
        "n_generation_relations": None,
        "n_technical_mentions": None,
    }
    record = {
        **_base_record(document=document, prepared=prepared),
        "classification_status": extraction.classification_status.value if extraction else None,
        "document_scope": (
            extraction.document_scope.value
            if extraction and extraction.document_scope
            else None
        ),
        "classification_reason": extraction.classification_reason if extraction else None,
        **counts,
        "extraction_json": extraction.model_dump_json() if extraction else None,
        "extracted_at": datetime.now(timezone.utc),
        "duration_seconds": duration_seconds,
        **_usage_values(usage),
        "extraction_status": "error",
        "error_type": type(error).__name__,
        "error_message": str(error)[:100_000],
        "processing_stage": processing_stage,
        "document_validation_status": "failed" if issues else None,
        "document_validation_issue_count": len(issues),
        "validation_issues_json": (
            json.dumps(issues, ensure_ascii=False) if issues else None
        ),
        "deterministic_adjustment_count": len(adjustments),
        "deterministic_adjustments_json": (
            json.dumps(adjustments, ensure_ascii=False) if adjustments else None
        ),
    }
    return normalise_ai_extraction_attempts_log(pd.DataFrame([record])).iloc[0].to_dict()


debug_state: dict[str, Any] = {}


async def extract_documents(
    run_df: pd.DataFrame,
    *,
    agent: Agent,
    attempts_path: Path = BOE_AI_EXTRACTION_ATTEMPTS_PATH,
    checkpoint_every: int = CHECKPOINT_EVERY,
) -> list[dict[str, Any]]:
    if checkpoint_every < 1:
        raise ValueError("checkpoint_every debe ser mayor que cero.")

    records: list[dict[str, Any]] = []
    checkpoint: list[dict[str, Any]] = []
    global debug_state

    for position, (_, row) in enumerate(run_df.iterrows(), start=1):
        document = build_source_document(row)
        prepared: PreparedDocumentPrompt | None = None
        project_extraction: BOEProjectExtraction | None = None
        adjustments: list[str] = []
        usage = RunUsage()
        processing_stage = "build_prompt"
        started = perf_counter()
        print(f"[{position}/{len(run_df)}] {document.boe_id}")

        try:
            project_extraction, adjustments = preclassify_document_without_model(
                document
            )
            if project_extraction is not None:
                processing_stage = "deterministic_scope_guard"
                validate_extraction_against_document(
                    document=document,
                    extraction=project_extraction,
                )
                record = build_success_record(
                    document=document,
                    prepared=None,
                    extraction=project_extraction,
                    duration_seconds=perf_counter() - started,
                    usage=usage,
                    adjustments=adjustments,
                    processing_stage=processing_stage,
                )
            else:
                if agent is None:
                    raise RuntimeError(
                        "El documento requiere IA, pero no se ha construido el agente. "
                        "Instala pydantic-ai y configura el proveedor."
                    )
                prepared = build_document_prompt(document)
                original_prompt = prepared.prompt
                effective_prompt = original_prompt
                validation_retry_count = 0

                while True:
                    remaining = DOCUMENT_TIMEOUT_SECONDS - (perf_counter() - started)
                    if remaining <= 0:
                        raise TimeoutError(
                            f"{document.boe_id} superó {DOCUMENT_TIMEOUT_SECONDS:.0f} s."
                        )

                    processing_stage = "agent_run"
                    result = await asyncio.wait_for(
                        run_agent_with_transient_retries(
                            agent=agent,
                            prompt=effective_prompt,
                            usage=usage,
                        ),
                        timeout=remaining,
                    )
                    ai_extraction = result.output
                    if not isinstance(ai_extraction, BOEAIExtraction):
                        raise TypeError("El agente no devolvió BOEAIExtraction.")

                    processing_stage = "canonicalization"
                    project_extraction = build_boe_project_extraction(
                        ai_extraction,
                        boe_id=document.boe_id,
                        publication_date=document.publication_date,
                    )
                    project_extraction, adjustments = canonicalize_project_extraction(
                        project_extraction,
                        source_text=f"{document.title}\n{document.text}",
                        document_title=document.title,
                    )

                    processing_stage = "document_validation"
                    try:
                        validate_extraction_against_document(
                            document=document,
                            extraction=project_extraction,
                        )
                    except DocumentExtractionValidationError as validation_error:
                        can_retry = (
                            validation_retry_count < DOCUMENT_VALIDATION_RETRY_ATTEMPTS
                            and getattr(usage, "requests", 0)
                            < MAX_MODEL_REQUESTS_PER_DOCUMENT
                        )
                        if not can_retry:
                            raise
                        validation_retry_count += 1
                        effective_prompt = build_document_validation_retry_prompt(
                            original_prompt=original_prompt,
                            validation_error=validation_error,
                        )
                        continue
                    break

                record = build_success_record(
                    document=document,
                    prepared=prepared,
                    extraction=project_extraction,
                    duration_seconds=perf_counter() - started,
                    usage=usage,
                    adjustments=adjustments,
                    processing_stage="completed",
                )
            print(
                "  OK "
                f"events={record['n_publication_events']}, "
                f"plants={record['n_generation_assets']}, "
                f"components={record['n_associated_components']}, "
                f"actions={record['n_administrative_actions']}"
            )
        except Exception as error:
            record = build_error_record(
                document=document,
                prepared=prepared,
                error=error,
                processing_stage=processing_stage,
                duration_seconds=perf_counter() - started,
                usage=usage,
                extraction=project_extraction,
                adjustments=adjustments,
            )
            print(f"  ERROR {type(error).__name__}: {error}")

        debug_state = {
            "document": document,
            "prepared_prompt": prepared,
            "project_extraction": project_extraction,
            "adjustments": adjustments,
            "record": record,
        }
        records.append(record)
        checkpoint.append(record)
        if len(checkpoint) >= checkpoint_every or position == len(run_df):
            append_ai_extraction_attempts(pd.DataFrame(checkpoint), attempts_path)
            checkpoint.clear()

    return records


async def run_and_finalize_extractions(
    run_df: pd.DataFrame,
    source_df: pd.DataFrame,
    *,
    agent: Agent,
    attempts_path: Path = BOE_AI_EXTRACTION_ATTEMPTS_PATH,
    current_path: Path = BOE_AI_EXTRACTIONS_PATH,
    review_queue_path: Path | None = None,
    quality_metrics_path: Path | None = None,
    manual_reviews: pd.DataFrame | None = None,
    run_scope: str = "production",
    minimum_auto_validation_rate: float = 0.95,
    checkpoint_every: int = CHECKPOINT_EVERY,
) -> dict[str, pd.DataFrame]:
    records = await extract_documents(
        run_df,
        agent=agent,
        attempts_path=attempts_path,
        checkpoint_every=checkpoint_every,
    )
    attempts = load_ai_extraction_attempts(attempts_path)
    manual_reviews = (
        empty_manual_reviews()
        if manual_reviews is None
        else normalise_manual_reviews(manual_reviews)
    )
    current = select_best_valid_extractions(
        attempts=attempts,
        source_df=source_df,
        manual_reviews=manual_reviews,
    )
    save_parquet_atomic(current, current_path)

    review_queue = build_review_queue(
        attempts=attempts,
        source_df=source_df,
        manual_reviews=manual_reviews,
    )
    if review_queue_path is not None:
        save_parquet_atomic(review_queue, review_queue_path)

    quality_metric = build_quality_metric(
        attempts=attempts,
        source_df=source_df,
        manual_reviews=manual_reviews,
        run_scope=run_scope,
        minimum_auto_validation_rate=minimum_auto_validation_rate,
    )
    if quality_metrics_path is not None:
        append_quality_metric(quality_metric, quality_metrics_path)

    return {
        "new_attempts": normalise_ai_extraction_attempts_log(pd.DataFrame(records)),
        "all_attempts": attempts,
        "current_extractions": current,
        "review_queue": review_queue,
        "quality_metric": quality_metric,
    }


def get_extraction_model(
    current_extractions: pd.DataFrame,
    boe_id: str,
) -> BOEProjectExtraction:
    rows = current_extractions.loc[
        current_extractions["identificador_boe"].astype(str).eq(boe_id)
        & current_extractions["extraction_status"].eq("ok")
        & current_extractions["document_validation_status"].eq("passed")
        & current_extractions["document_validation_version"].eq(
            DOCUMENT_VALIDATION_VERSION
        )
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Se esperaba una extracción vigente para {boe_id!r}; encontradas={len(rows)}."
        )
    return BOEProjectExtraction.model_validate_json(
        str(rows.iloc[0]["extraction_json"])
    )

## 7. Calidad, revisión manual y selección de la extracción vigente

La extracción automática y la corrección humana se conservan como fuentes
separadas. La tabla vigente se reconstruye de forma determinista: una revisión
manual validada prevalece sobre la extracción automática; una revisión manual
rechazada excluye el documento; en ausencia de revisión se usa la extracción
automática válida más reciente.

La cola de revisión es derivada y no se edita. Cada corrección humana se guarda
como un archivo JSON versionable en `data/manual/boe_ai_reviews/` y se materializa
en `boe_ai_manual_reviews.parquet`.


In [8]:

REVIEW_QUEUE_COLUMNS = [
    "review_queue_id",
    "identificador_boe",
    "fecha_publicacion",
    "titulo",
    "source_document_sha256",
    "source_attempt_id",
    "extraction_config_id",
    "document_validation_version",
    "error_type",
    "error_message",
    "processing_stage",
    "validation_issues_json",
    "proposed_extraction_json",
    "queue_status",
    "queued_at",
]

MANUAL_REVIEW_COLUMNS = [
    "manual_review_id",
    "identificador_boe",
    "source_document_sha256",
    "source_attempt_id",
    "review_status",
    "corrected_extraction_json",
    "reviewer",
    "review_notes",
    "reviewed_at",
    "contract_schema_sha256",
    "document_validation_version",
]

QUALITY_METRIC_COLUMNS = [
    "quality_run_id",
    "measured_at",
    "run_scope",
    "extraction_config_id",
    "document_validation_version",
    "model_provider",
    "model_name",
    "n_source_documents",
    "n_latest_attempts",
    "n_auto_validated",
    "n_review_required",
    "n_manually_validated",
    "n_rejected",
    "automatic_validation_rate",
    "effective_validation_rate",
    "minimum_auto_validation_rate",
    "n_scope_evaluated",
    "n_scope_mismatches",
    "scope_accuracy",
    "minimum_scope_accuracy",
    "quality_status",
    "quality_alert",
]


def _normalise_table(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    dataframe = dataframe.copy()
    for column in columns:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA
    extra = [column for column in dataframe.columns if column not in columns]
    return dataframe[columns + extra]


def empty_review_queue() -> pd.DataFrame:
    return pd.DataFrame(columns=REVIEW_QUEUE_COLUMNS)


def empty_manual_reviews() -> pd.DataFrame:
    return pd.DataFrame(columns=MANUAL_REVIEW_COLUMNS)


def empty_quality_metrics() -> pd.DataFrame:
    return pd.DataFrame(columns=QUALITY_METRIC_COLUMNS)


def normalise_manual_reviews(dataframe: pd.DataFrame) -> pd.DataFrame:
    return _normalise_table(dataframe, MANUAL_REVIEW_COLUMNS)


def _latest_attempts_for_current_sources(
    attempts: pd.DataFrame,
    source_df: pd.DataFrame,
) -> pd.DataFrame:
    attempts = normalise_ai_extraction_attempts_log(attempts)
    if attempts.empty or source_df.empty:
        return attempts.iloc[0:0].copy()

    sources = source_df[
        ["identificador", "source_document_sha256"]
    ].rename(columns={"identificador": "identificador_boe"})
    current = attempts.loc[
        attempts["extraction_config_id"].eq(EXTRACTION_CONFIG_ID).fillna(False)
        & attempts["document_validation_version"]
        .eq(DOCUMENT_VALIDATION_VERSION)
        .fillna(False)
    ].merge(
        sources,
        on=["identificador_boe", "source_document_sha256"],
        how="inner",
        validate="many_to_one",
    )
    if current.empty:
        return normalise_ai_extraction_attempts_log(current)

    current["extracted_at"] = pd.to_datetime(
        current["extracted_at"],
        errors="coerce",
        utc=True,
    )
    return (
        current.sort_values(
            ["extracted_at", "attempt_id"],
            na_position="first",
            kind="stable",
        )
        .drop_duplicates("identificador_boe", keep="last")
        .reset_index(drop=True)
    )


def _latest_manual_reviews_for_current_sources(
    manual_reviews: pd.DataFrame,
    source_df: pd.DataFrame,
) -> pd.DataFrame:
    manual_reviews = normalise_manual_reviews(manual_reviews)
    if manual_reviews.empty or source_df.empty:
        return manual_reviews.iloc[0:0].copy()

    sources = source_df[
        ["identificador", "source_document_sha256"]
    ].rename(columns={"identificador": "identificador_boe"})
    current = manual_reviews.merge(
        sources,
        on=["identificador_boe", "source_document_sha256"],
        how="inner",
        validate="many_to_one",
    )
    if current.empty:
        return normalise_manual_reviews(current)

    current["reviewed_at"] = pd.to_datetime(
        current["reviewed_at"],
        errors="coerce",
        utc=True,
    )
    return (
        current.sort_values(
            ["reviewed_at", "manual_review_id"],
            na_position="first",
            kind="stable",
        )
        .drop_duplicates("identificador_boe", keep="last")
        .reset_index(drop=True)
    )


def build_review_queue(
    *,
    attempts: pd.DataFrame,
    source_df: pd.DataFrame,
    manual_reviews: pd.DataFrame | None = None,
) -> pd.DataFrame:
    latest_attempts = _latest_attempts_for_current_sources(attempts, source_df)
    if latest_attempts.empty:
        return empty_review_queue()

    manual_reviews = (
        empty_manual_reviews()
        if manual_reviews is None
        else normalise_manual_reviews(manual_reviews)
    )
    latest_manual = _latest_manual_reviews_for_current_sources(
        manual_reviews,
        source_df,
    )
    resolved_ids = set(
        latest_manual.loc[
            latest_manual["review_status"].isin(
                ["manually_validated", "rejected"]
            ),
            "identificador_boe",
        ].astype(str)
    )

    needs_review = latest_attempts.loc[
        ~(
            latest_attempts["extraction_status"].eq("ok").fillna(False)
            & latest_attempts["document_validation_status"].eq("passed").fillna(False)
        )
        & ~latest_attempts["identificador_boe"].astype(str).isin(resolved_ids)
    ].copy()
    if needs_review.empty:
        return empty_review_queue()

    queued_at = datetime.now(timezone.utc)
    records: list[dict[str, Any]] = []
    for row in needs_review.itertuples(index=False):
        queue_key = (
            f"{row.identificador_boe}|{row.source_document_sha256}|"
            f"{row.attempt_id}"
        )
        records.append({
            "review_queue_id": sha256(queue_key.encode("utf-8")).hexdigest()[:24],
            "identificador_boe": row.identificador_boe,
            "fecha_publicacion": row.fecha_publicacion,
            "titulo": row.titulo,
            "source_document_sha256": row.source_document_sha256,
            "source_attempt_id": row.attempt_id,
            "extraction_config_id": row.extraction_config_id,
            "document_validation_version": row.document_validation_version,
            "error_type": row.error_type,
            "error_message": row.error_message,
            "processing_stage": row.processing_stage,
            "validation_issues_json": row.validation_issues_json,
            "proposed_extraction_json": row.extraction_json,
            "queue_status": "pending",
            "queued_at": queued_at,
        })
    return _normalise_table(pd.DataFrame(records), REVIEW_QUEUE_COLUMNS)


def create_manual_review_file(
    review_queue: pd.DataFrame,
    boe_id: str,
    *,
    output_dir: Path = BOE_AI_MANUAL_REVIEW_DIR,
    overwrite: bool = False,
) -> Path:
    rows = review_queue.loc[
        review_queue["identificador_boe"].astype(str).eq(str(boe_id))
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Se esperaba un único elemento pendiente para {boe_id!r}; "
            f"encontrados={len(rows)}."
        )

    row = rows.iloc[0]
    proposed_json = row.get("proposed_extraction_json")
    proposed_extraction = None
    if pd.notna(proposed_json) and str(proposed_json).strip():
        proposed_extraction = json.loads(str(proposed_json))

    payload = {
        "identificador_boe": str(row["identificador_boe"]),
        "source_document_sha256": str(row["source_document_sha256"]),
        "source_attempt_id": str(row["source_attempt_id"]),
        "review_status": "pending",
        "reviewer": None,
        "review_notes": None,
        "corrected_extraction": proposed_extraction,
    }

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{boe_id}.json"
    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"Ya existe {output_path}. Usa overwrite=True solo si quieres reemplazarlo."
        )
    output_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return output_path


def load_manual_review_files(
    source_df: pd.DataFrame,
    *,
    review_dir: Path = BOE_AI_MANUAL_REVIEW_DIR,
    output_path: Path | None = BOE_AI_MANUAL_REVIEWS_PATH,
) -> pd.DataFrame:
    if not review_dir.exists():
        reviews = empty_manual_reviews()
        if output_path is not None:
            save_parquet_atomic(reviews, output_path)
        return reviews

    source_rows = {
        str(row["identificador"]): row
        for _, row in source_df.iterrows()
    }
    records: list[dict[str, Any]] = []

    for review_path in sorted(review_dir.glob("*.json")):
        payload = json.loads(review_path.read_text(encoding="utf-8"))
        boe_id = str(payload.get("identificador_boe", "")).strip()
        if boe_id not in source_rows:
            raise ValueError(
                f"{review_path}: identificador_boe no existe en el corpus actual."
            )

        document = build_source_document(source_rows[boe_id])
        if payload.get("source_document_sha256") != document.source_document_sha256:
            raise ValueError(
                f"{review_path}: la fuente BOE cambió; la revisión debe repetirse."
            )

        status = str(payload.get("review_status", "")).strip()
        if status not in {"pending", "manually_validated", "rejected"}:
            raise ValueError(
                f"{review_path}: review_status debe ser pending, "
                "manually_validated o rejected."
            )

        corrected_json: str | None = None
        if status == "manually_validated":
            reviewer = str(payload.get("reviewer") or "").strip()
            if not reviewer:
                raise ValueError(
                    f"{review_path}: reviewer es obligatorio al validar manualmente."
                )
            corrected_payload = payload.get("corrected_extraction")
            if not isinstance(corrected_payload, dict):
                raise ValueError(
                    f"{review_path}: corrected_extraction debe contener un objeto JSON."
                )
            extraction = BOEProjectExtraction.model_validate(corrected_payload)
            extraction, _ = canonicalize_project_extraction(
                extraction,
                source_text=f"{document.title}\n{document.text}",
                document_title=document.title,
            )
            validate_extraction_against_document(
                document=document,
                extraction=extraction,
            )
            corrected_json = extraction.model_dump_json()

        reviewed_at_raw = payload.get("reviewed_at")
        reviewed_at = (
            pd.to_datetime(reviewed_at_raw, utc=True)
            if reviewed_at_raw
            else pd.Timestamp(review_path.stat().st_mtime, unit="s", tz="UTC")
        )
        review_key = (
            f"{boe_id}|{document.source_document_sha256}|"
            f"{payload.get('source_attempt_id')}|{status}|{reviewed_at.isoformat()}"
        )
        records.append({
            "manual_review_id": sha256(review_key.encode("utf-8")).hexdigest()[:24],
            "identificador_boe": boe_id,
            "source_document_sha256": document.source_document_sha256,
            "source_attempt_id": payload.get("source_attempt_id"),
            "review_status": status,
            "corrected_extraction_json": corrected_json,
            "reviewer": payload.get("reviewer"),
            "review_notes": payload.get("review_notes"),
            "reviewed_at": reviewed_at,
            "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
            "document_validation_version": DOCUMENT_VALIDATION_VERSION,
        })

    reviews = normalise_manual_reviews(pd.DataFrame(records))
    if output_path is not None:
        save_parquet_atomic(reviews, output_path)
    return reviews


def _manual_review_as_extraction_record(
    review: pd.Series,
    source_row: pd.Series,
) -> dict[str, Any]:
    extraction = BOEProjectExtraction.model_validate_json(
        str(review["corrected_extraction_json"])
    )
    counts = _count_extracted_nodes(extraction)
    return {
        **{column: pd.NA for column in AI_EXTRACTION_LOG_COLUMNS},
        "attempt_id": str(review["manual_review_id"]),
        "identificador_boe": extraction.boe_id,
        "fecha_publicacion": pd.Timestamp(extraction.publication_date),
        "titulo": source_row["titulo"],
        "source_document_sha256": source_row["source_document_sha256"],
        "extraction_config_id": EXTRACTION_CONFIG_ID,
        "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
        "instructions_sha256": INSTRUCTIONS_SHA256,
        "model_provider": "manual",
        "model_name": "human_review",
        "document_validation_version": DOCUMENT_VALIDATION_VERSION,
        "classification_status": extraction.classification_status.value,
        "document_scope": (
            extraction.document_scope.value
            if extraction.document_scope
            else None
        ),
        "classification_reason": extraction.classification_reason,
        **counts,
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": review["reviewed_at"],
        "extraction_status": "ok",
        "processing_stage": "manual_review",
        "document_validation_status": "passed",
        "document_validation_issue_count": 0,
        "deterministic_adjustment_count": 0,
        "selection_source": "manually_validated",
        "manual_review_id": review["manual_review_id"],
        "reviewer": review["reviewer"],
        "review_notes": review["review_notes"],
    }


def select_best_valid_extractions(
    *,
    attempts: pd.DataFrame,
    source_df: pd.DataFrame,
    manual_reviews: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Selecciona una única extracción vigente por BOE.

    Precedencia:
    1. última revisión manual válida para la fuente actual;
    2. última extracción automática validada;
    3. ninguna extracción si la última revisión manual la rechaza.
    """

    auto = current_successful_ai_extractions(attempts, source_df).copy()
    if not auto.empty:
        auto["selection_source"] = "auto_validated"

    manual_reviews = (
        empty_manual_reviews()
        if manual_reviews is None
        else normalise_manual_reviews(manual_reviews)
    )
    latest_manual = _latest_manual_reviews_for_current_sources(
        manual_reviews,
        source_df,
    )
    manual_decision_ids = set(
        latest_manual.loc[
            latest_manual["review_status"].isin(
                ["manually_validated", "rejected"]
            ),
            "identificador_boe",
        ].astype(str)
    )
    if not auto.empty and manual_decision_ids:
        auto = auto.loc[
            ~auto["identificador_boe"].astype(str).isin(manual_decision_ids)
        ].copy()

    source_rows = {
        str(row["identificador"]): row
        for _, row in source_df.iterrows()
    }
    manual_records: list[dict[str, Any]] = []
    for _, review in latest_manual.loc[
        latest_manual["review_status"].eq("manually_validated")
    ].iterrows():
        boe_id = str(review["identificador_boe"])
        manual_records.append(
            _manual_review_as_extraction_record(review, source_rows[boe_id])
        )

    manual = (
        normalise_ai_extraction_attempts_log(pd.DataFrame(manual_records))
        if manual_records
        else normalise_ai_extraction_attempts_log(pd.DataFrame())
    )

    frames = [frame for frame in (auto, manual) if not frame.empty]
    if not frames:
        return normalise_ai_extraction_attempts_log(pd.DataFrame())
    if len(frames) == 1:
        selected = frames[0].copy()
    else:
        selected = pd.concat(frames, ignore_index=True)

    selected = (
        selected.sort_values(
            ["identificador_boe", "extracted_at", "attempt_id"],
            kind="stable",
        )
        .drop_duplicates("identificador_boe", keep="last")
        .reset_index(drop=True)
    )
    for row in selected.itertuples(index=False):
        BOEProjectExtraction.model_validate_json(str(row.extraction_json))
    return normalise_ai_extraction_attempts_log(selected)


def build_quality_metric(
    *,
    attempts: pd.DataFrame,
    source_df: pd.DataFrame,
    manual_reviews: pd.DataFrame | None,
    run_scope: str,
    minimum_auto_validation_rate: float = 0.95,
) -> pd.DataFrame:
    if not 0 <= minimum_auto_validation_rate <= 1:
        raise ValueError("minimum_auto_validation_rate debe estar entre 0 y 1.")

    latest_attempts = _latest_attempts_for_current_sources(attempts, source_df)
    auto_ids = set(
        latest_attempts.loc[
            latest_attempts["extraction_status"].eq("ok").fillna(False)
            & latest_attempts["document_validation_status"].eq("passed").fillna(False),
            "identificador_boe",
        ].astype(str)
    )

    manual_reviews = (
        empty_manual_reviews()
        if manual_reviews is None
        else normalise_manual_reviews(manual_reviews)
    )
    latest_manual = _latest_manual_reviews_for_current_sources(
        manual_reviews,
        source_df,
    )
    manual_valid_ids = set(
        latest_manual.loc[
            latest_manual["review_status"].eq("manually_validated"),
            "identificador_boe",
        ].astype(str)
    )
    rejected_ids = set(
        latest_manual.loc[
            latest_manual["review_status"].eq("rejected"),
            "identificador_boe",
        ].astype(str)
    )
    manual_decision_ids = manual_valid_ids | rejected_ids
    effective_valid_ids = (auto_ids - manual_decision_ids) | manual_valid_ids

    evaluated_ids = set(latest_attempts["identificador_boe"].astype(str))
    effective_evaluated_ids = effective_valid_ids & evaluated_ids
    rejected_evaluated_ids = rejected_ids & evaluated_ids

    n_source = int(len(source_df))
    n_evaluated = len(evaluated_ids)
    n_auto = len(auto_ids & evaluated_ids)
    n_effective = len(effective_evaluated_ids)
    n_review_required = max(
        0,
        n_evaluated - n_effective - len(rejected_evaluated_ids),
    )
    auto_rate = n_auto / n_evaluated if n_evaluated else 1.0
    effective_rate = n_effective / n_evaluated if n_evaluated else 1.0
    quality_alert = auto_rate < minimum_auto_validation_rate

    record = {
        "quality_run_id": uuid4().hex,
        "measured_at": datetime.now(timezone.utc),
        "run_scope": run_scope,
        "extraction_config_id": EXTRACTION_CONFIG_ID,
        "document_validation_version": DOCUMENT_VALIDATION_VERSION,
        "model_provider": MODEL_PROVIDER,
        "model_name": AI_MODEL_NAME,
        "n_source_documents": n_source,
        "n_latest_attempts": n_evaluated,
        "n_auto_validated": n_auto,
        "n_review_required": n_review_required,
        "n_manually_validated": len(manual_valid_ids & evaluated_ids),
        "n_rejected": len(rejected_evaluated_ids),
        "automatic_validation_rate": auto_rate,
        "effective_validation_rate": effective_rate,
        "minimum_auto_validation_rate": minimum_auto_validation_rate,
        "n_scope_evaluated": pd.NA,
        "n_scope_mismatches": pd.NA,
        "scope_accuracy": pd.NA,
        "minimum_scope_accuracy": pd.NA,
        "quality_status": "degraded" if quality_alert else "healthy",
        "quality_alert": quality_alert,
    }
    return _normalise_table(pd.DataFrame([record]), QUALITY_METRIC_COLUMNS)


def combine_quality_metrics(
    existing: pd.DataFrame,
    new_metric: pd.DataFrame,
) -> pd.DataFrame:
    """Combina métricas sin concatenar columnas vacías o totalmente nulas.

    `DataFrame.from_records` evita el `FutureWarning` de pandas asociado a
    `pd.concat` con columnas all-NA y conserva una fila completa por ejecución.
    """

    existing = _normalise_table(existing, QUALITY_METRIC_COLUMNS)
    new_metric = _normalise_table(new_metric, QUALITY_METRIC_COLUMNS)

    if existing.empty:
        combined = new_metric.copy()
    elif new_metric.empty:
        combined = existing.copy()
    else:
        records = [
            *existing.to_dict(orient="records"),
            *new_metric.to_dict(orient="records"),
        ]
        combined = _normalise_table(
            pd.DataFrame.from_records(records),
            QUALITY_METRIC_COLUMNS,
        )

    return combined.drop_duplicates(
        subset=["quality_run_id"],
        keep="last",
    ).reset_index(drop=True)


def append_quality_metric(
    new_metric: pd.DataFrame,
    path: Path = BOE_AI_QUALITY_METRICS_PATH,
) -> pd.DataFrame:
    if path.exists():
        existing = pd.read_parquet(path)
    else:
        existing = empty_quality_metrics()

    combined = combine_quality_metrics(existing, new_metric)
    save_parquet_atomic(combined, path)
    return combined


## 8. Tablas planas para agrupación y cronología


In [9]:
FLAT_TABLE_COLUMNS: dict[str, list[str]] = {
    "publication_events": [
        "event_id", "identificador_boe", "fecha_publicacion", "event_index",
        "event_summary", "n_generation_assets", "n_associated_components",
        "n_administrative_actions",
    ],
    "generation_asset_mentions": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "generation_asset_mention_id", "local_generation_asset_ref",
        "generation_type", "evidence",
    ],
    "generation_asset_names": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "generation_asset_mention_id", "name_index", "name_raw",
    ],
    "associated_components": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "associated_component_id", "local_component_ref", "component_type",
        "description_raw", "evidence",
    ],
    "associated_component_names": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "associated_component_id", "name_index", "name_raw",
    ],
    "associated_component_generation_links": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "associated_component_id", "link_index", "local_generation_asset_ref",
        "generation_asset_mention_id",
    ],
    "administrative_actions": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "administrative_action_id", "action_index", "action_type", "decision",
        "is_modification", "evidence",
    ],
    "administrative_action_targets": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "administrative_action_id", "target_index", "target_ref",
        "target_entity_id", "target_kind",
    ],
    "participant_mentions": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "participant_mention_id", "participant_name_raw", "participant_role",
        "evidence",
    ],
    "location_mentions": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "location_mention_id", "location_name_raw", "location_level",
        "province_hint_raw", "autonomous_community_hint_raw", "evidence",
    ],
    "generation_asset_relations": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "generation_asset_relation_id", "source_generation_asset_ref",
        "source_generation_asset_mention_id", "target_generation_asset_ref",
        "target_generation_asset_mention_id", "relation_type", "evidence",
    ],
    "technical_mentions": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "technical_mention_id", "owner_kind", "owner_ref",
        "generation_asset_mention_id", "associated_component_id",
        "attribute_type", "value_raw", "evidence",
    ],
    "case_file_references": [
        "event_id", "identificador_boe", "fecha_publicacion",
        "case_file_reference_id", "reference_index", "case_file_reference_raw",
    ],
}


def flatten_current_extractions(
    current_extractions: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    """Regenera tablas planas. Solo generation_asset_mentions es agrupable."""

    rows: dict[str, list[dict[str, Any]]] = {
        "publication_events": [],
        "generation_asset_mentions": [],
        "generation_asset_names": [],
        "associated_components": [],
        "associated_component_names": [],
        "associated_component_generation_links": [],
        "administrative_actions": [],
        "administrative_action_targets": [],
        "participant_mentions": [],
        "location_mentions": [],
        "generation_asset_relations": [],
        "technical_mentions": [],
        "case_file_references": [],
    }

    for extraction_row in current_extractions.itertuples(index=False):
        extraction = BOEProjectExtraction.model_validate_json(
            str(extraction_row.extraction_json)
        )
        for event_index, event in enumerate(extraction.publication_events, start=1):
            event_id = f"{extraction.boe_id}_event_{event_index}"
            base = {
                "event_id": event_id,
                "identificador_boe": extraction.boe_id,
                "fecha_publicacion": pd.Timestamp(extraction.publication_date),
            }
            rows["publication_events"].append({
                **base,
                "event_index": event_index,
                "event_summary": event.event_summary,
                "n_generation_assets": len(event.generation_assets),
                "n_associated_components": len(event.associated_components),
                "n_administrative_actions": len(event.administrative_actions),
            })

            for asset_index, asset in enumerate(event.generation_assets, start=1):
                mention_id = f"{event_id}_generation_asset_{asset_index}"
                rows["generation_asset_mentions"].append({
                    **base,
                    "generation_asset_mention_id": mention_id,
                    "local_generation_asset_ref": asset.local_generation_asset_ref,
                    "generation_type": asset.generation_type.value,
                    "evidence": asset.evidence,
                })
                for name_index, name in enumerate(asset.names_raw, start=1):
                    rows["generation_asset_names"].append({
                        **base,
                        "generation_asset_mention_id": mention_id,
                        "name_index": name_index,
                        "name_raw": name,
                    })
                for technical_index, mention in enumerate(
                    asset.technical_mentions, start=1
                ):
                    rows["technical_mentions"].append({
                        **base,
                        "technical_mention_id": f"{mention_id}_technical_{technical_index}",
                        "owner_kind": "generation_asset",
                        "owner_ref": asset.local_generation_asset_ref,
                        "generation_asset_mention_id": mention_id,
                        "associated_component_id": None,
                        "attribute_type": mention.attribute_type.value,
                        "value_raw": mention.value_raw,
                        "evidence": mention.evidence,
                    })

            for component_index, component in enumerate(
                event.associated_components, start=1
            ):
                component_id = f"{event_id}_component_{component_index}"
                rows["associated_components"].append({
                    **base,
                    "associated_component_id": component_id,
                    "local_component_ref": component.local_component_ref,
                    "component_type": component.component_type.value,
                    "description_raw": component.description_raw,
                    "evidence": component.evidence,
                })
                for name_index, name in enumerate(component.names_raw, start=1):
                    rows["associated_component_names"].append({
                        **base,
                        "associated_component_id": component_id,
                        "name_index": name_index,
                        "name_raw": name,
                    })
                for link_index, generation_ref in enumerate(
                    component.related_generation_asset_refs,
                    start=1,
                ):
                    rows["associated_component_generation_links"].append({
                        **base,
                        "associated_component_id": component_id,
                        "link_index": link_index,
                        "local_generation_asset_ref": generation_ref,
                        "generation_asset_mention_id": (
                            f"{event_id}_{generation_ref}"
                        ),
                    })
                for technical_index, mention in enumerate(
                    component.technical_mentions, start=1
                ):
                    rows["technical_mentions"].append({
                        **base,
                        "technical_mention_id": f"{component_id}_technical_{technical_index}",
                        "owner_kind": "associated_component",
                        "owner_ref": component.local_component_ref,
                        "generation_asset_mention_id": None,
                        "associated_component_id": component_id,
                        "attribute_type": mention.attribute_type.value,
                        "value_raw": mention.value_raw,
                        "evidence": mention.evidence,
                    })

            for action_index, action in enumerate(
                event.administrative_actions, start=1
            ):
                action_id = f"{event_id}_action_{action_index}"
                rows["administrative_actions"].append({
                    **base,
                    "administrative_action_id": action_id,
                    "action_index": action_index,
                    "action_type": action.action_type.value,
                    "decision": action.decision.value,
                    "is_modification": action.is_modification,
                    "evidence": action.evidence,
                })
                for target_index, target_ref in enumerate(action.targets, start=1):
                    rows["administrative_action_targets"].append({
                        **base,
                        "administrative_action_id": action_id,
                        "target_index": target_index,
                        "target_ref": target_ref,
                        "target_entity_id": (
                            event_id
                            if target_ref == "event"
                            else f"{event_id}_{target_ref}"
                        ),
                        "target_kind": (
                            "event"
                            if target_ref == "event"
                            else "generation_asset"
                            if target_ref.startswith("generation_asset_")
                            else "associated_component"
                        ),
                    })

            for participant_index, participant in enumerate(
                event.participants, start=1
            ):
                participant_id = f"{event_id}_participant_{participant_index}"
                rows["participant_mentions"].append({
                    **base,
                    "participant_mention_id": participant_id,
                    "participant_name_raw": participant.participant_name_raw,
                    "participant_role": participant.participant_role.value,
                    "evidence": participant.evidence,
                })
            for location_index, location in enumerate(
                event.administrative_locations, start=1
            ):
                location_id = f"{event_id}_location_{location_index}"
                rows["location_mentions"].append({
                    **base,
                    "location_mention_id": location_id,
                    "location_name_raw": location.location_name_raw,
                    "location_level": location.location_level.value,
                    "province_hint_raw": location.province_hint_raw,
                    "autonomous_community_hint_raw": location.autonomous_community_hint_raw,
                    "evidence": location.evidence,
                })
            for relation_index, relation in enumerate(
                event.generation_relations, start=1
            ):
                rows["generation_asset_relations"].append({
                    **base,
                    "generation_asset_relation_id": f"{event_id}_relation_{relation_index}",
                    "source_generation_asset_ref": relation.source_generation_asset_ref,
                    "source_generation_asset_mention_id": (
                        f"{event_id}_{relation.source_generation_asset_ref}"
                    ),
                    "target_generation_asset_ref": relation.target_generation_asset_ref,
                    "target_generation_asset_mention_id": (
                        f"{event_id}_{relation.target_generation_asset_ref}"
                    ),
                    "relation_type": relation.relation_type.value,
                    "evidence": relation.evidence,
                })

            for reference_index, reference in enumerate(
                event.case_file_references, start=1
            ):
                rows["case_file_references"].append({
                    **base,
                    "case_file_reference_id": f"{event_id}_case_file_{reference_index}",
                    "reference_index": reference_index,
                    "case_file_reference_raw": reference,
                })

    return {
        table_name: pd.DataFrame(
            table_rows,
            columns=FLAT_TABLE_COLUMNS[table_name],
        )
        for table_name, table_rows in rows.items()
    }


def save_flattened_extractions(
    current_extractions: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    tables = flatten_current_extractions(current_extractions)
    paths = {
        "publication_events": PUBLICATION_EVENTS_PATH,
        "generation_asset_mentions": GENERATION_ASSET_MENTIONS_PATH,
        "generation_asset_names": GENERATION_ASSET_NAMES_PATH,
        "associated_components": ASSOCIATED_COMPONENTS_PATH,
        "associated_component_names": ASSOCIATED_COMPONENT_NAMES_PATH,
        "associated_component_generation_links": (
            ASSOCIATED_COMPONENT_GENERATION_LINKS_PATH
        ),
        "administrative_actions": ADMINISTRATIVE_ACTIONS_PATH,
        "administrative_action_targets": ADMINISTRATIVE_ACTION_TARGETS_PATH,
        "participant_mentions": PARTICIPANT_MENTIONS_PATH,
        "location_mentions": LOCATION_MENTIONS_PATH,
        "generation_asset_relations": GENERATION_RELATIONS_PATH,
        "technical_mentions": TECHNICAL_MENTIONS_PATH,
        "case_file_references": CASE_FILE_REFERENCES_PATH,
    }
    for name, dataframe in tables.items():
        save_parquet_atomic(dataframe, paths[name])
    return tables

## 9. Suite de regresión determinista


In [10]:
RUN_DETERMINISTIC_REGRESSION_TESTS = True


def _test_document(
    boe_id: str,
    title: str,
    text: str | None = None,
) -> BOESourceDocument:
    body = text or title
    publication_date = date(2026, 1, 1)
    return BOESourceDocument(
        boe_id=boe_id,
        publication_date=publication_date,
        title=title,
        text=body,
        source_document_sha256=_source_document_hash(
            boe_id=boe_id,
            publication_date=publication_date,
            title=title,
            text=body,
        ),
    )


def _test_extraction(
    boe_id: str,
    events: list[PublicationEvent],
) -> BOEProjectExtraction:
    return BOEProjectExtraction(
        classification_status=ClassificationStatus.CLASSIFIED,
        document_scope=DocumentScope.GENERATION_PROJECT_SPECIFIC,
        classification_reason="Publicación relativa a una planta de generación.",
        publication_events=events,
        extraction_notes=None,
        boe_id=boe_id,
        publication_date=date(2026, 1, 1),
    )


def _asset(
    ref: str,
    name: str,
    generation_type: GenerationType,
    evidence: str,
    technical_mentions: list[TechnicalMention] | None = None,
) -> GenerationAssetMention:
    return GenerationAssetMention(
        local_generation_asset_ref=ref,
        names_raw=[name],
        generation_type=generation_type,
        technical_mentions=technical_mentions or [],
        evidence=evidence,
    )


def _action(
    action_type: AdministrativeActionType,
    decision: AdministrativeDecision,
    evidence: str,
    targets: list[str] | None = None,
    *,
    is_modification: bool = False,
) -> AdministrativeAction:
    return AdministrativeAction(
        action_type=action_type,
        decision=decision,
        is_modification=is_modification,
        targets=targets or [],
        evidence=evidence,
    )


def _canonicalize_test(
    document: BOESourceDocument,
    extraction: BOEProjectExtraction,
) -> BOEProjectExtraction:
    canonical, _ = canonicalize_project_extraction(
        extraction,
        source_text=f"{document.title}\n{document.text}",
        document_title=document.title,
    )
    validate_extraction_against_document(
        document=document,
        extraction=canonical,
    )
    return canonical


def _run_regression_tests() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    def record(name: str, fn) -> None:
        try:
            fn()
            rows.append({"test": name, "passed": True, "error": None})
        except Exception as error:
            rows.append({"test": name, "passed": False, "error": str(error)})

    def test_carbo_event_target() -> None:
        title = (
            "Anuncio por el que se somete a Información Pública la solicitud de "
            "Declaración, en concreto, de Utilidad Pública de la planta solar "
            "fotovoltaica Carbo, de 81,4 MW de potencia instalada, e infraestructura "
            "de evacuación a 30 kV."
        )
        document = _test_document("BOE-B-2024-27073", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Carbo", GenerationType.PHOTOVOLTAIC, title
                )],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                    names_raw=[],
                    description_raw="infraestructura de evacuación a 30 kV",
                    related_generation_asset_refs=[],
                    technical_mentions=[],
                    evidence="infraestructura de evacuación a 30 kV",
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PUBLIC_UTILITY_DECLARATION,
                    AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
                    title,
                )],
                event_summary="Información pública de la DUP de Carbo y su evacuación.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        event = canonical.publication_events[0]
        assert len(event.generation_assets) == 1
        assert len(event.associated_components) == 1
        assert event.associated_components[0].related_generation_asset_refs == [
            "generation_asset_1"
        ]
        assert event.administrative_actions[0].targets == ["event"]

    def test_entrenucleos_component_only() -> None:
        title = (
            "Anuncio por el que se convoca el levantamiento de actas previas a la "
            "ocupación de los bienes afectados por la infraestructura de evacuación "
            "de la planta fotovoltaica HSF Entrenucleos Ten."
        )
        document = _test_document("BOE-B-2026-17277", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "HSF Entrenucleos Ten",
                    GenerationType.PHOTOVOLTAIC,
                    title,
                )],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                    description_raw="infraestructura de evacuación",
                    related_generation_asset_refs=[],
                    evidence=title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_OCCUPATION_RECORDS,
                    AdministrativeDecision.ANNOUNCED,
                    title,
                )],
                event_summary="Actas previas de la evacuación de Entrenucleos Ten.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        action = canonical.publication_events[0].administrative_actions[0]
        assert action.targets == ["component_1"]

    def test_armus_storage_and_evacuation_targets() -> None:
        title = (
            "Resolución por la que se formula informe de impacto ambiental del "
            "proyecto Módulo de almacenamiento de energía «Armus», de 20 MW y "
            "80 MWh, y su infraestructura de evacuación, para su hibridación con "
            "la instalación híbrida «Armus Solar», integrada por 35 MW eólicos "
            "y 49,88 MW fotovoltaicos."
        )
        document = _test_document("BOE-A-2026-11838", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[
                    _asset(
                        "generation_asset_1",
                        "Armus Solar",
                        GenerationType.WIND,
                        title,
                        [TechnicalMention(
                            attribute_type=TechnicalAttributeType.INSTALLED_POWER,
                            value_raw="35 MW",
                            evidence=title,
                        )],
                    ),
                    _asset(
                        "generation_asset_2",
                        "Armus Solar",
                        GenerationType.PHOTOVOLTAIC,
                        title,
                        [TechnicalMention(
                            attribute_type=TechnicalAttributeType.INSTALLED_POWER,
                            value_raw="49,88 MW",
                            evidence=title,
                        )],
                    ),
                ],
                associated_components=[
                    AssociatedComponent(
                        local_component_ref="component_1",
                        component_type=AssociatedComponentType.ENERGY_STORAGE,
                        names_raw=["Armus"],
                        description_raw="Módulo de almacenamiento de energía «Armus»",
                        related_generation_asset_refs=[],
                        technical_mentions=[
                            TechnicalMention(
                                attribute_type=TechnicalAttributeType.STORAGE_POWER,
                                value_raw="20 MW",
                                evidence=title,
                            ),
                            TechnicalMention(
                                attribute_type=TechnicalAttributeType.STORAGE_CAPACITY,
                                value_raw="80 MWh",
                                evidence=title,
                            ),
                        ],
                        evidence=title,
                    ),
                    AssociatedComponent(
                        local_component_ref="component_2",
                        component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                        names_raw=[],
                        description_raw="infraestructura de evacuación",
                        related_generation_asset_refs=[],
                        technical_mentions=[],
                        evidence=title,
                    ),
                ],
                administrative_actions=[_action(
                    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
                    AdministrativeDecision.FORMULATED,
                    title,
                )],
                generation_relations=[GenerationAssetRelation(
                    source_generation_asset_ref="generation_asset_1",
                    target_generation_asset_ref="generation_asset_2",
                    relation_type=GenerationRelationType.HYBRIDIZED_WITH,
                    evidence=title,
                )],
                event_summary="Informe ambiental del almacenamiento Armus y su evacuación.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        event = canonical.publication_events[0]
        storage = next(
            component
            for component in event.associated_components
            if component.component_type == AssociatedComponentType.ENERGY_STORAGE
        )
        evacuation = next(
            component
            for component in event.associated_components
            if component.component_type == AssociatedComponentType.EVACUATION_SYSTEM
        )
        assert set(storage.related_generation_asset_refs) == {
            "generation_asset_1", "generation_asset_2"
        }
        assert set(evacuation.related_generation_asset_refs) == {
            "generation_asset_1", "generation_asset_2"
        }
        assert set(event.administrative_actions[0].targets) == {
            storage.local_component_ref,
            evacuation.local_component_ref,
        }

    def test_component_link_does_not_require_same_quote() -> None:
        title = "Autorización de la planta fotovoltaica Prado Gris y su evacuación."
        text = (
            "La planta fotovoltaica Prado Gris se ubica en el municipio indicado. "
            "La infraestructura de evacuación comprende una línea y una subestación."
        )
        document = _test_document("BOE-A-2026-10001", title, text)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Prado Gris", GenerationType.PHOTOVOLTAIC,
                    "La planta fotovoltaica Prado Gris se ubica en el municipio indicado."
                )],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.POWER_LINE,
                    description_raw="La infraestructura de evacuación comprende una línea",
                    related_generation_asset_refs=[],
                    evidence="La infraestructura de evacuación comprende una línea y una subestación.",
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                )],
                event_summary="Autorización de Prado Gris.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        component = canonical.publication_events[0].associated_components[0]
        assert component.related_generation_asset_refs == ["generation_asset_1"]

    def test_auxiliary_components_are_aggregated() -> None:
        title = "Autorización de la planta eólica Norte y sus líneas y subestación de evacuación."
        document = _test_document("BOE-A-2026-10002", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Norte", GenerationType.WIND, title
                )],
                associated_components=[
                    AssociatedComponent(
                        local_component_ref="component_1",
                        component_type=AssociatedComponentType.POWER_LINE,
                        description_raw="líneas",
                        related_generation_asset_refs=[],
                        evidence=title,
                    ),
                    AssociatedComponent(
                        local_component_ref="component_2",
                        component_type=AssociatedComponentType.ELECTRICAL_SUBSTATION,
                        description_raw="subestación",
                        related_generation_asset_refs=[],
                        evidence=title,
                    ),
                ],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                )],
                event_summary="Autorización de Norte.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        components = canonical.publication_events[0].associated_components
        assert len(components) == 1
        assert components[0].component_type == AssociatedComponentType.EVACUATION_SYSTEM

    def test_nonliteral_optional_data_is_dropped() -> None:
        title = "Autorización de la planta solar Alba y su infraestructura de evacuación."
        document = _test_document("BOE-A-2026-10003", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "Alba",
                    GenerationType.PHOTOVOLTAIC,
                    title,
                    [TechnicalMention(
                        attribute_type=TechnicalAttributeType.INSTALLED_POWER,
                        value_raw="999 MW",
                        evidence="potencia inventada",
                    )],
                )],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                    names_raw=["SET Alba inventada"],
                    description_raw="descripción reconstruida",
                    related_generation_asset_refs=[],
                    evidence=title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                )],
                event_summary="Autorización de Alba.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        event = canonical.publication_events[0]
        assert event.generation_assets[0].technical_mentions == []
        assert event.associated_components[0].names_raw == []
        assert event.associated_components[0].description_raw is not None

    def test_title_completes_coordinated_aap_aac() -> None:
        title = (
            "Resolución por la que se otorga autorización administrativa previa "
            "de las modificaciones y autorización administrativa de construcción "
            "a la planta fotovoltaica Delta."
        )
        document = _test_document("BOE-A-2026-10004", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Delta", GenerationType.PHOTOVOLTAIC, title
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                )],
                event_summary="Autorizaciones de Delta.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        actions = canonical.publication_events[0].administrative_actions
        by_type = {action.action_type: action for action in actions}
        assert AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION in by_type
        assert AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION in by_type
        assert by_type[AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION].is_modification
        assert not by_type[AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION].is_modification

    def test_historical_action_is_pruned() -> None:
        title = "Resolución por la que se autoriza la construcción de la planta solar Magaz."
        text = (
            f"{title} Mediante Resolución de 2024 se formuló informe ambiental del proyecto."
        )
        document = _test_document("BOE-A-2026-10005", title, text)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Magaz", GenerationType.PHOTOVOLTAIC, title
                )],
                administrative_actions=[
                    _action(
                        AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
                        AdministrativeDecision.AUTHORIZED,
                        title,
                    ),
                    _action(
                        AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
                        AdministrativeDecision.FORMULATED,
                        "Mediante Resolución de 2024 se formuló informe ambiental del proyecto.",
                    ),
                ],
                event_summary="Autorización de construcción de Magaz.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        actions = canonical.publication_events[0].administrative_actions
        assert [action.action_type for action in actions] == [
            AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION
        ]

    def test_independent_plants_are_split() -> None:
        title = "Autorización de las plantas solares Alfa y Beta y su evacuación común."
        document = _test_document("BOE-A-2026-10006", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[
                    _asset("generation_asset_1", "Alfa", GenerationType.PHOTOVOLTAIC, title),
                    _asset("generation_asset_2", "Beta", GenerationType.PHOTOVOLTAIC, title),
                ],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                    description_raw="evacuación común",
                    related_generation_asset_refs=[
                        "generation_asset_1", "generation_asset_2"
                    ],
                    evidence=title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                    ["event"],
                )],
                event_summary="Autorización conjunta de Alfa y Beta.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert len(canonical.publication_events) == 2
        assert all(len(event.generation_assets) == 1 for event in canonical.publication_events)

    def test_same_name_hybrid_relation_is_valid() -> None:
        title = "Hibridación de la instalación Armus Solar con tecnología eólica y fotovoltaica."
        document = _test_document("BOE-A-2026-10007", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[
                    _asset("generation_asset_1", "Armus Solar", GenerationType.WIND, title),
                    _asset("generation_asset_2", "Armus Solar", GenerationType.PHOTOVOLTAIC, title),
                ],
                administrative_actions=[_action(
                    AdministrativeActionType.OTHER,
                    AdministrativeDecision.OTHER,
                    title,
                    ["event"],
                )],
                generation_relations=[GenerationAssetRelation(
                    source_generation_asset_ref="generation_asset_1",
                    target_generation_asset_ref="generation_asset_2",
                    relation_type=GenerationRelationType.HYBRIDIZED_WITH,
                    evidence=title,
                )],
                event_summary="Hibridación de Armus Solar.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert len(canonical.publication_events[0].generation_relations) == 1

    def test_nonliteral_asset_and_action_evidence_are_repaired() -> None:
        title = "Resolución por la que se otorga autorización administrativa previa a la planta fotovoltaica Horizonte."
        document = _test_document("BOE-A-2026-10008", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "Horizonte",
                    GenerationType.PHOTOVOLTAIC,
                    "planta fotovoltaica Horizonte con redacción reconstruida",
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    "se autoriza Horizonte con redacción reconstruida",
                )],
                event_summary="Autorización de Horizonte.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        event = canonical.publication_events[0]
        assert event.generation_assets[0].evidence == title
        assert event.administrative_actions[0].evidence == title

    def test_environmental_public_information_is_not_final_dia() -> None:
        title = (
            "Anuncio por el que se somete a información pública el estudio de "
            "impacto ambiental y la solicitud de la planta solar Vega."
        )
        document = _test_document("BOE-B-2026-10009", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Vega", GenerationType.PHOTOVOLTAIC, title
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
                    AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
                    title,
                )],
                event_summary="Información pública ambiental de Vega.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        action = canonical.publication_events[0].administrative_actions[0]
        assert action.action_type == AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT
        assert action.decision == AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION

    def test_error_correction_does_not_republish_original_action() -> None:
        title = (
            "Corrección de errores de la Resolución por la que se formula la "
            "declaración de impacto ambiental de la planta solar Luna."
        )
        document = _test_document("BOE-A-2026-10010", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Luna", GenerationType.PHOTOVOLTAIC, title
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.ERROR_CORRECTION,
                    AdministrativeDecision.RECTIFIED,
                    title,
                )],
                event_summary="Corrección de errores de la resolución ambiental de Luna.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert [
            action.action_type
            for action in canonical.publication_events[0].administrative_actions
        ] == [AdministrativeActionType.ERROR_CORRECTION]

    def test_descriptive_irrigation_project_is_not_generation_project() -> None:
        title = (
            "Resolución por la que se somete a información pública el Proyecto de "
            "«Implementación de energías renovables mediante paneles fotovoltaicos "
            "flotantes en la Comunidad de Regantes de Balazote - La Herrera»."
        )
        document = _test_document("BOE-B-2022-40999", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    (
                        "Implementación de energías renovables mediante paneles "
                        "fotovoltaicos flotantes en la Comunidad de Regantes de "
                        "Balazote - La Herrera"
                    ),
                    GenerationType.PHOTOVOLTAIC,
                    title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PUBLIC_INFORMATION,
                    AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
                    title,
                )],
                event_summary="Información pública de una obra de regadío.",
            )],
        )
        canonical, _ = canonicalize_project_extraction(
            extraction,
            source_text=title,
            document_title=title,
        )
        assert canonical.document_scope == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        assert canonical.publication_events == []

    def test_contextual_name_inside_evacuation_is_not_generation_target() -> None:
        title = (
            "Anuncio por el que se convoca el levantamiento de actas previas a la "
            "ocupación de fincas afectas por la implantación de la infraestructura "
            "eléctricas de evacuación asociada a la instalación de generación de "
            "energía eléctrica denominada HSF Entrenucleos Ten."
        )
        document = _test_document("BOE-B-2026-17277", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "HSF Entrenucleos Ten",
                    GenerationType.PHOTOVOLTAIC,
                    title,
                )],
                associated_components=[AssociatedComponent(
                    local_component_ref="component_1",
                    component_type=AssociatedComponentType.EVACUATION_SYSTEM,
                    description_raw="infraestructura eléctricas de evacuación",
                    related_generation_asset_refs=["generation_asset_1"],
                    evidence=title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_OCCUPATION_RECORDS,
                    AdministrativeDecision.ANNOUNCED,
                    title,
                )],
                event_summary="Actas previas de la evacuación de Entrenucleos Ten.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert canonical.publication_events[0].administrative_actions[0].targets == [
            "component_1"
        ]

    def test_review_queue_and_manual_precedence() -> None:
        title = "Autorización de la planta fotovoltaica Aurora."
        document = _test_document("BOE-A-2026-10100", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "Aurora",
                    GenerationType.PHOTOVOLTAIC,
                    title,
                )],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                    ["event"],
                )],
                event_summary="Autorización de Aurora.",
            )],
        )
        extraction = _canonicalize_test(document, extraction)
        source_df = pd.DataFrame([{
            "identificador": document.boe_id,
            "fecha_publicacion": pd.Timestamp(document.publication_date),
            "titulo": document.title,
            "texto_limpio": document.text,
            "source_document_sha256": document.source_document_sha256,
        }])
        attempt_record = {
            **{column: pd.NA for column in AI_EXTRACTION_LOG_COLUMNS},
            "attempt_id": "automatic_error",
            "identificador_boe": document.boe_id,
            "fecha_publicacion": pd.Timestamp(document.publication_date),
            "titulo": document.title,
            "source_document_sha256": document.source_document_sha256,
            "extraction_config_id": EXTRACTION_CONFIG_ID,
            "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
            "instructions_sha256": INSTRUCTIONS_SHA256,
            "model_provider": MODEL_PROVIDER,
            "model_name": AI_MODEL_NAME,
            "document_validation_version": DOCUMENT_VALIDATION_VERSION,
            "extraction_json": extraction.model_dump_json(),
            "extracted_at": pd.Timestamp("2026-01-01", tz="UTC"),
            "extraction_status": "error",
            "error_type": "DocumentExtractionValidationError",
            "error_message": "Revisión necesaria.",
            "processing_stage": "document_validation",
            "document_validation_status": "failed",
        }
        attempts = normalise_ai_extraction_attempts_log(
            pd.DataFrame([attempt_record])
        )
        queue = build_review_queue(
            attempts=attempts,
            source_df=source_df,
            manual_reviews=empty_manual_reviews(),
        )
        assert len(queue) == 1

        manual_reviews = normalise_manual_reviews(pd.DataFrame([{
            "manual_review_id": "manual_valid",
            "identificador_boe": document.boe_id,
            "source_document_sha256": document.source_document_sha256,
            "source_attempt_id": "automatic_error",
            "review_status": "manually_validated",
            "corrected_extraction_json": extraction.model_dump_json(),
            "reviewer": "test",
            "review_notes": "Validación de regresión.",
            "reviewed_at": pd.Timestamp("2026-01-02", tz="UTC"),
            "contract_schema_sha256": CONTRACT_SCHEMA_SHA256,
            "document_validation_version": DOCUMENT_VALIDATION_VERSION,
        }]))
        selected = select_best_valid_extractions(
            attempts=attempts,
            source_df=source_df,
            manual_reviews=manual_reviews,
        )
        assert len(selected) == 1
        assert selected.iloc[0]["selection_source"] == "manually_validated"
        assert build_review_queue(
            attempts=attempts,
            source_df=source_df,
            manual_reviews=manual_reviews,
        ).empty


    def test_modified_environmental_resolution_sets_flag() -> None:
        title = (
            "Resolución por la que se modifica la de 28 de febrero de 2018, "
            "por la que se formula declaración de impacto ambiental sobre el "
            "proyecto Parque Eólico Campillo."
        )
        document = _test_document("BOE-A-2022-24405", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "Parque Eólico Campillo",
                    GenerationType.WIND,
                    title,
                )],
                associated_components=[],
                administrative_actions=[_action(
                    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT,
                    AdministrativeDecision.FORMULATED,
                    title,
                )],
                event_summary="Modificación de la declaración ambiental.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert canonical.publication_events[0].administrative_actions[0].is_modification is True

    def test_contract_procurement_is_not_generation_project() -> None:
        title = (
            "Anuncio de formalización de contratos. Objeto: Contratación del "
            "suministro e instalación de placas fotovoltaicas para autoconsumo "
            "en el edificio sede."
        )
        document = _test_document("BOE-B-2022-40990", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "placas fotovoltaicas para autoconsumo en el edificio sede",
                    GenerationType.PHOTOVOLTAIC,
                    title,
                )],
                associated_components=[],
                administrative_actions=[_action(
                    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                    ["event"],
                )],
                event_summary="Instalación de placas en un edificio.",
            )],
        )
        canonical, adjustments = canonicalize_project_extraction(
            extraction,
            source_text=title,
            document_title=title,
        )
        assert canonical.document_scope == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        assert canonical.publication_events == []
        assert any("Alcance corregido" in item for item in adjustments)

    def test_auxiliary_pv_in_water_project_is_not_generation_project() -> None:
        title = (
            "Resolución por la que se formula informe de impacto ambiental del "
            "proyecto Construcción para la mejora del abastecimiento de agua."
        )
        source = title + " Se instala una planta solar fotovoltaica de 60 kWp para alimentar el bombeo."
        document = _test_document("BOE-A-2026-10868", title, source)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "planta solar fotovoltaica",
                    GenerationType.PHOTOVOLTAIC,
                    "planta solar fotovoltaica de 60 kWp",
                )],
                associated_components=[],
                administrative_actions=[_action(
                    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT,
                    AdministrativeDecision.FORMULATED,
                    title,
                    ["event"],
                )],
                event_summary="Informe ambiental de abastecimiento.",
            )],
        )
        canonical, _ = canonicalize_project_extraction(
            extraction,
            source_text=source,
            document_title=title,
        )
        assert canonical.document_scope == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        assert canonical.publication_events == []

    def test_gas_infrastructure_is_preclassified_without_model() -> None:
        title = (
            "Resolución por la que se otorga autorización administrativa y "
            "declaración de utilidad pública del proyecto Anexo al gasoducto "
            "Salamanca-Zamora. Nueva posición O-12X con estación de medida "
            "G-65 para inyección de biometano."
        )
        document = _test_document("BOE-A-2026-10656", title)
        extraction, adjustments = preclassify_document_without_model(document)
        assert extraction is not None
        assert (
            extraction.document_scope
            == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        )
        assert extraction.publication_events == []
        assert any("antes de llamar al modelo" in item for item in adjustments)
        validate_extraction_against_document(
            document=document,
            extraction=extraction,
        )


    def test_contract_procurement_is_preclassified_without_model() -> None:
        title = (
            "Anuncio de formalización de contratos de un suministro e instalación "
            "de placas fotovoltaicas para autoconsumo en un edificio público."
        )
        document = _test_document("BOE-B-2022-40990", title)
        extraction, adjustments = preclassify_document_without_model(document)
        assert extraction is not None
        assert (
            extraction.document_scope
            == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        )
        assert extraction.publication_events == []
        assert any("antes de llamar al modelo" in item for item in adjustments)


    def test_standalone_storage_is_preclassified_without_model() -> None:
        title = (
            "Resolución por la que se otorga autorización administrativa previa "
            "para la planta de almacenamiento de energía Glauco Almacena, de "
            "55,384 MW, y su infraestructura de evacuación."
        )
        document = _test_document("BOE-A-2026-10655", title)
        extraction, adjustments = preclassify_document_without_model(document)
        assert extraction is not None
        assert (
            extraction.document_scope
            == DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS
        )
        assert extraction.publication_events == []
        assert any("almacenamiento" in item.casefold() for item in adjustments)


    def test_storage_linked_to_generation_is_not_preclassified() -> None:
        title = (
            "Anuncio por el que se somete a información pública el módulo de "
            "almacenamiento BESS Hibridación FV Andévalo, asociado a la planta "
            "fotovoltaica FV Andévalo."
        )
        document = _test_document("BOE-B-2026-99992", title)
        extraction, adjustments = preclassify_document_without_model(document)
        assert extraction is None
        assert adjustments == []


    def test_quality_metric_combination_has_no_future_warning() -> None:
        existing = empty_quality_metrics()
        new_metric = _normalise_table(
            pd.DataFrame([{
                "quality_run_id": "quality_test",
                "run_scope": "pilot",
                "n_source_documents": 1,
                "quality_alert": False,
            }]),
            QUALITY_METRIC_COLUMNS,
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", FutureWarning)
            combined = combine_quality_metrics(existing, new_metric)
        assert len(combined) == 1
        assert not any(
            issubclass(item.category, FutureWarning)
            for item in caught
        )


    def test_attempt_log_combination_has_no_future_warning() -> None:
        existing = normalise_ai_extraction_attempts_log(
            pd.DataFrame([{
                "attempt_id": "attempt_1",
                "identificador_boe": "BOE-A-TEST-1",
                "extraction_status": "ok",
                "error_type": pd.NA,
                "error_message": pd.NA,
                "validation_issues_json": pd.NA,
            }])
        )
        new_attempts = normalise_ai_extraction_attempts_log(
            pd.DataFrame([{
                "attempt_id": "attempt_2",
                "identificador_boe": "BOE-A-TEST-2",
                "extraction_status": "ok",
                "error_type": pd.NA,
                "error_message": pd.NA,
                "validation_issues_json": pd.NA,
            }])
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", FutureWarning)
            combined = combine_ai_extraction_attempt_frames(
                existing,
                new_attempts,
            )
        assert set(combined["attempt_id"].astype(str)) == {
            "attempt_1",
            "attempt_2",
        }
        assert not any(
            issubclass(item.category, FutureWarning)
            for item in caught
        )


    def test_public_information_decisions_are_canonical() -> None:
        title = (
            "Anuncio por el que se somete a información pública la solicitud de "
            "autorización administrativa previa y autorización administrativa "
            "de construcción de la planta fotovoltaica Prueba."
        )
        document = _test_document("BOE-B-2026-99991", title)
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1", "Prueba", GenerationType.PHOTOVOLTAIC, title
                )],
                associated_components=[],
                administrative_actions=[
                    _action(
                        AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                        AdministrativeDecision.REQUESTED,
                        title,
                    ),
                    _action(
                        AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
                        AdministrativeDecision.REQUESTED,
                        title,
                    ),
                    _action(
                        AdministrativeActionType.PUBLIC_INFORMATION,
                        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
                        title,
                    ),
                ],
                event_summary="Información pública de autorizaciones.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        actions = canonical.publication_events[0].administrative_actions
        assert {a.action_type for a in actions} == {
            AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
            AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION,
        }
        assert all(
            a.decision == AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION
            for a in actions
        )

    def test_water_concession_is_not_prior_authorization() -> None:
        title = (
            "Resolución por la que se otorga la concesión para el aprovechamiento "
            "de agua con destino a producción de energía eléctrica."
        )
        document = _test_document("BOE-B-2023-19087", title, title + " Central Hidroeléctrica Navaleo.")
        extraction = _test_extraction(
            document.boe_id,
            [PublicationEvent(
                generation_assets=[_asset(
                    "generation_asset_1",
                    "Central Hidroeléctrica Navaleo",
                    GenerationType.HYDROPOWER,
                    "Central Hidroeléctrica Navaleo",
                )],
                associated_components=[],
                administrative_actions=[_action(
                    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION,
                    AdministrativeDecision.AUTHORIZED,
                    title,
                )],
                event_summary="Concesión hidroeléctrica.",
            )],
        )
        canonical = _canonicalize_test(document, extraction)
        assert canonical.publication_events[0].administrative_actions[0].action_type == AdministrativeActionType.WATER_CONCESSION

    for name, function in [
        ("modified_environmental_resolution", test_modified_environmental_resolution_sets_flag),
        ("scope_contract_procurement", test_contract_procurement_is_not_generation_project),
        ("scope_auxiliary_pv_water", test_auxiliary_pv_in_water_project_is_not_generation_project),
        ("scope_gas_infrastructure_pre_model", test_gas_infrastructure_is_preclassified_without_model),
        ("scope_contract_procurement_pre_model", test_contract_procurement_is_preclassified_without_model),
        ("scope_standalone_storage_pre_model", test_standalone_storage_is_preclassified_without_model),
        ("scope_storage_linked_generation", test_storage_linked_to_generation_is_not_preclassified),
        ("quality_metric_no_future_warning", test_quality_metric_combination_has_no_future_warning),
        ("attempt_log_no_future_warning", test_attempt_log_combination_has_no_future_warning),
        ("public_information_decisions", test_public_information_decisions_are_canonical),
        ("water_concession", test_water_concession_is_not_prior_authorization),
        ("carbo_event_target", test_carbo_event_target),
        ("entrenucleos_component_only", test_entrenucleos_component_only),
        ("armus_storage_and_evacuation_targets", test_armus_storage_and_evacuation_targets),
        ("component_link_contextual", test_component_link_does_not_require_same_quote),
        ("aggregate_auxiliary_components", test_auxiliary_components_are_aggregated),
        ("drop_nonliteral_optional_data", test_nonliteral_optional_data_is_dropped),
        ("complete_coordinated_aap_aac", test_title_completes_coordinated_aap_aac),
        ("prune_historical_action", test_historical_action_is_pruned),
        ("split_independent_plants", test_independent_plants_are_split),
        ("same_name_hybrid_relation", test_same_name_hybrid_relation_is_valid),
        ("repair_nonliteral_asset_action_evidence", test_nonliteral_asset_and_action_evidence_are_repaired),
        ("environmental_public_information", test_environmental_public_information_is_not_final_dia),
        ("error_correction_only", test_error_correction_does_not_republish_original_action),
        ("descriptive_irrigation_project_scope", test_descriptive_irrigation_project_is_not_generation_project),
        ("contextual_generation_name_not_target", test_contextual_name_inside_evacuation_is_not_generation_target),
        ("review_queue_manual_precedence", test_review_queue_and_manual_precedence),
    ]:
        record(name, function)

    return pd.DataFrame(rows)


if RUN_DETERMINISTIC_REGRESSION_TESTS:
    deterministic_regression_results = _run_regression_tests()
    display(deterministic_regression_results)
    failed_regressions = deterministic_regression_results.loc[
        ~deterministic_regression_results["passed"]
    ]
    if not failed_regressions.empty:
        raise AssertionError(
            "Fallaron regresiones deterministas:\n"
            + failed_regressions.to_string(index=False)
        )


,test,passed,error
0,modified_environmental_resolution,True,None
1,scope_contract_procurement,True,None
2,scope_auxiliary_pv_water,True,None
3,scope_gas_infrastructure_pre_model,True,None
4,scope_contract_procurement_pre_model,True,None
5,scope_standalone_storage_pre_model,True,None
6,scope_storage_linked_generation,True,None
7,quality_metric_no_future_warning,True,None
8,attempt_log_no_future_warning,True,None
9,public_information_decisions,True,None


## 10. Piloto estratificado de 100 documentos: validación estructural y de alcance (opt-in)

El piloto utiliza una muestra estratificada y congelada, independiente de la
versión del prompt. No incorpora revisiones manuales: mide exclusivamente la
calidad automática. Se considera superado únicamente cuando los 100 documentos
tienen un intento vigente, una extracción automática válida y cero fallos de
las invariantes generales.

Los casos conocidos difíciles se controlan en la suite de regresión anterior;
no se fuerzan dentro de la muestra del piloto.


El piloto utiliza un archivo de etiquetas revisadas manualmente exclusivamente para evaluar `document_scope`. Estas etiquetas no se pasan al modelo, no intervienen en la canonicalización y no seleccionan la muestra. Evitan declarar éxito cuando el JSON es válido pero un documento fuera del alcance se ha convertido en proyecto.


### Sobre la celda roja al final del piloto

La instrucción `raise AssertionError(...)` no es un defecto sintáctico. Es una
barrera deliberada: VS Code marca la celda en rojo cuando `test_passed=False`.
No debe eliminarse ni desactivarse para ocultar un fallo. Con una ejecución
correcta, la condición no se cumple y la celda termina sin error.

En la muestra congelada actual se esperan **43 documentos específicos de
proyectos de generación** y **57 documentos no relevantes**. Las instalaciones
autónomas de almacenamiento y los anuncios de contratación se descartan antes
de llamar al modelo.

### Qué demuestra y qué no demuestra el piloto

`test_passed=True` acredita que los 100 documentos:

- tienen una extracción vigente;
- cumplen el contrato Pydantic y las invariantes documentales;
- coinciden con la etiqueta revisada de alcance;
- no dejan plantas sin nombre, eventos sin actuación ni componentes sin vínculo.

No acredita por sí solo que todos los campos opcionales sean exhaustivos. La tabla
`pilot_boe_ai_semantic_audit.parquet` se genera para revisar de forma transparente
los nombres, tecnologías, componentes y actuaciones de los documentos específicos.


In [11]:

RUN_STRATIFIED_PILOT = True
RESET_PILOT_OUTPUTS = False  # Después de superar el piloto RESET_PILOT_OUTPUTS = False
REBUILD_PILOT_SAMPLE = False
RAISE_ON_PILOT_FAILURE = True

PILOT_SAMPLE_SIZE = 100
PILOT_RANDOM_SEED = 20260720
PILOT_CHECKPOINT_EVERY = 1
PILOT_MIN_AUTO_VALIDATION_RATE = 1.0

PILOT_SAMPLE_PATH = SILVER_BOE_AI_DIR / "pilot_sample_100.parquet"
PILOT_ATTEMPTS_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_extraction_attempts.parquet"
)
PILOT_CURRENT_PATH = SILVER_BOE_AI_DIR / "pilot_boe_ai_extractions.parquet"
PILOT_REVIEW_QUEUE_PATH = SILVER_BOE_AI_DIR / "pilot_boe_ai_review_queue.parquet"
PILOT_QUALITY_METRICS_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_quality_metrics.parquet"
)
PILOT_SCOPE_LABELS_PATH = SILVER_BOE_AI_DIR / "pilot_scope_labels_100.csv"
PILOT_SCOPE_LABELS_FALLBACK_PATH = Path.cwd() / "pilot_scope_labels_100.csv"
PILOT_SEMANTIC_AUDIT_PATH = (
    SILVER_BOE_AI_DIR / "pilot_boe_ai_semantic_audit.parquet"
)
PILOT_MIN_SCOPE_ACCURACY = 1.0


def load_pilot_scope_labels(
    *,
    expected_sample_ids: set[str],
) -> pd.DataFrame:
    candidate_paths = [
        PILOT_SCOPE_LABELS_PATH,
        PILOT_SCOPE_LABELS_FALLBACK_PATH,
    ]
    path = next((item for item in candidate_paths if item.exists()), None)
    if path is None:
        raise FileNotFoundError(
            "No se encontró pilot_scope_labels_100.csv. Colócalo en "
            f"{PILOT_SCOPE_LABELS_PATH} o en {PILOT_SCOPE_LABELS_FALLBACK_PATH}."
        )
    labels = pd.read_csv(path, dtype={"identificador_boe": "string"})
    validate_required_columns(
        labels,
        {"identificador_boe", "expected_document_scope"},
    )
    if labels["identificador_boe"].duplicated().any():
        raise ValueError("Las etiquetas del piloto contienen BOE duplicados.")
    label_ids = set(labels["identificador_boe"].astype(str))
    if label_ids != expected_sample_ids:
        raise ValueError(
            "Las etiquetas no coinciden exactamente con la muestra congelada. "
            f"faltan={sorted(expected_sample_ids - label_ids)}, "
            f"sobran={sorted(label_ids - expected_sample_ids)}."
        )
    valid_scopes = {item.value for item in DocumentScope}
    invalid = set(labels["expected_document_scope"].dropna().astype(str)) - valid_scopes
    if invalid:
        raise ValueError(f"Etiquetas de alcance inválidas: {sorted(invalid)}")
    return labels.copy()


def _pilot_topic_group(title: str) -> str:
    key = _canonical_documentary_text(title).casefold()
    for group, pattern in [
        ("hybrid_storage", r"hibrid|almacen|bess|bater"),
        ("grid", r"subestaci|l[ií]nea|evacuaci|conexi"),
        ("environmental", r"impacto ambiental|afecci[oó]n ambiental"),
        ("public_utility", r"utilidad p[uú]blica|expropi|ocupaci[oó]n"),
        ("authorizations", r"autorizaci[oó]n administrativa"),
    ]:
        if re.search(pattern, key):
            return group
    return "other"


def build_stratified_pilot_sample(
    source_df: pd.DataFrame,
    *,
    sample_size: int,
) -> pd.DataFrame:
    if sample_size < 1:
        raise ValueError("sample_size debe ser mayor que cero.")
    if sample_size > len(source_df):
        raise ValueError(
            f"La muestra solicitada ({sample_size}) supera el corpus ({len(source_df)})."
        )

    pilot = source_df.copy()
    pilot["source_text_chars"] = pilot["texto_limpio"].astype("string").str.len()
    pilot["publication_year"] = pd.to_datetime(
        pilot["fecha_publicacion"], errors="coerce"
    ).dt.year.astype("Int64")
    pilot["length_band"] = pd.cut(
        pilot["source_text_chars"],
        bins=[0, 20_000, 50_000, 100_000, float("inf")],
        labels=["short", "medium", "long", "very_long"],
        include_lowest=True,
    ).astype("string")
    pilot["topic_group"] = pilot["titulo"].astype(str).map(_pilot_topic_group)
    pilot["stratum"] = (
        pilot["publication_year"].astype("string")
        + "|"
        + pilot["length_band"]
        + "|"
        + pilot["topic_group"]
    )
    # La muestra no cambia cuando cambia el prompt o la versión del validador.
    pilot["sample_priority"] = pilot["identificador"].astype(str).map(
        lambda boe_id: sha256(
            f"{PILOT_RANDOM_SEED}|{boe_id}".encode("utf-8")
        ).hexdigest()
    )

    groups = [
        group.sort_values("sample_priority", kind="stable")
        for _, group in pilot.groupby("stratum", sort=True, dropna=False)
    ]
    selected_indices: list[Any] = []
    for round_index in range(max(len(group) for group in groups)):
        for group in groups:
            if round_index < len(group):
                selected_indices.append(group.index[round_index])
            if len(selected_indices) >= sample_size:
                break
        if len(selected_indices) >= sample_size:
            break

    selected = pilot.loc[selected_indices].copy()
    if len(selected) != sample_size:
        raise AssertionError(
            f"No se pudo construir la muestra completa: {len(selected)} != {sample_size}."
        )
    return selected.reset_index(drop=True)


def load_or_build_pilot_sample(
    source_df: pd.DataFrame,
    *,
    sample_size: int,
    path: Path = PILOT_SAMPLE_PATH,
    rebuild: bool = False,
) -> pd.DataFrame:
    if path.exists() and not rebuild:
        stored = pd.read_parquet(path)
        validate_required_columns(stored, {"identificador"})
        stored_ids = stored["identificador"].astype(str)
        if len(stored_ids) != sample_size or stored_ids.duplicated().any():
            raise ValueError(
                "La muestra congelada no coincide con PILOT_SAMPLE_SIZE. "
                "Usa REBUILD_PILOT_SAMPLE=True para regenerarla deliberadamente."
            )
        missing = set(stored_ids) - set(source_df["identificador"].astype(str))
        if missing:
            raise ValueError(
                f"La muestra congelada contiene BOE ausentes del corpus: {sorted(missing)}"
            )
        order = {boe_id: index for index, boe_id in enumerate(stored_ids)}
        selected = source_df.loc[
            source_df["identificador"].astype(str).isin(order)
        ].copy()
        selected["_pilot_order"] = selected["identificador"].astype(str).map(order)
        return (
            selected.sort_values("_pilot_order", kind="stable")
            .drop(columns="_pilot_order")
            .reset_index(drop=True)
        )

    selected = build_stratified_pilot_sample(
        source_df,
        sample_size=sample_size,
    )
    sample_columns = [
        "identificador",
        "fecha_publicacion",
        "titulo",
        "publication_year",
        "length_band",
        "topic_group",
        "stratum",
        "sample_priority",
    ]
    save_parquet_atomic(selected[sample_columns], path)
    return selected


def evaluate_pilot(
    pilot_sample: pd.DataFrame,
    attempts: pd.DataFrame,
    current: pd.DataFrame,
    scope_labels: pd.DataFrame,
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    selected_ids = set(pilot_sample["identificador"].astype(str))
    current_ids = set(current["identificador_boe"].astype(str))
    latest_attempts = _latest_attempts_for_current_sources(
        attempts,
        pilot_sample,
    )
    latest_ids = set(latest_attempts["identificador_boe"].astype(str))

    source_rows = {
        str(row["identificador"]): row
        for _, row in pilot_sample.iterrows()
    }
    expected_scope_by_id = dict(zip(
        scope_labels["identificador_boe"].astype(str),
        scope_labels["expected_document_scope"].astype(str),
        strict=True,
    ))
    checks: list[dict[str, Any]] = []
    scope_checks: list[dict[str, Any]] = []

    for boe_id in sorted(selected_ids & current_ids):
        extraction = get_extraction_model(current, boe_id)
        predicted_scope = (
            extraction.document_scope.value
            if extraction.document_scope is not None
            else None
        )
        expected_scope = expected_scope_by_id[boe_id]
        generation_assets = [
            asset
            for event in extraction.publication_events
            for asset in event.generation_assets
        ]
        components = [
            component
            for event in extraction.publication_events
            for component in event.associated_components
        ]
        actions = [
            action
            for event in extraction.publication_events
            for action in event.administrative_actions
        ]
        current_row = current.loc[
            current["identificador_boe"].astype(str).eq(boe_id)
        ].iloc[-1]

        scope_checks.append({
            "identificador_boe": boe_id,
            "titulo": str(source_rows[boe_id]["titulo"]),
            "expected_document_scope": expected_scope,
            "predicted_document_scope": predicted_scope,
            "n_publication_events": len(extraction.publication_events),
            "n_generation_assets": len(generation_assets),
            "n_associated_components": len(components),
            "n_administrative_actions": len(actions),
            "generation_asset_names_json": json.dumps(
                [asset.names_raw for asset in generation_assets],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "generation_types_json": json.dumps(
                [asset.generation_type.value for asset in generation_assets],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "component_types_json": json.dumps(
                [component.component_type.value for component in components],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "action_types_json": json.dumps(
                [action.action_type.value for action in actions],
                ensure_ascii=False,
                sort_keys=True,
            ),
            "processing_stage": current_row.get("processing_stage"),
            "classification_reason": extraction.classification_reason,
            "scope_match": predicted_scope == expected_scope,
        })
        try:
            source_document = build_source_document(source_rows[boe_id])
            validate_extraction_against_document(
                document=source_document,
                extraction=extraction,
            )

            for event in extraction.publication_events:
                if not event.generation_assets:
                    raise AssertionError("Evento sin planta de generación.")
                if not event.administrative_actions:
                    raise AssertionError("Evento sin actuación administrativa.")
                if any(not asset.names_raw for asset in event.generation_assets):
                    raise AssertionError("Planta sin nombre documental.")
                if any(
                    not component.related_generation_asset_refs
                    for component in event.associated_components
                ):
                    raise AssertionError("Componente sin vínculo con una planta.")
                if any(
                    not action.targets
                    for action in event.administrative_actions
                ):
                    raise AssertionError("Actuación sin destinatario.")

            passed = True
            reason = None
        except Exception as error:
            passed = False
            reason = str(error)

        checks.append({
            "identificador_boe": boe_id,
            "check": "document_contract_and_tfm_core_invariants",
            "passed": passed,
            "reason": reason,
        })

    checks_df = pd.DataFrame(
        checks,
        columns=["identificador_boe", "check", "passed", "reason"],
    )
    n_errors = int(
        latest_attempts["extraction_status"].ne("ok").fillna(True).sum()
    )
    semantic_failures = (
        int((~checks_df["passed"]).sum())
        if not checks_df.empty
        else len(selected_ids)
    )
    auto_validation_rate = (
        len(current_ids & selected_ids) / len(selected_ids)
        if selected_ids
        else 1.0
    )
    scope_checks_df = pd.DataFrame(scope_checks)
    specific_scope = DocumentScope.GENERATION_PROJECT_SPECIFIC.value
    not_relevant_scope = (
        DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
    )
    scope_event_inconsistency = (
        scope_checks_df["predicted_document_scope"].eq(specific_scope)
        & scope_checks_df["n_publication_events"].eq(0)
    ) | (
        scope_checks_df["predicted_document_scope"].eq(not_relevant_scope)
        & scope_checks_df["n_publication_events"].gt(0)
    )
    n_scope_event_inconsistencies = int(scope_event_inconsistency.sum())

    n_scope_mismatches = (
        int((~scope_checks_df["scope_match"]).sum())
        if not scope_checks_df.empty
        else len(selected_ids)
    )
    scope_accuracy = (
        float(scope_checks_df["scope_match"].mean())
        if not scope_checks_df.empty
        else 0.0
    )
    test_passed = bool(
        len(selected_ids) == PILOT_SAMPLE_SIZE
        and latest_ids == selected_ids
        and current_ids == selected_ids
        and n_errors == 0
        and semantic_failures == 0
        and n_scope_mismatches == 0
        and n_scope_event_inconsistencies == 0
        and auto_validation_rate >= PILOT_MIN_AUTO_VALIDATION_RATE
        and scope_accuracy >= PILOT_MIN_SCOPE_ACCURACY
    )
    summary = {
        "status": "passed" if test_passed else "failed",
        "test_passed": test_passed,
        "n_selected_documents": len(selected_ids),
        "n_latest_attempts": len(latest_ids),
        "n_canonical_extractions": len(current_ids),
        "n_missing_latest_attempts": len(selected_ids - latest_ids),
        "n_missing_canonical_extractions": len(selected_ids - current_ids),
        "n_unexpected_latest_attempts": len(latest_ids - selected_ids),
        "n_unexpected_canonical_extractions": len(current_ids - selected_ids),
        "n_errors": n_errors,
        "n_semantic_failures": semantic_failures,
        "automatic_validation_rate": auto_validation_rate,
        "n_scope_evaluated": len(scope_checks_df),
        "n_scope_mismatches": n_scope_mismatches,
        "scope_accuracy": scope_accuracy,
        "n_scope_event_inconsistencies": n_scope_event_inconsistencies,
        "n_zero_event_documents": int(
            scope_checks_df["n_publication_events"].eq(0).sum()
        ),
        "n_project_specific_documents": int(
            scope_checks_df["predicted_document_scope"].eq(
                DocumentScope.GENERATION_PROJECT_SPECIFIC.value
            ).sum()
        ),
        "n_not_relevant_documents": int(
            scope_checks_df["predicted_document_scope"].eq(
                DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
            ).sum()
        ),
        "n_total_publication_events": int(
            scope_checks_df["n_publication_events"].sum()
        ),
        "n_total_generation_assets": int(
            scope_checks_df["n_generation_assets"].sum()
        ),
        "n_total_associated_components": int(
            scope_checks_df["n_associated_components"].sum()
        ),
        "n_total_administrative_actions": int(
            scope_checks_df["n_administrative_actions"].sum()
        ),
        "n_deterministic_scope_guards": int(
            scope_checks_df["processing_stage"]
            .eq("deterministic_scope_guard")
            .sum()
        ),
    }
    return summary, checks_df, scope_checks_df


if RUN_STRATIFIED_PILOT:
    candidates = load_and_prepare_candidates()
    pilot_sample = load_or_build_pilot_sample(
        candidates,
        sample_size=PILOT_SAMPLE_SIZE,
        rebuild=REBUILD_PILOT_SAMPLE,
    )

    pilot_scope_labels = load_pilot_scope_labels(
        expected_sample_ids=set(pilot_sample["identificador"].astype(str)),
    )

    if RESET_PILOT_OUTPUTS:
        for path in (
            PILOT_ATTEMPTS_PATH,
            PILOT_CURRENT_PATH,
            PILOT_REVIEW_QUEUE_PATH,
        ):
            path.unlink(missing_ok=True)

    attempts = load_ai_extraction_attempts(PILOT_ATTEMPTS_PATH)
    pilot_manual_reviews = empty_manual_reviews()
    current, pending = build_pending_candidates(
        pilot_sample,
        attempts,
        manual_reviews=pilot_manual_reviews,
    )
    pilot_result = await run_and_finalize_extractions(
        pending,
        pilot_sample,
        agent=agent,
        attempts_path=PILOT_ATTEMPTS_PATH,
        current_path=PILOT_CURRENT_PATH,
        review_queue_path=PILOT_REVIEW_QUEUE_PATH,
        quality_metrics_path=PILOT_QUALITY_METRICS_PATH,
        manual_reviews=pilot_manual_reviews,
        run_scope="pilot",
        minimum_auto_validation_rate=PILOT_MIN_AUTO_VALIDATION_RATE,
        checkpoint_every=PILOT_CHECKPOINT_EVERY,
    )
    pilot_summary, pilot_checks, pilot_scope_checks = evaluate_pilot(
        pilot_sample,
        pilot_result["all_attempts"],
        pilot_result["current_extractions"],
        pilot_scope_labels,
    )

    pilot_quality_metric = pilot_result["quality_metric"].copy()
    pilot_quality_metric["n_scope_evaluated"] = pilot_summary["n_scope_evaluated"]
    pilot_quality_metric["n_scope_mismatches"] = pilot_summary["n_scope_mismatches"]
    pilot_quality_metric["scope_accuracy"] = pilot_summary["scope_accuracy"]
    pilot_quality_metric["minimum_scope_accuracy"] = PILOT_MIN_SCOPE_ACCURACY
    pilot_quality_metric["quality_alert"] = (
        pilot_quality_metric["quality_alert"].fillna(False)
        | (pilot_summary["scope_accuracy"] < PILOT_MIN_SCOPE_ACCURACY)
    )
    pilot_quality_metric["quality_status"] = pilot_quality_metric["quality_alert"].map(
        {True: "degraded", False: "healthy"}
    )
    append_quality_metric(pilot_quality_metric, PILOT_QUALITY_METRICS_PATH)

    save_parquet_atomic(
        pilot_scope_checks,
        PILOT_SEMANTIC_AUDIT_PATH,
    )

    expected_distribution = (
        pilot_scope_checks["expected_document_scope"]
        .value_counts(dropna=False)
        .rename("expected_count")
    )
    predicted_distribution = (
        pilot_scope_checks["predicted_document_scope"]
        .value_counts(dropna=False)
        .rename("predicted_count")
    )
    pilot_scope_distribution = (
        pd.concat(
            [expected_distribution, predicted_distribution],
            axis=1,
        )
        .fillna(0)
        .astype(int)
        .rename_axis("document_scope")
        .reset_index()
    )

    print(
        "Composición del piloto: "
        f"{pilot_summary['n_project_specific_documents']} documentos "
        "específicos de proyectos y "
        f"{pilot_summary['n_not_relevant_documents']} no relevantes."
    )
    display(pd.DataFrame([pilot_summary]))
    display(pilot_scope_distribution)

    print("Fallos de invariantes documentales o del contrato:")
    display(pilot_checks.loc[~pilot_checks["passed"]])

    print("Discordancias respecto de las etiquetas de alcance:")
    display(pilot_scope_checks.loc[~pilot_scope_checks["scope_match"]])

    print("Documentos específicos que alimentarán agrupamiento y cronología:")
    display(
        pilot_scope_checks.loc[
            pilot_scope_checks["predicted_document_scope"].eq(
                DocumentScope.GENERATION_PROJECT_SPECIFIC.value
            ),
            [
                "identificador_boe",
                "titulo",
                "n_publication_events",
                "n_generation_assets",
                "n_associated_components",
                "n_administrative_actions",
                "generation_asset_names_json",
                "generation_types_json",
                "action_types_json",
                "processing_stage",
            ],
        ]
    )

    print(
        "Documentos no relevantes: por contrato deben tener cero eventos. "
        "Esta tabla no representa los 100 documentos del piloto."
    )
    display(
        pilot_scope_checks.loc[
            pilot_scope_checks["predicted_document_scope"].eq(
                DocumentScope.NOT_RELEVANT_FOR_GENERATION_PROJECTS.value
            ),
            [
                "identificador_boe",
                "titulo",
                "classification_reason",
                "processing_stage",
                "scope_match",
            ],
        ]
    )
    display(pilot_result["review_queue"])
    display(pilot_quality_metric)

    pilot_passed = bool(pilot_summary.get("test_passed", False))
    if RAISE_ON_PILOT_FAILURE and not pilot_passed:
        failed_dimensions = {
            key: pilot_summary[key]
            for key in (
                "n_errors",
                "n_semantic_failures",
                "n_scope_mismatches",
                "n_scope_event_inconsistencies",
                "automatic_validation_rate",
                "scope_accuracy",
            )
        }
        raise AssertionError(
            "El piloto estratificado no se considera superado. "
            f"Controles fallidos: {failed_dimensions}. "
            f"Resumen completo: {pilot_summary}"
        )


Composición del piloto: 43 documentos específicos de proyectos y 57 no relevantes.


,status,test_passed,n_selected_documents,n_latest_attempts,n_canonical_extractions,n_missing_latest_attempts,n_missing_canonical_extractions,n_unexpected_latest_attempts,n_unexpected_canonical_extractions,n_errors,...,scope_accuracy,n_scope_event_inconsistencies,n_zero_event_documents,n_project_specific_documents,n_not_relevant_documents,n_total_publication_events,n_total_generation_assets,n_total_associated_components,n_total_administrative_actions,n_deterministic_scope_guards
0,passed,True,100,100,100,0,0,0,0,0,...,1.0,0,57,43,57,49,59,53,73,14


,document_scope,expected_count,predicted_count
0,not_relevant_for_generation_projects,57,57
1,generation_project_specific,43,43


Fallos de invariantes documentales o del contrato:


,identificador_boe,check,passed,reason


...

,identificador_boe,titulo,n_publication_events,n_generation_assets,n_associated_components,n_administrative_actions,generation_asset_names_json,generation_types_json,action_types_json,processing_stage
3,BOE-A-2022-24404,"Resolución de 22 de diciembre de 2022, de la D...",1,1,1,1,"[[""Planta fotovoltaica híbrida Majal Alto"", ""P...","[""fotovoltaica""]","[""declaracion_impacto_ambiental""]",completed
4,BOE-A-2022-24405,"Resolución de 22 de diciembre de 2022, de la D...",1,1,1,1,"[[""Parque Eólico Campillo de Altobuey, Fase I""]]","[""eolica""]","[""declaracion_impacto_ambiental""]",completed
6,BOE-A-2023-10297,"Resolución de 14 de abril de 2023, de la Direc...",1,2,1,1,"[[""Planta fotovoltaica hibridación PE Angostil...","[""fotovoltaica"", ""eolica""]","[""informe_determinacion_afeccion_ambiental""]",completed
7,BOE-A-2023-10301,"Resolución de 17 de abril de 2023, de la Direc...",1,1,1,1,"[[""FV Garoña Alfacuarta""]]","[""fotovoltaica""]","[""autorizacion_administrativa_previa""]",completed
8,BOE-A-2023-10303,"Resolución de 17 de abril de 2023, de la Direc...",1,1,1,1,"[[""parque eólico La Senda""]]","[""eolica""]","[""autorizacion_administrativa_previa""]",completed
9,BOE-A-2023-1935,"Resolución de 12 de enero de 2023, de la Direc...",3,3,3,3,"[[""Los Quincetos""], [""El Espino""], [""Las Coron...","[""fotovoltaica"", ""fotovoltaica"", ""fotovoltaica""]","[""declaracion_impacto_ambiental"", ""declaracion...",completed
10,BOE-A-2023-1936,"Resolución de 12 de enero de 2023, de la Direc...",1,1,1,1,"[[""Planta solar fotovoltaica Aldehuela""]]","[""fotovoltaica""]","[""declaracion_impacto_ambiental""]",completed
11,BOE-A-2023-1939,"Resolución de 16 de enero de 2023, de la Direc...",1,1,1,1,"[[""Planta fotovoltaica Elawan Ayora III""]]","[""fotovoltaica""]","[""informe_determinacion_afeccion_ambiental""]",completed
12,BOE-A-2023-1941,"Resolución de 17 de enero de 2023, de la Direc...",1,2,1,1,"[[""Planta híbrida fotovoltaica Dehesilla I""], ...","[""fotovoltaica"", ""eolica""]","[""informe_determinacion_afeccion_ambiental""]",completed
13,BOE-A-2023-1942,"Resolución de 17 de enero de 2023, de la Direc...",1,1,1,1,"[[""Parque eólico Carballoso""]]","[""eolica""]","[""informe_determinacion_afeccion_ambiental""]",completed


Documentos no relevantes: por contrato deben tener cero eventos. Esta tabla no representa los 100 documentos del piloto.


,identificador_boe,titulo,classification_reason,processing_stage,scope_match
0,BOE-A-2022-23752,"Orden TED/1315/2022, de 23 de diciembre, por l...",El documento establece un marco regulatorio ge...,completed,True
1,BOE-A-2022-24383,"Resolución de 21 de diciembre de 2022, de la P...",El documento trata sobre la tercera prórroga d...,completed,True
2,BOE-A-2022-24403,"Resolución de 22 de diciembre de 2022, de la D...",El objeto principal del título es un proyecto ...,deterministic_scope_guard,True
5,BOE-A-2022-24406,"Orden TED/1343/2022, de 23 de diciembre, por l...",El documento establece la retribución de las e...,completed,True
18,BOE-A-2023-2613,"Resolución de 19 de enero de 2023, de la Comis...",El documento establece provisionalmente la ret...,completed,True
19,BOE-A-2023-2614,"Resolución de 19 de enero de 2023, de la Comis...",El documento establece provisionalmente la ret...,completed,True
20,BOE-A-2024-14378,"Resolución de 27 de junio de 2024, de la Direc...",El documento describe la autorización administ...,completed,True
21,BOE-A-2024-14379,"Resolución de 27 de junio de 2024, de la Direc...",El documento describe la ampliación de una sub...,completed,True
22,BOE-A-2024-14387,"Resolución de 27 de junio de 2024, de la Comis...",El documento trata sobre el ajuste retributivo...,completed,True
23,BOE-A-2024-15084,"Resolución de 11 de julio de 2024, de la Direc...",El documento trata sobre el informe de impacto...,completed,True


,review_queue_id,identificador_boe,fecha_publicacion,titulo,source_document_sha256,source_attempt_id,extraction_config_id,document_validation_version,error_type,error_message,processing_stage,validation_issues_json,proposed_extraction_json,queue_status,queued_at


,quality_run_id,measured_at,run_scope,extraction_config_id,document_validation_version,model_provider,model_name,n_source_documents,n_latest_attempts,n_auto_validated,...,n_rejected,automatic_validation_rate,effective_validation_rate,minimum_auto_validation_rate,n_scope_evaluated,n_scope_mismatches,scope_accuracy,minimum_scope_accuracy,quality_status,quality_alert
0,7aad74d10540408c918320a414b94885,2026-07-23 06:16:49.076508+00:00,pilot,db2bc8c3564ce062,25,gemini,google:gemini-2.5-flash,100,100,100,...,0,1.0,1.0,1.0,100,0,1.0,1.0,healthy,False


## 11. Producción, revisión manual y regeneración de tablas (opt-in)

La producción no se bloquea por documentos excepcionales. Los intentos no
validados se escriben en `boe_ai_review_queue.parquet`. Para corregir uno:

1. ejecuta `create_manual_review_file(review_queue, "BOE-...")`;
2. edita el JSON creado en `data/manual/boe_ai_reviews/`;
3. cambia `review_status` a `manually_validated`, indica `reviewer` y corrige
   `corrected_extraction`;
4. ejecuta esta sección con `REFRESH_REVIEW_WORKFLOW=True`.

El JSON se valida con el mismo contrato y contra el BOE. Después, la revisión
manual se integra automáticamente con las extracciones automáticas válidas.


### Interpretación del piloto

`OK events=0` significa únicamente que la salida cumple el contrato como documento no relevante. Desde la versión 23 el piloto también compara esa decisión con `pilot_scope_labels_100.csv`. El piloto solo se considera superado si la exactitud de alcance y la validación estructural son ambas del 100 %.


Los descartes deterministas de alta precisión (contratación pública, infraestructura gasista y almacenamiento autónomo sin planta asociada) se guardan como extracciones válidas con `processing_stage=deterministic_scope_guard` y no pasan por la cola de revisión.


In [ ]:

RUN_PRODUCTION_EXTRACTION = False
RESET_PRODUCTION_OUTPUTS = False
RESET_QUALITY_METRICS = False
REFRESH_REVIEW_WORKFLOW = False
REGENERATE_FLAT_TABLES = False

PRODUCTION_BOE_IDS: tuple[str, ...] = ()
PRODUCTION_MAX_DOCUMENTS: int | None = None
PRODUCTION_MIN_AUTO_VALIDATION_RATE = 0.95


if RUN_PRODUCTION_EXTRACTION or REFRESH_REVIEW_WORKFLOW:
    all_candidates = load_and_prepare_candidates()

    if RESET_PRODUCTION_OUTPUTS:
        for path in (
            BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            BOE_AI_EXTRACTIONS_PATH,
            BOE_AI_REVIEW_QUEUE_PATH,
        ):
            path.unlink(missing_ok=True)
    if RESET_QUALITY_METRICS:
        BOE_AI_QUALITY_METRICS_PATH.unlink(missing_ok=True)

    manual_reviews = load_manual_review_files(
        all_candidates,
        review_dir=BOE_AI_MANUAL_REVIEW_DIR,
        output_path=BOE_AI_MANUAL_REVIEWS_PATH,
    )
    attempts = load_ai_extraction_attempts()

    run_candidates = all_candidates.copy()
    if PRODUCTION_BOE_IDS:
        run_candidates = run_candidates.loc[
            run_candidates["identificador"].astype(str).isin(PRODUCTION_BOE_IDS)
        ].copy()
    if PRODUCTION_MAX_DOCUMENTS is not None:
        run_candidates = run_candidates.head(PRODUCTION_MAX_DOCUMENTS).copy()

    if RUN_PRODUCTION_EXTRACTION:
        _, pending = build_pending_candidates(
            run_candidates,
            attempts,
            manual_reviews=manual_reviews,
        )
        production_result = await run_and_finalize_extractions(
            pending,
            all_candidates,
            agent=agent,
            attempts_path=BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            current_path=BOE_AI_EXTRACTIONS_PATH,
            review_queue_path=BOE_AI_REVIEW_QUEUE_PATH,
            quality_metrics_path=BOE_AI_QUALITY_METRICS_PATH,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
            checkpoint_every=CHECKPOINT_EVERY,
        )
        attempts = production_result["all_attempts"]
        current_extractions = production_result["current_extractions"]
        review_queue = production_result["review_queue"]
        quality_metric = production_result["quality_metric"]
    else:
        current_extractions = select_best_valid_extractions(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(current_extractions, BOE_AI_EXTRACTIONS_PATH)

        review_queue = build_review_queue(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(review_queue, BOE_AI_REVIEW_QUEUE_PATH)

        quality_metric = build_quality_metric(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
        )
        append_quality_metric(
            quality_metric,
            BOE_AI_QUALITY_METRICS_PATH,
        )

    display(current_extractions[
        [
            "identificador_boe",
            "selection_source",
            "document_scope",
            "n_publication_events",
            "n_generation_assets",
            "n_associated_components",
            "n_administrative_actions",
        ]
    ])
    display(review_queue)
    display(quality_metric)


if REGENERATE_FLAT_TABLES:
    current_extractions = normalise_ai_extraction_attempts_log(
        pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)
    )
    flattened_tables = save_flattened_extractions(current_extractions)
    for table_name, dataframe in flattened_tables.items():
        print(f"{table_name}: {len(dataframe):,} filas")
